# Gold

In [ ]:
#!/usr/bin/env python3
"""
================================================================================
WARM START ACTIVE LEARNING PIPELINE - VERSION 5-FOLD CV - LOW BUDGET FOCUS
================================================================================
Complete script to simulate the personalised adaptation of a classifier
for French legal texts.

VERSION v10 - Optimised for limited budgets (20-150 annotations)
- Detailed tracking: % Yes/No annotated, true positives among the AL selections
- Extended P@k: k ∈ {5, 10, 20, 50, 100, 150, 200, 300}
- Reduced warm start: 20-50 examples
- Short-budget-focused visualisations with checkpoints

================================================================================
"""

# =============================================================================
# SECTION 1: IMPORTS AND CONFIGURATION
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Callable, Union
from dataclasses import dataclass, field
from enum import Enum
from collections import Counter
import copy

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, recall_score, precision_score,
    classification_report, confusion_matrix, ndcg_score, accuracy_score
)
from scipy.special import softmax
from scipy.stats import entropy as scipy_entropy
import scipy.sparse as sp

# BM25
from rank_bm25 import BM25Okapi

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ Imports loaded successfully!")


# =============================================================================
# SECTION 2: CONFIGURATION - OPTIMISED FOR LIMITED BUDGET
# =============================================================================

@dataclass
class Config:
    """Centralised configuration - optimised for limited budgets."""

    # Paths
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    output_dir: str = field(default="")

    # Input files
    main_dataset: str = "DATA/outputs/benchmark.csv"

    # LLM predictions
    llm_predictions: Dict[str, str] = field(default_factory=lambda: {
        "llama_8b": "DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx",
        "qwen_7b": "DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx",
        "mistral_12b": "DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx",
        "qwen_32b": "DATA/outputs/predictions/zeroshot_llms/qwen2.5-32b.xlsx",
        "gemma_27b": "DATA/outputs/predictions/zeroshot_llms/gemma2-27b.xlsx",
        "commandr": "DATA/outputs/predictions/zeroshot_llms/command-r-35b.xlsx",
        "aya_32b": "DATA/outputs/predictions/zeroshot_llms/aya-expanse-32b.xlsx",
        "llama_70b": "DATA/outputs/predictions/zeroshot_llms/llama3.1-70b.xlsx",
        "saul": "DATA/outputs/predictions/zeroshot_llms/saul-7b.xlsx"
    })

    # Key columns
    key_cols: List[str] = field(default_factory=lambda: ["decision_id", "chunk_id", "pred_art"])
    annotator_cols: List[str] = field(default_factory=lambda: ["eval_A1", "eval_A2", "eval_A3"])
    text_col: str = "text"
    article_text_col: str = "article_text"

    # === PARAMETERS OPTIMISED FOR LIMITED BUDGET ===
    warm_start_min: int = 10  # Reduced from 50 to 20
    warm_start_max: int = 10  # Reduced from 150 to 50
    batch_size: int = 5      # Reduced from 20 to 10 for fine granularity
    n_iterations: int = 15    # ~20 + 15*10 = 170 annotations max

    # 5-Fold CV parameters
    n_folds: int = 5

    # Model parameters
    random_seed: int = 42

    # Extended P@k for large scans
    precision_at_k_values: List[int] = field(default_factory=lambda: [5, 10, 20, 50, 100, 150, 200, 300])

    # Checkpoints for short-budget analysis
    budget_checkpoints: List[int] = field(default_factory=lambda: [5, 10, 15, 20, 30, 50, 75, 100, 125, 150])

    # Cross-Encoder disabled by default
    use_cross_encoder: bool = False
    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"

    def __post_init__(self):
        self.output_dir = f"artifacts/outputs_al_lowbudget_5fold_{datetime.now().strftime('%Y%m%d_%H%M')}"


class TargetType(Enum):
    """Target type for the optimisation."""
    GOLD = "gold"
    A1 = "A1"
    A2 = "A2"


class ALStrategy(Enum):
    """Active Learning strategies."""
    RANDOM = "random"
    ENTROPY = "entropy"
    HIGH_CONFIDENCE = "high_confidence"
    QUERY_BY_COMMITTEE = "qbc"


print("✅ Configuration defined (optimised for limited budget)!")


# =============================================================================
# SECTION 3: DATA LOADING AND PREPARATION
# =============================================================================

class DataLoader:
    """Handles loading and merging of the data."""

    def __init__(self, config: Config):
        self.config = config
        self.df_main = None
        self.df_merged = None
        self.llm_cols = []

    def load_main_dataset(self) -> pd.DataFrame:
        """Load the main dataset with the annotations."""
        path = f"{self.config.base_path}/{self.config.main_dataset}"
        print(f"📂 Loading the main dataset: {path}")
        self.df_main = pd.read_csv(path)
        print(f"   → {len(self.df_main)} entries loaded")
        return self.df_main

    def load_llm_predictions(self) -> Dict[str, pd.DataFrame]:
        """Load all LLM predictions."""
        llm_dfs = {}
        print("\n📂 Loading LLM predictions:")

        for llm_name, rel_path in self.config.llm_predictions.items():
            path = f"{self.config.base_path}/{rel_path}"
            if os.path.exists(path):
                df = pd.read_excel(path)
                llm_dfs[llm_name] = df
                print(f"   ✓ {llm_name}: {len(df)} predictions")
            else:
                print(f"   ✗ {llm_name}: file not found ({path})")

        return llm_dfs

    def merge_predictions(self, llm_dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """Merge the LLM predictions with the main dataset."""
        print("\n🔗 Merging predictions...")

        df_merged = self.df_main.copy()

        LLM_PRED_COL = {
            "llama_8b": "Llama_Pred",
            "qwen_7b": "Qwen_Pred",
            "mistral_12b": "Mistral_Pred",
            "qwen_32b": "Qwen_Pred",
            "gemma_27b": "Gemma_Pred",
            "commandr": "CommandR_Pred",
            "aya_32b": "AyaExpanse_Pred",
            "llama_70b": "Llama_Pred",
            "saul": "Saul_Pred",
        }

        self.llm_cols = []

        missing_main_keys = [c for c in self.config.key_cols if c not in df_merged.columns]
        if missing_main_keys:
            raise KeyError(f"Main dataset: missing key columns: {missing_main_keys}")

        for llm_name, df_llm in llm_dfs.items():
            if df_llm is None or len(df_llm) == 0:
                continue

            missing_llm_keys = [c for c in self.config.key_cols if c not in df_llm.columns]
            if missing_llm_keys:
                continue

            pred_col = LLM_PRED_COL.get(llm_name)

            if pred_col is None:
                candidates = [c for c in df_llm.columns if c.lower().endswith("_pred") and c != "pred_art"]
                if len(candidates) >= 1:
                    pred_col = candidates[0]
                else:
                    continue

            if pred_col not in df_llm.columns:
                continue

            new_col_name = f"llm_{llm_name}"
            self.llm_cols.append(new_col_name)

            df_llm_subset = df_llm[self.config.key_cols + [pred_col]].copy()
            df_llm_subset = df_llm_subset.rename(columns={pred_col: new_col_name})

            try:
                df_llm_subset[new_col_name] = pd.to_numeric(df_llm_subset[new_col_name], errors="coerce")
            except Exception:
                pass

            df_merged = df_merged.merge(df_llm_subset, on=self.config.key_cols, how="left")
            print(f"   ✓ {llm_name} merged -> {new_col_name}")

        self.df_merged = df_merged
        print(f"\n   → Dataset final: {len(df_merged)} entries, {len(df_merged.columns)} columns")
        return df_merged

    def create_target_labels(self) -> pd.DataFrame:
        """Create ALL target label columns + stratification key."""
        df = self.df_merged.copy()

        print(f"\n🎯 Building target labels and stratification key...")

        for col in self.config.annotator_cols:
            if col in df.columns:
                df[col] = df[col].apply(self._normalize_label)

        def compute_gold(row):
            A1 = row.get('eval_A1')
            A2 = row.get('eval_A2')
            A3 = row.get('eval_A3')

            if pd.isna(A1) or pd.isna(A2):
                return np.nan

            if A1 == A2:
                return A1
            else:
                return A3 if pd.notna(A3) else np.nan

        df['target_gold'] = df.apply(compute_gold, axis=1)
        df['target_A1'] = df['eval_A1']
        df['target_A2'] = df['eval_A2']

        # Stratification key based on (A1, A2)
        df['strat_key'] = (
            df['eval_A1'].fillna(-1).astype(int).astype(str) +
            "-" +
            df['eval_A2'].fillna(-1).astype(int).astype(str)
        )

        df['annotator_disagreement'] = (df['eval_A1'] != df['eval_A2']).astype(int)

        df['valid_for_all'] = (
            df['target_gold'].notna() &
            df['target_A1'].notna() &
            df['target_A2'].notna()
        )

        # Stats
        print(f"\n   📊 Stratification key distribution:")
        strat_dist = df.loc[df['valid_for_all'], 'strat_key'].value_counts()
        for key, count in strat_dist.items():
            label = "agreement" if key in ["0-0", "1-1"] else "DISAGREEMENT"
            print(f"   ├── {key} ({label}): {count} ({count/strat_dist.sum()*100:.1f}%)")

        print(f"\n   └── Valid for ALL: {df['valid_for_all'].sum()}")

        self.df_merged = df
        return df

    def set_target(self, target_type: TargetType) -> pd.DataFrame:
        """Set the 'target' column according to the chosen type."""
        df = self.df_merged.copy()

        if target_type == TargetType.GOLD:
            df['target'] = df['target_gold']
        elif target_type == TargetType.A1:
            df['target'] = df['target_A1']
        elif target_type == TargetType.A2:
            df['target'] = df['target_A2']

        print(f"\n   🎯 Target: {target_type.value}")
        valid_mask = df['target'].notna() & df['valid_for_all']
        dist = df.loc[valid_mask, 'target'].value_counts().to_dict()
        total = sum(dist.values())
        print(f"   └── Distribution: Yes={dist.get(1,0)} ({dist.get(1,0)/total*100:.1f}%), No={dist.get(0,0)} ({dist.get(0,0)/total*100:.1f}%)")

        self.df_merged = df
        return df

    @staticmethod
    def _normalize_label(val) -> Optional[int]:
        """Normalise labels to 0/1."""
        if pd.isna(val):
            return np.nan
        val_str = str(val).lower().strip()
        if val_str.startswith('oui'):
            return 1
        elif val_str.startswith('non'):
            return 0
        elif val_str in ['1', '1.0', 'true']:
            return 1
        elif val_str in ['0', '0.0', 'false']:
            return 0
        return np.nan

    def get_clean_dataset(self) -> pd.DataFrame:
        """Return the cleaned dataset."""
        df = self.df_merged[self.df_merged['valid_for_all']].copy()
        df['target'] = df['target'].astype(int)
        return df


print("✅ DataLoader defined!")


# =============================================================================
# SECTION 4: FEATURE ENGINEERING
# =============================================================================

class FeatureEngineer:
    """Feature handling with a guarantee of zero data leakage."""

    def __init__(self, config: Config, llm_cols: List[str]):
        self.config = config
        self.llm_cols = llm_cols
        self.tfidf_vectorizer = None
        self.scaler = None
        self._bm25_scores_cache = None

    def prepare_llm_features(self, df: pd.DataFrame) -> np.ndarray:
        """Prepare the LLM features."""
        llm_features = []
        for col in self.llm_cols:
            if col in df.columns:
                normalized = df[col].apply(DataLoader._normalize_label).fillna(0.5).values
                llm_features.append(normalized)
        if not llm_features:
            return np.zeros((len(df), 1))
        return np.column_stack(llm_features)

    def compute_llm_agreement_features(self, df: pd.DataFrame) -> np.ndarray:
        """Compute the LLM agreement features."""
        llm_predictions = self.prepare_llm_features(df)
        n_samples, _ = llm_predictions.shape
        features = []

        for i in range(n_samples):
            votes = llm_predictions[i]
            valid_votes = votes[votes != 0.5]

            if len(valid_votes) == 0:
                features.append([0.5, 1.0, 0.5, 0])
                continue

            vote_counts = Counter(valid_votes)
            n_yes = vote_counts.get(1, 0)
            n_no = vote_counts.get(0, 0)
            total = n_yes + n_no

            majority = max(n_yes, n_no)
            agreement_ratio = majority / total if total > 0 else 0.5

            if total > 0:
                p_yes = n_yes / total
                p_no = n_no / total
                probs = np.array([p for p in [p_yes, p_no] if p > 0])
                vote_entropy = scipy_entropy(probs, base=2)
            else:
                vote_entropy = 1.0

            majority_vote = 1 if n_yes > n_no else (0 if n_no > n_yes else 0.5)
            features.append([agreement_ratio, vote_entropy, majority_vote, n_yes])

        return np.array(features)

    def fit_tfidf(self, texts: List[str], max_features: int = 1000) -> None:
        """Fit the TF-IDF on the training texts."""
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        self.tfidf_vectorizer.fit(texts)

    def transform_tfidf(self, texts: List[str]) -> np.ndarray:
        """Transform the texts."""
        if self.tfidf_vectorizer is None:
            raise ValueError("TF-IDF not fitted.")
        return self.tfidf_vectorizer.transform(texts).toarray()

    def compute_bm25_scores(self, df: pd.DataFrame, force_recompute: bool = False) -> np.ndarray:
        """Compute the BM25 scores."""
        if self._bm25_scores_cache is not None and not force_recompute:
            return self._bm25_scores_cache

        print("   📊 Computing BM25 scores...")
        texts = df[self.config.text_col].fillna("").tolist()
        articles = df[self.config.article_text_col].fillna("").tolist()
        scores = []

        for text, article in tqdm(zip(texts, articles), total=len(texts), desc="BM25", leave=False):
            if not text or not article:
                scores.append(0.0)
                continue
            article_tokens = article.lower().split()
            if len(article_tokens) == 0:
                scores.append(0.0)
                continue
            bm25 = BM25Okapi([article_tokens])
            query_tokens = text.lower().split()
            score = bm25.get_scores(query_tokens)[0]
            scores.append(score)

        self._bm25_scores_cache = np.array(scores).reshape(-1, 1)
        return self._bm25_scores_cache

    def fit_scaler(self, features: np.ndarray) -> None:
        """Fit the scaler."""
        self.scaler = StandardScaler()
        self.scaler.fit(features)

    def transform_scaler(self, features: np.ndarray) -> np.ndarray:
        """Transform with the scaler."""
        if self.scaler is None:
            raise ValueError("Scaler not fitted.")
        return self.scaler.transform(features)

    def reset_fitted_transformers(self) -> None:
        """Reset for a new fold."""
        self.tfidf_vectorizer = None
        self.scaler = None

    def build_features(
        self,
        df: pd.DataFrame,
        train_indices: np.ndarray,
        fit_on_train: bool = True
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Build all features."""
        llm_feats = self.prepare_llm_features(df)
        llm_agreement_feats = self.compute_llm_agreement_features(df)
        bm25_scores = self.compute_bm25_scores(df)

        texts = df[self.config.text_col].fillna("").tolist()

        if fit_on_train:
            train_texts = [texts[i] for i in train_indices]
            self.fit_tfidf(train_texts)

        tfidf_feats = self.transform_tfidf(texts)

        numeric_feats = np.hstack([llm_feats, llm_agreement_feats, bm25_scores])

        if fit_on_train:
            self.fit_scaler(numeric_feats[train_indices])

        numeric_feats_scaled = self.transform_scaler(numeric_feats)
        all_features = np.hstack([numeric_feats_scaled, tfidf_feats])
        train_features = all_features[train_indices]

        return train_features, all_features


print("✅ FeatureEngineer defined!")


# =============================================================================
# SECTION 5: MODELS
# =============================================================================

class ModelWrapper:
    """Common interface for the models."""

    def __init__(self, model_type: str = "lr", random_seed: int = 42):
        self.model_type = model_type
        self.random_seed = random_seed
        self.model = None
        self._init_model()

    def _init_model(self):
        if self.model_type == "lr":
            self.model = LogisticRegression(
                random_state=self.random_seed,
                max_iter=1000,
                class_weight='balanced',
                C=1.0,
                solver='lbfgs'
            )
        elif self.model_type == "mlp":
            self.model = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                random_state=self.random_seed,
                max_iter=500,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
                alpha=0.001
            )
        else:
            raise ValueError(f"Unknown type: {self.model_type}")

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'ModelWrapper':
        self.model.fit(X, y)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(X)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict_proba(X)

    def reset(self) -> 'ModelWrapper':
        self._init_model()
        return self


print("✅ ModelWrapper defined!")


# =============================================================================
# SECTION 6: ACTIVE LEARNING STRATEGIES
# =============================================================================

class ALStrategies:
    """Active Learning selection strategies."""

    @staticmethod
    def random_sampling(model, X_pool, pool_indices, n_samples, **kwargs) -> np.ndarray:
        if len(pool_indices) <= n_samples:
            return pool_indices
        return np.random.choice(pool_indices, size=n_samples, replace=False)

    @staticmethod
    def entropy_sampling(model, X_pool, pool_indices, n_samples, **kwargs) -> np.ndarray:
        if len(pool_indices) <= n_samples:
            return pool_indices
        proba = model.predict_proba(X_pool)
        entropies = scipy_entropy(proba.T)
        top_indices = np.argsort(entropies)[-n_samples:]
        return pool_indices[top_indices]

    @staticmethod
    def high_confidence_sampling(model, X_pool, pool_indices, n_samples, **kwargs) -> np.ndarray:
        if len(pool_indices) <= n_samples:
            return pool_indices
        proba = model.predict_proba(X_pool)
        confidence = np.max(proba, axis=1)
        top_indices = np.argsort(confidence)[-n_samples:]
        return pool_indices[top_indices]

    @staticmethod
    def query_by_committee(model, X_pool, pool_indices, n_samples, llm_predictions=None, **kwargs) -> np.ndarray:
        if len(pool_indices) <= n_samples:
            return pool_indices
        if llm_predictions is None:
            return ALStrategies.entropy_sampling(model, X_pool, pool_indices, n_samples)

        disagreements = []
        for i, idx in enumerate(pool_indices):
            votes = llm_predictions[idx]
            valid_votes = votes[~np.isnan(votes) & (votes != 0.5)]
            if len(valid_votes) == 0:
                disagreements.append(0)
                continue
            vote_counts = Counter(valid_votes)
            total = sum(vote_counts.values())
            probs = [count / total for count in vote_counts.values()]
            disagreements.append(scipy_entropy(probs, base=2))

        top_indices = np.argsort(disagreements)[-n_samples:]
        return pool_indices[top_indices]

    @staticmethod
    def get_strategy_function(strategy: ALStrategy) -> Callable:
        return {
            ALStrategy.RANDOM: ALStrategies.random_sampling,
            ALStrategy.ENTROPY: ALStrategies.entropy_sampling,
            ALStrategy.HIGH_CONFIDENCE: ALStrategies.high_confidence_sampling,
            ALStrategy.QUERY_BY_COMMITTEE: ALStrategies.query_by_committee
        }[strategy]


print("✅ ALStrategies defined!")


# =============================================================================
# SECTION 7: METRICS - EXTENDED FOR LIMITED BUDGET
# =============================================================================

@dataclass
class AnnotationStats:
    """Statistics on the annotations made."""
    n_total: int = 0
    n_yes: int = 0
    n_no: int = 0
    pct_yes: float = 0.0
    pct_no: float = 0.0

    # Among the AL selections of this batch
    batch_n_yes: int = 0
    batch_n_no: int = 0
    batch_pct_yes: float = 0.0

    # Cumulative: among all AL annotations (excluding warm start)
    al_cumul_n_yes: int = 0
    al_cumul_n_no: int = 0
    al_cumul_pct_yes: float = 0.0


@dataclass
class EvaluationMetrics:
    """Container for the evaluation metrics - extended."""

    # Classification metrics
    f1_macro: float = 0.0
    f1_yes: float = 0.0
    f1_no: float = 0.0
    balanced_accuracy: float = 0.0
    recall_yes: float = 0.0
    recall_no: float = 0.0
    precision_yes: float = 0.0
    precision_no: float = 0.0

    # Extended recommendation metrics
    precision_at_k: Dict[int, float] = field(default_factory=dict)
    ndcg_at_k: Dict[int, float] = field(default_factory=dict)

    # Annotation statistics
    annotation_stats: AnnotationStats = field(default_factory=AnnotationStats)

    # Metadata
    n_train: int = 0
    iteration: int = 0


@dataclass
class AggregatedMetrics:
    """Metrics aggregated over several folds."""

    f1_macro_mean: float = 0.0
    f1_macro_std: float = 0.0
    f1_yes_mean: float = 0.0
    f1_yes_std: float = 0.0
    f1_no_mean: float = 0.0
    f1_no_std: float = 0.0
    balanced_accuracy_mean: float = 0.0
    balanced_accuracy_std: float = 0.0
    recall_yes_mean: float = 0.0
    recall_yes_std: float = 0.0
    precision_yes_mean: float = 0.0
    precision_yes_std: float = 0.0

    # Aggregated P@k
    precision_at_k_mean: Dict[int, float] = field(default_factory=dict)
    precision_at_k_std: Dict[int, float] = field(default_factory=dict)

    # Aggregated NDCG@k
    ndcg_at_k_mean: Dict[int, float] = field(default_factory=dict)
    ndcg_at_k_std: Dict[int, float] = field(default_factory=dict)

    # Aggregated annotation stats
    pct_yes_mean: float = 0.0
    pct_yes_std: float = 0.0
    al_pct_yes_mean: float = 0.0
    al_pct_yes_std: float = 0.0

    n_train: int = 0
    iteration: int = 0
    n_folds: int = 5


class Evaluator:
    """Detailed evaluation - extended for limited budget."""

    def __init__(self, config: Config):
        self.config = config

    def evaluate_model(
        self,
        model: ModelWrapper,
        X_test: np.ndarray,
        y_test: np.ndarray,
        X_pool: np.ndarray = None,
        y_pool: np.ndarray = None,
        n_train: int = 0,
        iteration: int = 0,
        annotation_stats: AnnotationStats = None
    ) -> EvaluationMetrics:
        """Full evaluation with extended P@k."""
        metrics = EvaluationMetrics(n_train=n_train, iteration=iteration)

        if annotation_stats:
            metrics.annotation_stats = annotation_stats

        # Classification metrics
        y_pred = model.predict(X_test)

        metrics.f1_macro = f1_score(y_test, y_pred, average='macro')
        metrics.f1_yes = f1_score(y_test, y_pred, pos_label=1, average='binary')
        metrics.f1_no = f1_score(y_test, y_pred, pos_label=0, average='binary')
        metrics.balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
        metrics.recall_yes = recall_score(y_test, y_pred, pos_label=1)
        metrics.recall_no = recall_score(y_test, y_pred, pos_label=0)
        metrics.precision_yes = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
        metrics.precision_no = precision_score(y_test, y_pred, pos_label=0, zero_division=0)

        # Recommendation metrics with extended P@k
        if X_pool is not None and y_pool is not None and len(X_pool) > 0:
            pool_proba = model.predict_proba(X_pool)
            scores = pool_proba[:, 1] if pool_proba.shape[1] == 2 else pool_proba.ravel()

            sorted_indices = np.argsort(scores)[::-1]
            sorted_labels = y_pool[sorted_indices]

            # Extended P@k
            for k in self.config.precision_at_k_values:
                if k <= len(sorted_labels):
                    top_k_labels = sorted_labels[:k]
                    metrics.precision_at_k[k] = np.mean(top_k_labels == 1)

            # Extended NDCG@k
            for k in self.config.precision_at_k_values:
                if k <= len(sorted_labels):
                    relevance = (y_pool == 1).astype(float)
                    try:
                        ndcg = ndcg_score(relevance.reshape(1, -1), scores.reshape(1, -1), k=k)
                        metrics.ndcg_at_k[k] = ndcg
                    except:
                        metrics.ndcg_at_k[k] = 0.0

        return metrics

    @staticmethod
    def aggregate_metrics(metrics_list: List[EvaluationMetrics], config: Config) -> AggregatedMetrics:
        """Aggregate the metrics from several folds."""
        agg = AggregatedMetrics()

        if not metrics_list:
            return agg

        agg.n_folds = len(metrics_list)
        agg.n_train = metrics_list[0].n_train
        agg.iteration = metrics_list[0].iteration

        # Classification
        agg.f1_macro_mean = np.mean([m.f1_macro for m in metrics_list])
        agg.f1_macro_std = np.std([m.f1_macro for m in metrics_list])
        agg.f1_yes_mean = np.mean([m.f1_yes for m in metrics_list])
        agg.f1_yes_std = np.std([m.f1_yes for m in metrics_list])
        agg.f1_no_mean = np.mean([m.f1_no for m in metrics_list])
        agg.f1_no_std = np.std([m.f1_no for m in metrics_list])
        agg.balanced_accuracy_mean = np.mean([m.balanced_accuracy for m in metrics_list])
        agg.balanced_accuracy_std = np.std([m.balanced_accuracy for m in metrics_list])
        agg.recall_yes_mean = np.mean([m.recall_yes for m in metrics_list])
        agg.recall_yes_std = np.std([m.recall_yes for m in metrics_list])
        agg.precision_yes_mean = np.mean([m.precision_yes for m in metrics_list])
        agg.precision_yes_std = np.std([m.precision_yes for m in metrics_list])

        # Aggregated P@k
        for k in config.precision_at_k_values:
            values = [m.precision_at_k.get(k, 0) for m in metrics_list]
            agg.precision_at_k_mean[k] = np.mean(values)
            agg.precision_at_k_std[k] = np.std(values)

        # Aggregated NDCG@k
        for k in config.precision_at_k_values:
            values = [m.ndcg_at_k.get(k, 0) for m in metrics_list]
            agg.ndcg_at_k_mean[k] = np.mean(values)
            agg.ndcg_at_k_std[k] = np.std(values)

        # Annotation stats
        agg.pct_yes_mean = np.mean([m.annotation_stats.pct_yes for m in metrics_list])
        agg.pct_yes_std = np.std([m.annotation_stats.pct_yes for m in metrics_list])
        agg.al_pct_yes_mean = np.mean([m.annotation_stats.al_cumul_pct_yes for m in metrics_list])
        agg.al_pct_yes_std = np.std([m.annotation_stats.al_cumul_pct_yes for m in metrics_list])

        return agg

    def compute_aubc(self, f1_scores: List[float], n_samples: List[int]) -> float:
        """Compute the AUBC."""
        if len(f1_scores) < 2:
            return 0.0
        n_samples = np.array(n_samples)
        n_normalized = (n_samples - n_samples.min()) / (n_samples.max() - n_samples.min() + 1e-10)
        return np.trapz(f1_scores, n_normalized)

    def compute_aubc_budget(
        self,
        f1_scores: List[float],
        n_samples: List[int],
        max_budget: int = 150
    ) -> float:
        """Compute the AUBC limited to the max budget."""
        # Filter for n_samples <= max_budget
        mask = np.array(n_samples) <= max_budget
        filtered_f1 = np.array(f1_scores)[mask]
        filtered_n = np.array(n_samples)[mask]

        if len(filtered_f1) < 2:
            return 0.0

        n_normalized = (filtered_n - filtered_n.min()) / (filtered_n.max() - filtered_n.min() + 1e-10)
        return np.trapz(filtered_f1, n_normalized)


print("✅ Evaluator defined (extended for limited budget)!")


# =============================================================================
# SECTION 8: WARM START
# =============================================================================

class WarmStartSelector:
    """Warm-start selection - balanced Yes/No."""

    def __init__(self, config: Config, min_samples: int = 20, max_samples: int = 50):
        self.config = config
        self.min_samples = min_samples
        self.max_samples = max_samples

    def compute_llm_consensus(self, df: pd.DataFrame, llm_cols: List[str]) -> pd.DataFrame:
        """Compute the LLM consensus."""
        df = df.copy()
        n_agree_list, total_list, majority_list = [], [], []

        for idx, row in df.iterrows():
            votes = []
            for col in llm_cols:
                if col in df.columns:
                    val = DataLoader._normalize_label(row.get(col))
                    if val is not None and not np.isnan(val):
                        votes.append(int(val))

            if len(votes) == 0:
                n_agree_list.append(0)
                total_list.append(0)
                majority_list.append(np.nan)
                continue

            vote_counts = Counter(votes)
            majority = vote_counts.most_common(1)[0]
            n_agree_list.append(majority[1])
            total_list.append(len(votes))
            majority_list.append(majority[0])

        df['llm_n_agree'] = n_agree_list
        df['llm_total'] = total_list
        df['llm_majority_vote'] = majority_list
        df['llm_consensus_ratio'] = df['llm_n_agree'] / df['llm_total'].clip(lower=1)

        return df

    def select_warm_start_indices(
        self,
        df: pd.DataFrame,
        llm_cols: List[str],
        available_indices: np.ndarray
    ) -> np.ndarray:
        """Select the indices for the warm start - BALANCED YES/NO."""
        df_consensus = self.compute_llm_consensus(df, llm_cols)
        df_available = df_consensus.iloc[available_indices].copy()
        df_available['original_idx'] = available_indices

        # === STEP 1: Identify the unanimous examples ===
        unanimous_mask = df_available['llm_n_agree'] == df_available['llm_total']
        df_unanimous = df_available[unanimous_mask].copy()

        # Split into two groups: unanimous YES and unanimous NO
        unanimous_yes = df_unanimous[df_unanimous['llm_majority_vote'] == 1]['original_idx'].values
        unanimous_no = df_unanimous[df_unanimous['llm_majority_vote'] == 0]['original_idx'].values

        print(f"   📊 Warm start - Unanimous candidates: {len(unanimous_yes)} Yes, {len(unanimous_no)} No")

        # === STEP 2: Balanced selection strategy ===
        target_per_class = self.max_samples // 2  # Half and half

        selected_yes = []
        selected_no = []

        # Strategy 1: If we have enough in both classes
        if len(unanimous_yes) >= target_per_class and len(unanimous_no) >= target_per_class:
            selected_yes = np.random.choice(unanimous_yes, size=target_per_class, replace=False)
            selected_no = np.random.choice(unanimous_no, size=target_per_class, replace=False)
            print(f"   ✅ Balanced selection: {len(selected_yes)} Yes + {len(selected_no)} No (unanimous)")

        # Strategy 2: If a class is missing, top up with near-unanimous ones
        else:
            # Take all available unanimous ones
            selected_yes = unanimous_yes.copy()
            selected_no = unanimous_no.copy()

            # Identify the near-unanimous ones (n_agree >= total - 1, and at least 8 LLMs)
            near_unanimous_mask = (
                (df_available['llm_n_agree'] >= df_available['llm_total'] - 1) &
                (df_available['llm_total'] >= 8) &
                (~unanimous_mask)  # Exclude those already selected
            )
            df_near_unanimous = df_available[near_unanimous_mask].copy()

            near_unanimous_yes = df_near_unanimous[df_near_unanimous['llm_majority_vote'] == 1]['original_idx'].values
            near_unanimous_no = df_near_unanimous[df_near_unanimous['llm_majority_vote'] == 0]['original_idx'].values

            # Top up the missing class
            if len(selected_yes) < target_per_class:
                needed = target_per_class - len(selected_yes)
                if len(near_unanimous_yes) >= needed:
                    additional = np.random.choice(near_unanimous_yes, size=needed, replace=False)
                    selected_yes = np.concatenate([selected_yes, additional])
                else:
                    selected_yes = np.concatenate([selected_yes, near_unanimous_yes])

            if len(selected_no) < target_per_class:
                needed = target_per_class - len(selected_no)
                if len(near_unanimous_no) >= needed:
                    additional = np.random.choice(near_unanimous_no, size=needed, replace=False)
                    selected_no = np.concatenate([selected_no, additional])
                else:
                    selected_no = np.concatenate([selected_no, near_unanimous_no])

            print(f"   ⚖️ Selection with near-unanimous ones: {len(selected_yes)} Yes + {len(selected_no)} No")

        # === STEP 3: Adjust to reach min_samples if needed ===
        total_selected = len(selected_yes) + len(selected_no)

        if total_selected < self.min_samples:
            print(f"   ⚠️ Only {total_selected} selected, topping up with strong consensus...")

            # Take the examples with the strongest consensus (high ratio)
            already_selected = set(selected_yes) | set(selected_no)
            df_remaining = df_available[~df_available['original_idx'].isin(already_selected)].copy()
            df_remaining = df_remaining.sort_values('llm_consensus_ratio', ascending=False)

            needed = self.min_samples - total_selected
            additional = df_remaining['original_idx'].values[:needed]

            # Split evenly between yes and no
            for idx in additional:
                vote = df_remaining[df_remaining['original_idx'] == idx]['llm_majority_vote'].values[0]
                if vote == 1:
                    selected_yes = np.append(selected_yes, idx)
                else:
                    selected_no = np.append(selected_no, idx)

        # === STEP 4: Cap at max_samples while keeping the balance ===
        total_selected = len(selected_yes) + len(selected_no)
        if total_selected > self.max_samples:
            # Reduce proportionally
            keep_yes = min(len(selected_yes), self.max_samples // 2)
            keep_no = min(len(selected_no), self.max_samples // 2)

            selected_yes = np.random.choice(selected_yes, size=keep_yes, replace=False)
            selected_no = np.random.choice(selected_no, size=keep_no, replace=False)

        # Combine and shuffle
        selected = np.concatenate([selected_yes, selected_no])
        np.random.shuffle(selected)

        print(f"   🎯 Final warm start: {len(selected_yes)} Yes + {len(selected_no)} No = {len(selected)} total")

        return selected.astype(int)

    def select_warm_start_indices(
        self,
        df: pd.DataFrame,
        llm_cols: List[str],
        available_indices: np.ndarray
    ) -> np.ndarray:
        """Select the indices for the warm start."""
        df_consensus = self.compute_llm_consensus(df, llm_cols)
        df_available = df_consensus.iloc[available_indices].copy()
        df_available['original_idx'] = available_indices

        unanimous_mask = df_available['llm_n_agree'] == df_available['llm_total']
        unanimous_indices = df_available.loc[unanimous_mask, 'original_idx'].values

        if len(unanimous_indices) >= self.min_samples:
            if len(unanimous_indices) <= self.max_samples:
                selected = unanimous_indices
            else:
                selected = np.random.choice(unanimous_indices, size=self.max_samples, replace=False)
        else:
            near_unanimous_mask = (
                (df_available['llm_n_agree'] >= df_available['llm_total'] - 1) &
                (df_available['llm_total'] >= 8)
            )
            near_unanimous_indices = df_available.loc[near_unanimous_mask, 'original_idx'].values

            if len(near_unanimous_indices) >= self.min_samples:
                if len(near_unanimous_indices) <= self.max_samples:
                    selected = near_unanimous_indices
                else:
                    selected = np.random.choice(near_unanimous_indices, size=self.max_samples, replace=False)
            else:
                df_sorted = df_available.sort_values('llm_consensus_ratio', ascending=False)
                selected = df_sorted['original_idx'].values[:self.min_samples]

        return selected.astype(int)


print("✅ WarmStartSelector defined!")


# =============================================================================
# SECTION 9: 5-FOLD CROSS-VALIDATION RUNNER
# =============================================================================

class CrossValidationRunner:
    """Orchestrator for the stratified 5-Fold CV."""

    def __init__(self, config: Config):
        self.config = config
        self.folds = []

    def create_stratified_folds(self, df: pd.DataFrame, strat_col: str = 'strat_key'):
        """Create the stratified folds."""
        print(f"\n{'='*60}")
        print(f"📊 CREATING {self.config.n_folds} STRATIFIED FOLDS")
        print(f"{'='*60}")

        strat_distribution = df[strat_col].value_counts()
        print(f"\n   Stratification key distribution:")
        for key, count in strat_distribution.items():
            pct = count / len(df) * 100
            label = "agreement" if key in ["0-0", "1-1"] else "DISAGREEMENT"
            print(f"   ├── {key} ({label}): {count} ({pct:.1f}%)")

        min_class_size = strat_distribution.min()
        if min_class_size < self.config.n_folds:
            print(f"\n   ⚠️ Minority class ({min_class_size}) < n_folds → grouping")
            df['strat_key_grouped'] = df[strat_col].apply(
                lambda x: 'accord' if x in ['0-0', '1-1'] else 'desaccord'
            )
            strat_col = 'strat_key_grouped'

        skf = StratifiedKFold(n_splits=self.config.n_folds, shuffle=True, random_state=self.config.random_seed)

        X = np.arange(len(df))
        y = df[strat_col].values

        self.folds = []
        print(f"\n   📦 Generated folds:")

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
            self.folds.append((train_idx, test_idx))
            train_dist = df.iloc[train_idx]['strat_key'].value_counts().to_dict()
            test_dist = df.iloc[test_idx]['strat_key'].value_counts().to_dict()
            n_desaccord_train = train_dist.get('0-1', 0) + train_dist.get('1-0', 0)
            n_desaccord_test = test_dist.get('0-1', 0) + test_dist.get('1-0', 0)

            print(f"\n   Fold {fold_idx + 1}:")
            print(f"      ├── Train/Pool: {len(train_idx)} ({n_desaccord_train} disagreements)")
            print(f"      └── Test:       {len(test_idx)} ({n_desaccord_test} disagreements)")

        return self.folds


class ActiveLearnerCV:
    """Active Learner for 5-Fold CV with annotation tracking."""

    def __init__(self, config: Config, feature_engineer: FeatureEngineer,
                 evaluator: Evaluator, warm_start_selector: WarmStartSelector):
        self.config = config
        self.feature_engineer = feature_engineer
        self.evaluator = evaluator
        self.warm_start_selector = warm_start_selector

    def run_single_fold(
        self,
        df: pd.DataFrame,
        train_pool_indices: np.ndarray,
        test_indices: np.ndarray,
        llm_cols: List[str],
        strategy: ALStrategy,
        model_type: str = "lr",
        fold_idx: int = 0
    ) -> Dict:
        """Run AL on a single fold with annotation tracking."""

        self.feature_engineer.reset_fitted_transformers()

        # Warm start
        warm_start_indices = self.warm_start_selector.select_warm_start_indices(
            df, llm_cols, train_pool_indices
        )

        train_indices = warm_start_indices.tolist()
        pool_indices = [i for i in train_pool_indices if i not in warm_start_indices]

        # Tracking of AL annotations (excluding warm start)
        al_annotations_indices = []

        llm_predictions = self.feature_engineer.prepare_llm_features(df)

        history = {
            'iterations': [],
            'n_train': [],
            'n_pool': [],
            'metrics': [],
            'annotation_details': []  # New: details per iteration
        }

        model = ModelWrapper(model_type=model_type, random_seed=self.config.random_seed)
        strategy_fn = ALStrategies.get_strategy_function(strategy)

        y_all = df['target'].values
        y_test = y_all[test_indices]

        for iteration in range(self.config.n_iterations + 1):
            # Build features
            train_features, all_features = self.feature_engineer.build_features(
                df, train_indices=np.array(train_indices), fit_on_train=True
            )

            X_train = train_features
            y_train = y_all[train_indices]
            X_test = all_features[test_indices]
            X_pool = all_features[pool_indices] if pool_indices else np.array([])
            y_pool = y_all[pool_indices] if pool_indices else np.array([])

            # Train
            model.reset()
            model.fit(X_train, y_train)

            # === Compute annotation stats ===
            annotation_stats = AnnotationStats()

            # Total annotations
            annotation_stats.n_total = len(train_indices)
            annotation_stats.n_yes = int(np.sum(y_train == 1))
            annotation_stats.n_no = int(np.sum(y_train == 0))
            annotation_stats.pct_yes = annotation_stats.n_yes / annotation_stats.n_total * 100 if annotation_stats.n_total > 0 else 0
            annotation_stats.pct_no = annotation_stats.n_no / annotation_stats.n_total * 100 if annotation_stats.n_total > 0 else 0

            # AL cumulative (excluding warm start)
            if len(al_annotations_indices) > 0:
                al_labels = y_all[al_annotations_indices]
                annotation_stats.al_cumul_n_yes = int(np.sum(al_labels == 1))
                annotation_stats.al_cumul_n_no = int(np.sum(al_labels == 0))
                annotation_stats.al_cumul_pct_yes = annotation_stats.al_cumul_n_yes / len(al_annotations_indices) * 100

            # Evaluate
            metrics = self.evaluator.evaluate_model(
                model, X_test, y_test, X_pool, y_pool,
                n_train=len(train_indices), iteration=iteration,
                annotation_stats=annotation_stats
            )

            history['iterations'].append(iteration)
            history['n_train'].append(len(train_indices))
            history['n_pool'].append(len(pool_indices))
            history['metrics'].append(metrics)
            history['annotation_details'].append({
                'n_total': annotation_stats.n_total,
                'n_yes': annotation_stats.n_yes,
                'n_no': annotation_stats.n_no,
                'pct_yes': annotation_stats.pct_yes,
                'al_cumul_n_yes': annotation_stats.al_cumul_n_yes,
                'al_cumul_pct_yes': annotation_stats.al_cumul_pct_yes
            })

            # Select next batch
            if iteration < self.config.n_iterations and len(pool_indices) > 0:
                selected_indices = strategy_fn(
                    model=model,
                    X_pool=X_pool,
                    pool_indices=np.array(pool_indices),
                    n_samples=min(self.config.batch_size, len(pool_indices)),
                    llm_predictions=llm_predictions
                )

                # Track batch stats
                batch_labels = y_all[selected_indices]
                annotation_stats.batch_n_yes = int(np.sum(batch_labels == 1))
                annotation_stats.batch_n_no = int(np.sum(batch_labels == 0))
                annotation_stats.batch_pct_yes = annotation_stats.batch_n_yes / len(selected_indices) * 100 if len(selected_indices) > 0 else 0

                # Update
                train_indices.extend(selected_indices.tolist())
                al_annotations_indices.extend(selected_indices.tolist())
                pool_indices = [i for i in pool_indices if i not in selected_indices]

        # Compute AUBC
        f1_scores = [m.f1_macro for m in history['metrics']]
        n_samples = history['n_train']
        history['aubc'] = self.evaluator.compute_aubc(f1_scores, n_samples)
        history['aubc_budget_150'] = self.evaluator.compute_aubc_budget(f1_scores, n_samples, max_budget=150)
        history['aubc_budget_100'] = self.evaluator.compute_aubc_budget(f1_scores, n_samples, max_budget=100)

        return history


print("✅ CrossValidationRunner and ActiveLearnerCV defined!")


# =============================================================================
# SECTION 10: VISUALISATION - EXTENDED FOR LIMITED BUDGET
# =============================================================================

class ResultsVisualizer:
    """Extended visualisation for limited budget."""

    def __init__(self, config: Config):
        self.config = config
        os.makedirs(config.output_dir, exist_ok=True)

    def plot_learning_curves_cv(
        self,
        aggregated_results: Dict[str, Dict],
        metric: str = 'f1_macro',
        title: str = "Learning curves",
        save_name: str = "learning_curves.png",
        max_budget: int = None
    ) -> None:
        """Learning curves with an optional limited-budget view."""
        fig, ax = plt.subplots(figsize=(12, 7))

        colors = {
            'random_lr': '#1f77b4', 'entropy_lr': '#ff7f0e',
            'high_confidence_lr': '#2ca02c', 'qbc_lr': '#d62728',
            'random_mlp': '#9467bd', 'entropy_mlp': '#8c564b',
            'high_confidence_mlp': '#e377c2', 'qbc_mlp': '#7f7f7f'
        }
        linestyles = {'lr': '-', 'mlp': '--'}

        for exp_name, result in aggregated_results.items():
            n_train = np.array(result['n_train'])

            if metric == 'f1_macro':
                means = np.array([m.f1_macro_mean for m in result['aggregated_metrics']])
                stds = np.array([m.f1_macro_std for m in result['aggregated_metrics']])
            elif metric == 'balanced_accuracy':
                means = np.array([m.balanced_accuracy_mean for m in result['aggregated_metrics']])
                stds = np.array([m.balanced_accuracy_std for m in result['aggregated_metrics']])
            else:
                means = np.array([getattr(m, f'{metric}_mean', 0) for m in result['aggregated_metrics']])
                stds = np.array([getattr(m, f'{metric}_std', 0) for m in result['aggregated_metrics']])

            # Filter by budget if specified
            if max_budget:
                mask = n_train <= max_budget
                n_train = n_train[mask]
                means = means[mask]
                stds = stds[mask]

            model_type = 'mlp' if 'mlp' in exp_name.lower() else 'lr'
            color = colors.get(exp_name, '#333333')
            ls = linestyles[model_type]

            aubc_key = f'aubc_budget_{max_budget}' if max_budget else 'aubc_mean'
            aubc = result.get(aubc_key, result.get('aubc_mean', 0))
            aubc_std = result.get(f'{aubc_key}_std', result.get('aubc_std', 0))

            label = f"{exp_name} (AUBC={aubc:.3f})"
            ax.plot(n_train, means, label=label, color=color, linestyle=ls, linewidth=2)
            ax.fill_between(n_train, means - stds, means + stds, color=color, alpha=0.15)

        ax.set_xlabel("Number of annotations", fontsize=12)
        ax.set_ylabel(f"{metric} (mean ± std)", fontsize=12)

        budget_str = f" (Budget ≤ {max_budget})" if max_budget else ""
        ax.set_title(f"{title}{budget_str}", fontsize=14)
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(True, alpha=0.3)

        if max_budget:
            ax.set_xlim(0, max_budget + 10)

        plt.tight_layout()
        save_path = f"{self.config.output_dir}/{save_name}"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 Saved: {save_path}")
        plt.show()

    def plot_budget_comparison(
        self,
        aggregated_results: Dict[str, Dict],
        save_name: str = "budget_comparison.png"
    ) -> None:
        """Visual comparison at different budgets."""
        budgets = self.config.budget_checkpoints
        n_budgets = len(budgets)

        # Compute the optimal grid
        n_cols = 3
        n_rows = (n_budgets + n_cols - 1) // n_cols  # Ceiling division

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
        axes = axes.flatten()

        for idx, budget in enumerate(budgets):
            ax = axes[idx]

            data = []
            for exp_name, result in aggregated_results.items():
                # Find closest n_train to budget
                n_trains = result['n_train']
                closest_idx = np.argmin(np.abs(np.array(n_trains) - budget))

                if closest_idx < len(result['aggregated_metrics']):
                    m = result['aggregated_metrics'][closest_idx]
                    data.append({
                        'Strategy': exp_name.replace('_lr', ' (LR)').replace('_mlp', ' (MLP)'),
                        'F1': m.f1_macro_mean,
                        'F1_std': m.f1_macro_std,
                        'n_actual': n_trains[closest_idx]
                    })

            if data:
                df_plot = pd.DataFrame(data).sort_values('F1', ascending=True)
                colors = ['#2ecc71' if i == len(df_plot)-1 else '#3498db' for i in range(len(df_plot))]

                bars = ax.barh(df_plot['Strategy'], df_plot['F1'], xerr=df_plot['F1_std'],
                              color=colors, capsize=3, alpha=0.8)
                ax.set_xlabel('F1 Macro')
                ax.set_title(f'Budget ≈ {budget} annotations\n(actual: {df_plot["n_actual"].iloc[-1]})')
                ax.set_xlim(0, 1)
                ax.grid(True, alpha=0.3, axis='x')

                # Add value labels
                for bar, val in zip(bars, df_plot['F1']):
                    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                          f'{val:.3f}', va='center', fontsize=9)

        # Hide unused subplots if any
        for idx in range(n_budgets, len(axes)):
            axes[idx].set_visible(False)

        plt.suptitle("Performance by annotation budget", fontsize=14, y=1.01)
        plt.tight_layout()

        save_path = f"{self.config.output_dir}/{save_name}"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 Saved: {save_path}")
        plt.show()

    def plot_precision_at_k_extended(
        self,
        aggregated_results: Dict[str, Dict],
        save_name: str = "precision_at_k_extended.png"
    ) -> None:
        """Extended P@k for large scans."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Final iteration metrics
        k_values = self.config.precision_at_k_values

        # Plot 1: P@k for all strategies (final iteration)
        ax = axes[0, 0]
        for exp_name, result in aggregated_results.items():
            final_m = result['aggregated_metrics'][-1]
            p_at_k = [final_m.precision_at_k_mean.get(k, 0) for k in k_values]
            ax.plot(k_values, p_at_k, marker='o', label=exp_name, linewidth=2)

        ax.set_xlabel('k (number of results scanned)')
        ax.set_ylabel('Precision@k')
        ax.set_title('Precision@k at the end of AL')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Plot 2: NDCG@k
        ax = axes[0, 1]
        for exp_name, result in aggregated_results.items():
            final_m = result['aggregated_metrics'][-1]
            ndcg_at_k = [final_m.ndcg_at_k_mean.get(k, 0) for k in k_values]
            ax.plot(k_values, ndcg_at_k, marker='s', label=exp_name, linewidth=2)

        ax.set_xlabel('k')
        ax.set_ylabel('NDCG@k')
        ax.set_title('NDCG@k at the end of AL')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Plot 3: P@100 over iterations
        ax = axes[1, 0]
        for exp_name, result in aggregated_results.items():
            n_train = result['n_train']
            p_at_100 = [m.precision_at_k_mean.get(100, 0) for m in result['aggregated_metrics']]
            ax.plot(n_train, p_at_100, marker='o', label=exp_name, linewidth=2, markersize=4)

        ax.set_xlabel("Number of annotations")
        ax.set_ylabel('P@100')
        ax.set_title('Evolution of P@100 with budget')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Plot 4: P@200 over iterations
        ax = axes[1, 1]
        for exp_name, result in aggregated_results.items():
            n_train = result['n_train']
            p_at_200 = [m.precision_at_k_mean.get(200, 0) for m in result['aggregated_metrics']]
            ax.plot(n_train, p_at_200, marker='s', label=exp_name, linewidth=2, markersize=4)

        ax.set_xlabel("Number of annotations")
        ax.set_ylabel('P@200')
        ax.set_title('Evolution of P@200 with budget')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        save_path = f"{self.config.output_dir}/{save_name}"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 Saved: {save_path}")
        plt.show()

    def plot_annotation_distribution(
        self,
        aggregated_results: Dict[str, Dict],
        save_name: str = "annotation_distribution.png"
    ) -> None:
        """Distribution of Yes/No annotations by strategy."""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Plot 1: % Yes among total annotations
        ax = axes[0]
        for exp_name, result in aggregated_results.items():
            n_train = result['n_train']
            pct_yes = [m.pct_yes_mean for m in result['aggregated_metrics']]
            ax.plot(n_train, pct_yes, marker='o', label=exp_name, linewidth=2, markersize=4)

        ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% balance')
        ax.set_xlabel("Number of annotations")
        ax.set_ylabel('% of "Yes" annotated')
        ax.set_title('Annotation distribution (total)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 100)

        # Plot 2: % Yes among AL selections (excluding warm start)
        ax = axes[1]
        for exp_name, result in aggregated_results.items():
            n_train = result['n_train']
            al_pct_yes = [m.al_pct_yes_mean for m in result['aggregated_metrics']]
            ax.plot(n_train, al_pct_yes, marker='s', label=exp_name, linewidth=2, markersize=4)

        ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% balance')
        ax.set_xlabel("Number of annotations")
        ax.set_ylabel('% of "Yes" among AL selections')
        ax.set_title('AL selection distribution (excluding warm start)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 100)

        plt.tight_layout()
        save_path = f"{self.config.output_dir}/{save_name}"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 Saved: {save_path}")
        plt.show()

    def generate_budget_checkpoint_table(
        self,
        aggregated_results: Dict[str, Dict],
        target_type: TargetType
    ) -> pd.DataFrame:
        """Generate a summary table at the budget checkpoints."""

        print(f"\n{'='*80}")
        print(f"📊 SUMMARY TABLE - BUDGET CHECKPOINTS ({target_type.value})")
        print(f"{'='*80}")

        rows = []

        for exp_name, result in aggregated_results.items():
            n_trains = result['n_train']

            for budget in self.config.budget_checkpoints:
                closest_idx = np.argmin(np.abs(np.array(n_trains) - budget))

                if closest_idx < len(result['aggregated_metrics']):
                    m = result['aggregated_metrics'][closest_idx]
                    actual_n = n_trains[closest_idx]

                    rows.append({
                        'Strategy': exp_name,
                        'Budget_Target': budget,
                        'Budget_Actual': actual_n,
                        'F1_Macro': f"{m.f1_macro_mean:.3f}±{m.f1_macro_std:.3f}",
                        'F1_Macro_mean': m.f1_macro_mean,
                        'BalAcc': f"{m.balanced_accuracy_mean:.3f}±{m.balanced_accuracy_std:.3f}",
                        'P@50': f"{m.precision_at_k_mean.get(50, 0):.3f}",
                        'P@100': f"{m.precision_at_k_mean.get(100, 0):.3f}",
                        'P@200': f"{m.precision_at_k_mean.get(200, 0):.3f}",
                        'Pct_Oui': f"{m.pct_yes_mean:.1f}%",
                        'AL_Pct_Oui': f"{m.al_pct_yes_mean:.1f}%"
                    })

        df = pd.DataFrame(rows)

        # Print formatted table for each budget
        for budget in self.config.budget_checkpoints:
            df_budget = df[df['Budget_Target'] == budget].sort_values('F1_Macro_mean', ascending=False)

            print(f"\n📍 Budget ≈ {budget} annotations:")
            print("-" * 100)
            print(f"{'Strategy':<25} {'F1 Macro':<15} {'BalAcc':<15} {'P@100':<10} {'P@200':<10} {'%Yes(AL)':<10}")
            print("-" * 100)

            for _, row in df_budget.iterrows():
                best_marker = "🏆" if row.name == df_budget.index[0] else "  "
                print(f"{best_marker}{row['Strategy']:<23} {row['F1_Macro']:<15} {row['BalAcc']:<15} "
                      f"{row['P@100']:<10} {row['P@200']:<10} {row['AL_Pct_Oui']:<10}")

        return df

    def save_results_to_excel_cv(
        self,
        aggregated_results: Dict[str, Dict],
        target_type: TargetType,
        filename: str = None
    ) -> str:
        """Save the extended results to Excel."""
        if filename is None:
            filename = f"results_lowbudget_5fold_{target_type.value}.xlsx"

        save_path = f"{self.config.output_dir}/{filename}"

        with pd.ExcelWriter(save_path, engine='xlsxwriter') as writer:
            # Sheet 1: Summary
            summary_data = []
            for exp_name, result in aggregated_results.items():
                final_m = result['aggregated_metrics'][-1]
                summary_data.append({
                    'Strategy': exp_name,
                    'Target': target_type.value,
                    'AUBC_full': f"{result.get('aubc_mean', 0):.4f}±{result.get('aubc_std', 0):.4f}",
                    'AUBC_150': f"{result.get('aubc_budget_150_mean', 0):.4f}",
                    'AUBC_100': f"{result.get('aubc_budget_100_mean', 0):.4f}",
                    'Final_F1': f"{final_m.f1_macro_mean:.4f}±{final_m.f1_macro_std:.4f}",
                    'Final_BalAcc': f"{final_m.balanced_accuracy_mean:.4f}±{final_m.balanced_accuracy_std:.4f}",
                    'Final_P@50': f"{final_m.precision_at_k_mean.get(50, 0):.4f}",
                    'Final_P@100': f"{final_m.precision_at_k_mean.get(100, 0):.4f}",
                    'Final_P@200': f"{final_m.precision_at_k_mean.get(200, 0):.4f}",
                    'Final_P@300': f"{final_m.precision_at_k_mean.get(300, 0):.4f}",
                    'Pct_Oui_Total': f"{final_m.pct_yes_mean:.1f}%",
                    'Pct_Oui_AL': f"{final_m.al_pct_yes_mean:.1f}%"
                })

            df_summary = pd.DataFrame(summary_data)
            df_summary.to_excel(writer, sheet_name='Summary', index=False)

            # Sheet 2: Checkpoints
            df_checkpoints = self.generate_budget_checkpoint_table(aggregated_results, target_type)
            df_checkpoints.to_excel(writer, sheet_name='Budget_Checkpoints', index=False)

            # Detailed sheets
            for exp_name, result in aggregated_results.items():
                detail_data = []
                for agg_m in result['aggregated_metrics']:
                    row = {
                        'n_train': agg_m.n_train,
                        'F1_Macro': agg_m.f1_macro_mean,
                        'F1_Macro_std': agg_m.f1_macro_std,
                        'BalAcc': agg_m.balanced_accuracy_mean,
                        'Pct_Oui': agg_m.pct_yes_mean,
                        'AL_Pct_Oui': agg_m.al_pct_yes_mean,
                    }
                    for k in self.config.precision_at_k_values:
                        row[f'P@{k}'] = agg_m.precision_at_k_mean.get(k, 0)
                        row[f'NDCG@{k}'] = agg_m.ndcg_at_k_mean.get(k, 0)
                    detail_data.append(row)

                df_detail = pd.DataFrame(detail_data)
                sheet_name = exp_name[:31]
                df_detail.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"   💾 Excel saved: {save_path}")
        return save_path

    def generate_report_cv(
        self,
        aggregated_results: Dict[str, Dict],
        target_type: TargetType
    ) -> str:
        """Generate a limited-budget-focused report."""
        report = []
        report.append("="*70)
        report.append("REPORT - LIMITED-BUDGET ACTIVE LEARNING (5-FOLD CV)")
        report.append("="*70)
        report.append(f"\nDate: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        report.append(f"Target: {target_type.value}")
        report.append(f"Warm start: {self.config.warm_start_min}-{self.config.warm_start_max}")
        report.append(f"Batch size: {self.config.batch_size}")
        report.append(f"Folds: {self.config.n_folds}")

        report.append("\n" + "="*70)
        report.append("🏆 OVERALL RANKING (AUBC)")
        report.append("="*70)

        sorted_results = sorted(
            aggregated_results.items(),
            key=lambda x: x[1].get('aubc_mean', 0),
            reverse=True
        )

        for rank, (exp_name, result) in enumerate(sorted_results, 1):
            aubc = result.get('aubc_mean', 0)
            aubc_std = result.get('aubc_std', 0)
            final_m = result['aggregated_metrics'][-1]
            report.append(
                f"{rank}. {exp_name}: AUBC={aubc:.4f}±{aubc_std:.4f}, "
                f"F1={final_m.f1_macro_mean:.4f}"
            )

        report.append("\n" + "="*70)
        report.append("🎯 BEST STRATEGY PER BUDGET")
        report.append("="*70)

        for budget in self.config.budget_checkpoints:
            best_strategy = None
            best_f1 = 0

            for exp_name, result in aggregated_results.items():
                n_trains = result['n_train']
                closest_idx = np.argmin(np.abs(np.array(n_trains) - budget))
                if closest_idx < len(result['aggregated_metrics']):
                    m = result['aggregated_metrics'][closest_idx]
                    if m.f1_macro_mean > best_f1:
                        best_f1 = m.f1_macro_mean
                        best_strategy = exp_name

            if best_strategy:
                report.append(f"Budget ≈ {budget}: {best_strategy} (F1={best_f1:.4f})")

        report.append("\n" + "="*70)
        report.append("📊 ANNOTATION ANALYSIS")
        report.append("="*70)

        for exp_name, result in sorted_results[:4]:  # Top 4
            final_m = result['aggregated_metrics'][-1]
            report.append(f"\n{exp_name}:")
            report.append(f"  - Total: {final_m.pct_yes_mean:.1f}% Yes / {100-final_m.pct_yes_mean:.1f}% No")
            report.append(f"  - AL selections: {final_m.al_pct_yes_mean:.1f}% Yes")

        report_text = "\n".join(report)

        report_path = f"{self.config.output_dir}/report_{target_type.value}.txt"
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report_text)

        print(f"   💾 Report: {report_path}")
        print("\n" + report_text)

        return report_text


print("✅ ResultsVisualizer defined (extended for limited budget)!")


# =============================================================================
# SECTION 11: MAIN PIPELINE
# =============================================================================

def run_5fold_cv_pipeline(
    config: Config = None,
    target_type: TargetType = TargetType.GOLD,
    strategies: List[ALStrategy] = None,
    model_types: List[str] = None
) -> Dict[str, Dict]:
    """5-Fold CV pipeline optimised for limited budget."""

    if config is None:
        config = Config()

    if strategies is None:
        strategies = [
            ALStrategy.RANDOM,
            ALStrategy.ENTROPY,
            ALStrategy.HIGH_CONFIDENCE,
            ALStrategy.QUERY_BY_COMMITTEE
        ]

    if model_types is None:
        model_types = ["lr", "mlp"]

    print("\n" + "="*70)
    print("🚀 ACTIVE LEARNING - LIMITED BUDGET - 5-FOLD CV")
    print("="*70)
    print(f"   Target: {target_type.value}")
    print(f"   Warm start: {config.warm_start_min}-{config.warm_start_max}")
    print(f"   Batch size: {config.batch_size}")
    print(f"   Iterations: {config.n_iterations}")
    print(f"   → Estimated max budget: ~{config.warm_start_max + config.n_iterations * config.batch_size}")

    os.makedirs(config.output_dir, exist_ok=True)

    # Load data
    data_loader = DataLoader(config)
    data_loader.load_main_dataset()
    llm_dfs = data_loader.load_llm_predictions()
    data_loader.merge_predictions(llm_dfs)
    data_loader.create_target_labels()
    data_loader.set_target(target_type)
    df = data_loader.get_clean_dataset()

    print(f"\n✅ Dataset: {len(df)} examples")

    # Create folds
    cv_runner = CrossValidationRunner(config)
    folds = cv_runner.create_stratified_folds(df, strat_col='strat_key')

    # Feature engineering
    feature_engineer = FeatureEngineer(config, data_loader.llm_cols)
    _ = feature_engineer.compute_bm25_scores(df)

    # Run CV
    evaluator = Evaluator(config)
    warm_start_selector = WarmStartSelector(
        config, min_samples=config.warm_start_min, max_samples=config.warm_start_max
    )

    al_cv = ActiveLearnerCV(config, feature_engineer, evaluator, warm_start_selector)

    all_fold_results = {}

    for strategy in strategies:
        for model_type in model_types:
            exp_name = f"{strategy.value}_{model_type}"
            all_fold_results[exp_name] = []

            print(f"\n{'='*50}")
            print(f"   {exp_name}")
            print(f"{'='*50}")

            for fold_idx, (train_pool_idx, test_idx) in enumerate(folds):
                np.random.seed(config.random_seed + fold_idx)

                fold_history = al_cv.run_single_fold(
                    df=df,
                    train_pool_indices=train_pool_idx,
                    test_indices=test_idx,
                    llm_cols=data_loader.llm_cols,
                    strategy=strategy,
                    model_type=model_type,
                    fold_idx=fold_idx
                )

                all_fold_results[exp_name].append(fold_history)
                print(f"      Fold {fold_idx+1}: F1={fold_history['metrics'][-1].f1_macro:.4f}, "
                      f"AUBC={fold_history['aubc']:.4f}")

    # Aggregate
    print("\n" + "="*60)
    print("📊 AGGREGATION")
    print("="*60)

    aggregated_results = {}

    for exp_name, fold_histories in all_fold_results.items():
        n_iterations = len(fold_histories[0]['iterations'])
        aggregated_metrics_list = []
        n_train_list = fold_histories[0]['n_train']

        for iter_idx in range(n_iterations):
            iter_metrics = [fh['metrics'][iter_idx] for fh in fold_histories]
            agg_m = Evaluator.aggregate_metrics(iter_metrics, config)
            aggregated_metrics_list.append(agg_m)

        aubc_values = [fh['aubc'] for fh in fold_histories]
        aubc_150_values = [fh.get('aubc_budget_150', 0) for fh in fold_histories]
        aubc_100_values = [fh.get('aubc_budget_100', 0) for fh in fold_histories]

        aggregated_results[exp_name] = {
            'aggregated_metrics': aggregated_metrics_list,
            'n_train': n_train_list,
            'aubc_mean': np.mean(aubc_values),
            'aubc_std': np.std(aubc_values),
            'aubc_budget_150_mean': np.mean(aubc_150_values),
            'aubc_budget_100_mean': np.mean(aubc_100_values),
            'fold_histories': fold_histories
        }

        print(f"   {exp_name}: AUBC={np.mean(aubc_values):.4f}±{np.std(aubc_values):.4f}")

    # Visualize
    print("\n" + "="*60)
    print("📊 VISUALISATIONS")
    print("="*60)

    visualizer = ResultsVisualizer(config)

    # Learning curves (full)
    visualizer.plot_learning_curves_cv(
        aggregated_results, metric='f1_macro',
        title=f"Learning curves - {target_type.value}",
        save_name=f"learning_curves_full_{target_type.value}.png"
    )

    # Learning curves (budget limited)
    visualizer.plot_learning_curves_cv(
        aggregated_results, metric='f1_macro',
        title=f"Learning curves - {target_type.value}",
        save_name=f"learning_curves_budget150_{target_type.value}.png",
        max_budget=150
    )

    # Budget comparison
    visualizer.plot_budget_comparison(
        aggregated_results,
        save_name=f"budget_comparison_{target_type.value}.png"
    )

    # P@k extended
    visualizer.plot_precision_at_k_extended(
        aggregated_results,
        save_name=f"precision_at_k_{target_type.value}.png"
    )

    # Annotation distribution
    visualizer.plot_annotation_distribution(
        aggregated_results,
        save_name=f"annotation_dist_{target_type.value}.png"
    )

    # Checkpoint table
    visualizer.generate_budget_checkpoint_table(aggregated_results, target_type)

    # Save Excel
    visualizer.save_results_to_excel_cv(aggregated_results, target_type)

    # Report
    visualizer.generate_report_cv(aggregated_results, target_type)

    print("\n" + "="*70)
    print("✅ PIPELINE FINISHED!")
    print("="*70)
    print(f"   Results: {config.output_dir}")

    return aggregated_results


# =============================================================================
# SECTION 12: EXECUTION
# =============================================================================

if __name__ == "__main__":
    config = Config()

    # For a quick test:
    # config.n_iterations = 8
    # config.n_folds = 3

    results = run_5fold_cv_pipeline(
        config=config,
        target_type=TargetType.GOLD,
        strategies=[
            ALStrategy.RANDOM,
            ALStrategy.ENTROPY,
            ALStrategy.HIGH_CONFIDENCE,
            ALStrategy.QUERY_BY_COMMITTEE
        ],
        model_types=["lr"]
    )


# =============================================================================
# COLAB CELLS
# =============================================================================

"""
# CELL 1: Mount Drive

# CELL 2: Run
config = Config()
config.base_path = "."

results_gold = run_5fold_cv_pipeline(config=config, target_type=TargetType.GOLD)

# CELL 3: Compare targets
for target in [TargetType.A1, TargetType.A2]:
    run_5fold_cv_pipeline(config=config, target_type=target)
"""

# OPTUNA

In [ ]:
# ============================================================
# METHODOLOGICALLY STRONGER PIPELINE
# ============================================================
"""
Methodologically strengthened OOF pipeline:

A. CatBoostRanker with YetiRank/LambdaRank
   - Captures non-linearities (e.g. confident CE → LLM less important)
   - Industry standard for tabular reranking

B. Bayesian optimisation (Optuna)
   - Precise floating weights vs discrete grid
   - Fast and mathematically optimal convergence

C. Complete metrics: P@k, R@k, NDCG@k, MAP

MODELS COMPARED:
1. Random (baseline = prevalence)
2. Heuristic (Optuna-optimised)
3. Logistic Regression (linear baseline)
4. CatBoostRanker (YetiRank) - NEW
5. Cross-Encoder only (0-shot)
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Callable
from dataclasses import dataclass, field
from collections import defaultdict
import json

import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess

    packages = [
        ("optuna", "optuna"),
        ("catboost", "catboost"),
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]

    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    n_folds: int = 5
    random_seed: int = 42

    # Cross-Encoder config
    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    # Optuna config
    optuna_n_trials: int = 100
    optuna_timeout: Optional[int] = None  # seconds, None = no timeout
    optuna_sampler: str = "TPE"  # TPE, CMA-ES

    # CatBoost config
    catboost_iterations: int = 500
    catboost_learning_rate: float = 0.05
    catboost_depth: int = 6
    catboost_loss: str = "YetiRank"  # YetiRank, LambdaRank, PairLogit

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.output_dir = f"artifacts/outputs_catboost_optuna_oof"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def dcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        gains = y_sorted / np.log2(np.arange(2, k + 2))
        return float(np.sum(gains))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        dcg = Metrics.dcg_at_k(y_true, scores, k)
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:min(k, len(y_true))]
        k_actual = min(k, len(y_true))
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k_actual + 2))))
        if idcg == 0:
            return 0.0
        return dcg / idcg

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0

    @staticmethod
    def compute_all_metrics(y_true: np.ndarray, scores: np.ndarray, k_values: List[int]) -> Dict:
        """Compute all metrics at once."""
        results = {
            "AP": Metrics.average_precision(y_true, scores),
            "n_samples": len(y_true),
            "n_positives": int(np.sum(y_true))
        }
        for k in k_values:
            results[f"P@{k}"] = Metrics.precision_at_k(y_true, scores, k)
            results[f"R@{k}"] = Metrics.recall_at_k(y_true, scores, k)
            results[f"NDCG@{k}"] = Metrics.ndcg_at_k(y_true, scores, k)
        return results


# ============================================================
# DATA LOADING & FEATURE ENGINEERING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 3 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]

    for df in [df_main, df_llama, df_qwen, df_mistral]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)
    df = df.drop(columns=["_key"])

    return df




def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder BAAI/bge-reranker-v2-m3."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def compute_oof_crossencoder(df: pd.DataFrame, config: Config) -> np.ndarray:
    """
    Cross-encoder in OOF mode to avoid leakage.
    Each fold predicts only on its own test data.
    """
    print("\n" + "="*70)
    print("CROSS-ENCODER (OOF mode)")
    print("="*70)

    from sentence_transformers import CrossEncoder
    import torch

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device: {device}")

    # Load the model ONCE (0-shot, no fine-tuning)
    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_test = df.iloc[test_idx]

        # Prepare the pairs for the test fold
        texts = df_test["text"].fillna("").tolist()
        articles = df_test["article_text"].fillna("").tolist()
        pairs = [[t, a] for t, a in zip(texts, articles)]

        print(f"      Test: {len(pairs)} pairs")

        # Predict per batch
        scores = []
        batch_size = config.cross_encoder_batch_size

        for i in tqdm(range(0, len(pairs), batch_size),
                     desc=f"      Fold {fold_idx+1}",
                     leave=False):
            batch = pairs[i:i + batch_size]
            batch_scores = model.predict(batch, show_progress_bar=False)
            scores.extend(batch_scores)

        scores = np.array(scores)

        # Normalise per fold (important!)
        scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

        # Store in OOF
        oof_scores[test_idx] = scores_normalized

        print(f"      Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    print(f"\n  OOF Scores: min={oof_scores.min():.3f}, max={oof_scores.max():.3f}, mean={oof_scores.mean():.3f}")

    return oof_scores

def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all features."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION")
    print("="*70)

    df = df.copy()

    # LLM votes (unchanged)
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)

    # Aggregations (unchanged)
    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"]
    df["vote_union"] = ((df["vote_llama"] == 1) | (df["vote_qwen"] == 1) | (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)
    df["vote_inter3"] = (df["vote_sum"] == 3).astype(int)

    # OOF similarities
    print("  [1/3] TF-IDF + BM25 (OOF mode)...")
    df["sim_tfidf"], df["sim_bm25"] = compute_oof_tfidf_bm25(df, config)

    # ========================================
    # Cross-Encoder in OOF mode
    # ========================================
    if compute_crossencoder:
        print("  [2/3] Cross-Encoder (OOF mode)...")
        df["sim_crossencoder"] = compute_oof_crossencoder(df, config)
    else:
        print("  [2/3] Cross-Encoder: DISABLED")
        df["sim_crossencoder"] = 0.0

    # Interaction features
    df["ce_x_votesum"] = df["sim_crossencoder"] * df["vote_sum"]
    df["ce_x_tfidf"] = df["sim_crossencoder"] * df["sim_tfidf"]
    df["bm25_x_votesum"] = df["sim_bm25"] * df["vote_sum"]
    df["llm_disagreement"] = ((df["vote_sum"] > 0) & (df["vote_sum"] < 3)).astype(int)

    return df



# ============================================================
# A. HEURISTIC WITH OPTUNA OPTIMIZATION
# ============================================================
def compute_heuristic_score(df: pd.DataFrame, weights: Dict[str, float]) -> np.ndarray:
    """Parameterised heuristic score."""
    score = np.zeros(len(df))

    weight_map = {
        "w_llama": "vote_llama",
        "w_qwen": "vote_qwen",
        "w_mistral": "vote_mistral",
        "w_inter2": "vote_inter2",
        "w_inter3": "vote_inter3",
        "w_tfidf": "sim_tfidf",
        "w_bm25": "sim_bm25",
        "w_crossencoder": "sim_crossencoder"
    }

    for w_name, col_name in weight_map.items():
        if weights.get(w_name, 0) != 0 and col_name in df.columns:
            score += weights[w_name] * df[col_name].values

    return score


def optimize_heuristic_optuna(
    df_train: pd.DataFrame,
    config: Config,
    n_trials: int = 100
) -> Tuple[Dict[str, float], float]:
    """
    Bayesian optimisation of the heuristic weights with Optuna.
    """
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        weights = {
            "w_llama": trial.suggest_float("w_llama", 0.0, 1.0),
            "w_qwen": trial.suggest_float("w_qwen", 0.0, 1.0),
            "w_mistral": trial.suggest_float("w_mistral", 0.0, 1.0),
            "w_inter2": trial.suggest_float("w_inter2", 0.0, 2.0),
            "w_inter3": trial.suggest_float("w_inter3", 0.0, 3.0),
            "w_tfidf": trial.suggest_float("w_tfidf", 0.0, 0.5),
            "w_bm25": trial.suggest_float("w_bm25", 0.0, 0.5),
            "w_crossencoder": trial.suggest_float("w_crossencoder", 0.0, 1.5),
        }

        scores = compute_heuristic_score(df_train, weights)

        # Mean AP over all targets
        aps = []
        for target in config.targets:
            label_col = f"lbl_{target}"
            mask = df_train[label_col].notna()
            if mask.sum() == 0:
                continue
            y = df_train.loc[mask, label_col].values.astype(int)
            s = scores[mask.values]
            if np.sum(y) > 0:
                aps.append(Metrics.average_precision(y, s))

        return np.mean(aps) if aps else 0.0

    # Create the sampler
    if config.optuna_sampler == "CMA-ES":
        sampler = optuna.samplers.CmaEsSampler(seed=config.random_seed)
    else:  # TPE by default
        sampler = optuna.samplers.TPESampler(seed=config.random_seed)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        timeout=config.optuna_timeout,
        show_progress_bar=False
    )

    return study.best_params, study.best_value


def compute_oof_heuristic_optuna(
    df: pd.DataFrame,
    config: Config
) -> Tuple[np.ndarray, List[Dict]]:
    """OOF with Optuna optimisation per fold."""
    print("\n" + "="*70)
    print("HEURISTIC + OPTUNA OPTIMIZATION")
    print("="*70)

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        print(f"      Train: {len(train_idx)} | Test: {len(test_idx)}")

        # Optuna optimization
        best_weights, best_score = optimize_heuristic_optuna(
            df_train, config, n_trials=config.optuna_n_trials
        )

        print(f"      Best MAP (train): {best_score:.4f}")
        print(f"      Best weights: {json.dumps({k: round(v, 3) for k, v in best_weights.items()}, indent=8)}")

        # Score on test
        scores_test = compute_heuristic_score(df_test, best_weights)
        oof_scores[test_idx] = scores_test

        fold_info.append({
            "fold": fold_idx,
            "train_map": best_score,
            "weights": best_weights
        })

    return oof_scores, fold_info


# ============================================================
# B. CATBOOST RANKER
# ============================================================
def compute_oof_catboost_classifier(
    df: pd.DataFrame,
    config: Config,
    target_for_training: str = "Gold"
) -> Tuple[np.ndarray, List[Dict]]:
    """
    CatBoost Classifier (pointwise) - aligned with the global evaluation.

    Uses a pointwise model instead of ranking because:
    - The metrics evaluate a global ranking (not per decision)
    - Simpler and often better-performing for global top-k
    """
    print("\n" + "="*70)
    print("CATBOOST CLASSIFIER (pointwise)")
    print("="*70)

    from catboost import CatBoostClassifier

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    feature_cols = [
        "vote_llama", "vote_qwen", "vote_mistral",
        "vote_sum", "vote_inter2", "vote_inter3",
        "sim_tfidf", "sim_bm25", "sim_crossencoder",
        "ce_x_votesum", "ce_x_tfidf", "bm25_x_votesum",
        "llm_disagreement"
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]
    print(f"  Features ({len(feature_cols)}): {feature_cols}")

    label_col = f"lbl_{target_for_training}"

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        # Filter the valid labels
        train_mask = df_train[label_col].notna()
        df_train_valid = df_train[train_mask].copy()

        if len(df_train_valid) < 10:
            print(f"      [SKIP] Not enough labeled data: {len(df_train_valid)}")
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "skipped_insufficient_data"})
            continue

        # Normalisation
        normalizer = FeatureNormalizer(feature_cols, method='minmax')
        df_train_norm = normalizer.fit_transform(df_train_valid)
        df_test_norm = normalizer.transform(df_test)

        # Prepare the data
        X_train = df_train_norm[feature_cols].fillna(0).values
        y_train = df_train_norm[label_col].values.astype(int)

        X_test = df_test_norm[feature_cols].fillna(0).values

        n_pos = int(y_train.sum())
        n_neg = len(y_train) - n_pos

        print(f"      Training: {len(X_train)} samples")
        print(f"      Positives: {n_pos} ({100*n_pos/len(y_train):.1f}%)")

        # CatBoost Classifier
        model = CatBoostClassifier(
            iterations=config.catboost_iterations,
            learning_rate=config.catboost_learning_rate,
            depth=config.catboost_depth,
            random_seed=config.random_seed,
            verbose=False,
            task_type="CPU",
            auto_class_weights='Balanced'  # To handle class imbalance
        )

        try:
            model.fit(X_train, y_train)
        except Exception as e:
            print(f"      [ERROR] CatBoost training failed: {e}")
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "error", "error": str(e)})
            continue

        # Prediction (positive-class probabilities)
        scores_test = model.predict_proba(X_test)[:, 1]
        oof_scores[test_idx] = scores_test

        # Feature importance
        try:
            fi = model.get_feature_importance()
            if fi is not None and len(fi) == len(feature_cols):
                importance = dict(zip(feature_cols, fi))
                top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]
                print(f"      Top features: {top_features}")
            else:
                importance = {col: 0.0 for col in feature_cols}
        except Exception:
            importance = {col: 0.0 for col in feature_cols}

        # Evaluate on train
        train_pred = model.predict_proba(X_train)[:, 1]
        ap_train = Metrics.average_precision(y_train, train_pred)

        print(f"      Train AP: {ap_train:.4f}")

        fold_info.append({
            "fold": fold_idx,
            "train_ap": ap_train,
            "feature_importance": importance,
            "n_iterations": model.tree_count_,
            "n_pos": n_pos,
            "n_neg": n_neg
        })

    return oof_scores, fold_info

# ============================================================
# C. LOGISTIC REGRESSION (baseline)
# ============================================================
def compute_oof_lr(
    df: pd.DataFrame,
    config: Config,
    target_for_lr: str = "Gold"
) -> Tuple[np.ndarray, List[Dict]]:
    """Logistic Regression baseline."""
    print("\n" + "="*70)
    print("LOGISTIC REGRESSION (baseline)")
    print("="*70)

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    feature_cols = [
        "vote_llama", "vote_qwen", "vote_mistral",
        "sim_tfidf", "sim_bm25", "sim_crossencoder"
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]

    X_all = df[feature_cols].fillna(0).values
    label_col = f"lbl_{target_for_lr}"

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_train = df.iloc[train_idx]

        train_mask = df_train[label_col].notna()
        X_train = X_all[train_idx][train_mask.values]
        y_train = df_train.loc[train_mask, label_col].values.astype(int)

        if len(X_train) < 10 or len(np.unique(y_train)) < 2:
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "skipped"})
            continue

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)

        lr = LogisticRegression(
            C=1.0, class_weight="balanced",
            max_iter=1000, random_state=config.random_seed
        )
        lr.fit(X_train_scaled, y_train)

        X_test = X_all[test_idx]
        X_test_scaled = scaler.transform(X_test)
        proba_test = lr.predict_proba(X_test_scaled)[:, 1]
        oof_scores[test_idx] = proba_test

        proba_train = lr.predict_proba(X_train_scaled)[:, 1]
        ap_train = Metrics.average_precision(y_train, proba_train)

        print(f"      Train: {len(X_train)} | AP: {ap_train:.4f}")

        fold_info.append({
            "fold": fold_idx,
            "train_ap": ap_train,
            "coefs": dict(zip(feature_cols, lr.coef_[0]))
        })

    return oof_scores, fold_info


# ============================================================
# D. CROSS-ENCODER ONLY (0-shot)
# ============================================================
def compute_oof_crossencoder_only(df: pd.DataFrame) -> np.ndarray:
    """Cross-encoder only, no optimisation."""
    print("\n" + "="*70)
    print("CROSS-ENCODER ONLY (0-shot)")
    print("="*70)
    return df["sim_crossencoder"].values.copy()


# ============================================================
# RANDOM BASELINE
# ============================================================
def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """
    Compute the random baseline = positive rate (prevalence).
    For a random ranking, P@k = prevalence for all k.
    """
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples  # Yes rate = P@k for random

        for k in config.k_values:
            # For random: P@k = prevalence, R@k = min(k, n_pos) / n_pos
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,  # Approximation for random
                "count": int(expected_pos_at_k),
                "AP": prevalence,  # For random, AP ≈ prevalence
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# EVALUATION & REPORTING
# ============================================================
def evaluate_oof(
    df: pd.DataFrame,
    oof_scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate the OOF scores."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = oof_scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": Metrics.count_at_k(y, s, k),
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def print_comparison_table(
    results_dict: Dict[str, pd.DataFrame],
    config: Config,
    df_data: pd.DataFrame = None
):
    """Formatted comparison table with random baseline."""
    print("\n" + "="*80)
    print("COMPARISON OF ALL METHODS")
    print("="*80)

    # Ajouter random baseline
    if df_data is not None:
        random_baseline = compute_random_baseline(df_data, config)
        results_dict_full = {"Random": random_baseline, **results_dict}
    else:
        results_dict_full = results_dict

    methods = list(results_dict_full.keys())

    for target in config.targets:
        print(f"\n{'='*80}")
        print(f"TARGET: {target}")
        print(f"{'='*80}")

        # Get prevalence
        df_random = results_dict_full.get("Random")
        prevalence = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")

        # AP Summary
        print("\n  METHOD               | AP     | Δ vs Random")
        print("  " + "-"*45)
        for method, df_res in results_dict_full.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = (ap - prevalence) / prevalence * 100
                    delta_str = f"+{delta:.0f}%" if delta >= 0 else f"{delta:.0f}%"
                else:
                    delta_str = "-"
                print(f"  {method:<20} | {ap:.4f} | {delta_str}")

        # Detailed metrics table
        print(f"\n  {'k':<6}", end="")
        for method in methods:
            print(f" | {method[:10]:<10} {'Δ':<5}", end="")
        print()

        print("  " + "-"*90)

        for k in [10, 50, 100, 200, 300, 400, 500, 800]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method, df_res in results_dict_full.items():
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    if prevalence and method != "Random":
                        delta = (p - prevalence) / prevalence * 100
                        delta_str = f"+{delta:.0f}%" if delta >= 0 else f"{delta:.0f}%"
                    else:
                        delta_str = "-"
                    row += f" | {p:.3f}     {delta_str:<5}"
                else:
                    row += " | -         -    "
            print(row)

def plot_comparison(
    results_dict: Dict[str, pd.DataFrame],
    config: Config,
    output_path: Optional[str] = None
):
    """Multi-method visualisation with Random."""
    n_targets = len(config.targets)

    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))

    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'TF-IDF': 'blue',
        'BM25': 'cyan',
        'CrossEncoder': 'green',
        'LLM_Union': 'orange',
        'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple',
        'Heuristic': 'magenta'
    }

    linestyles = {
        'Random': '--',  # Dashed style for Random
    }

    for i, target in enumerate(config.targets):
        # P@k
        ax = axes[0, i]
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["P@k"],
                       color=colors.get(method, 'gray'),
                       linestyle=linestyles.get(method, '-'),  # Dashed line for Random
                       marker='o' if method != 'Random' else None,  # No marker for Random
                       linewidth=2 if method != 'Random' else 1.5,
                       label=method,
                       alpha=0.7 if method == 'Random' else 1.0)

        ax.set_xlabel("k")
        ax.set_ylabel("P@k")
        ax.set_title(f"{target} - Precision@k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)

        # NDCG@k
        ax = axes[1, i]
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["NDCG@k"],
                       color=colors.get(method, 'gray'),
                       linestyle=linestyles.get(method, '-'),  # Dashed line for Random
                       marker='o' if method != 'Random' else None,  # No marker for Random
                       linewidth=2 if method != 'Random' else 1.5,
                       label=method,
                       alpha=0.7 if method == 'Random' else 1.0)

        ax.set_xlabel("k")
        ax.set_ylabel("NDCG@k")
        ax.set_title(f"{target} - NDCG@k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")

    plt.show()

def compute_oof_tfidf_bm25(
    df: pd.DataFrame,
    config: Config
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute TF-IDF and BM25 in OOF mode to avoid leakage.
    """
    print("\n  Computing TF-IDF and BM25 in OOF mode...")

    from rank_bm25 import BM25Okapi

    n_samples = len(df)
    oof_tfidf = np.zeros(n_samples)
    oof_bm25 = np.zeros(n_samples)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    stopwords_fr = set([
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords_fr and len(t) > 2]

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"    Fold {fold_idx + 1}/{config.n_folds}...", end=" ")

        df_train = df.iloc[train_idx]
        df_test = df.iloc[test_idx]

        # ============ TF-IDF ============
        train_texts = df_train["text"].fillna("").tolist()
        train_articles = df_train["article_text"].fillna("").tolist()

        vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            stop_words=list(stopwords_fr),
            lowercase=True
        )
        vectorizer.fit(train_texts + train_articles)

        # Transform test
        test_texts = df_test["text"].fillna("").tolist()
        test_articles = df_test["article_text"].fillna("").tolist()

        tfidf_texts_test = vectorizer.transform(test_texts)
        tfidf_articles_test = vectorizer.transform(test_articles)

        for i, idx in enumerate(test_idx):
            sim = cosine_similarity(
                tfidf_texts_test[i],
                tfidf_articles_test[i]
            )[0, 0]
            oof_tfidf[idx] = sim

        # ============ BM25 ============
        # For BM25, we build a corpus with ALL the train articles
        train_articles_tokenized = [tokenize(a) for a in train_articles]

        # Create an article_text -> index mapping to find the right article
        train_article_to_idx = {a: i for i, a in enumerate(train_articles)}

        bm25 = BM25Okapi(train_articles_tokenized)

        # For each test example, we score its text vs its article
        for i, idx in enumerate(test_idx):
            query = tokenize(test_texts[i])
            target_article = test_articles[i]

            if not query:
                oof_bm25[idx] = 0.0
                continue

            # If the test article is in the train set (rare but possible),
            # utiliser son score direct
            if target_article in train_article_to_idx:
                all_scores = bm25.get_scores(query)
                oof_bm25[idx] = all_scores[train_article_to_idx[target_article]]
            else:
                # Otherwise, score vs the corpus and take the max as a proxy
                # (approximation needed since the article is not in the train set)
                all_scores = bm25.get_scores(query)
                oof_bm25[idx] = np.mean(all_scores)  # Mean as a proxy

        print("✓")

    # Normaliser
    if oof_bm25.max() > 0:
        oof_bm25 = oof_bm25 / oof_bm25.max()

    print(f"    TF-IDF: min={oof_tfidf.min():.3f}, max={oof_tfidf.max():.3f}, mean={oof_tfidf.mean():.3f}")
    print(f"    BM25:   min={oof_bm25.min():.3f}, max={oof_bm25.max():.3f}, mean={oof_bm25.mean():.3f}")

    return oof_tfidf, oof_bm25

# ============================================================
# MAIN
# ============================================================
def main(
    filter_union: bool = False,
    compute_crossencoder: bool = True,
    run_catboost: bool = True,
    optuna_trials: int = 100
):
    """
    Main pipeline.

    Args:
        filter_union: Filter on the LLM union
        compute_crossencoder: Compute the cross-encoder scores
        run_catboost: Run CatBoostRanker
        optuna_trials: Number of Optuna trials
    """
    print("="*70)
    print("METHODOLOGICALLY STRONGER PIPELINE")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print(f"  CatBoost: {run_catboost}")
    print(f"  Optuna trials: {optuna_trials}")
    print("="*70)

    # Install dependencies
    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    config.optuna_n_trials = optuna_trials

    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_methodological{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # 1. Load & prepare data
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    # 2. Filter if needed
    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # 3. Run all methods
    results_dict = {}
    all_fold_info = {}

    # A. Heuristic + Optuna
    oof_heur, fold_info_heur = compute_oof_heuristic_optuna(df, config)
    results_dict["Heuristic+Optuna"] = evaluate_oof(df, oof_heur, "Heuristic+Optuna", config)
    all_fold_info["heuristic"] = fold_info_heur

    # B. CatBoost Ranker
    if run_catboost:
        oof_cat, fold_info_cat = compute_oof_catboost_classifier(df, config)
        results_dict["CatBoost"] = evaluate_oof(df, oof_cat, "CatBoost", config)
        all_fold_info["catboost"] = fold_info_cat

    # C. Logistic Regression
    oof_lr, fold_info_lr = compute_oof_lr(df, config)
    results_dict["LR"] = evaluate_oof(df, oof_lr, "LR", config)
    all_fold_info["lr"] = fold_info_lr


    # D. Cross-Encoder only (separate evaluation, not as a feature)
    if compute_crossencoder:
        # IMPORTANT: The CE is already in df["sim_crossencoder"] in OOF mode
        # We evaluate it separately as a baseline
        oof_ce = df["sim_crossencoder"].values.copy()
        results_dict["CrossEncoder"] = evaluate_oof(df, oof_ce, "CrossEncoder", config)


    # 4. Comparison & reporting (with df for random baseline)
    print_comparison_table(results_dict, config, df_data=df)
    print_latex_table(results_dict, config, target="Gold", df_data=df)

    # 5. Visualization
    plot_comparison(results_dict, config,
                   output_path=f"{config.output_dir}/comparison.png",
                   df_data=df)

    # 6. Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    df["oof_heuristic_optuna"] = oof_heur
    df["oof_lr"] = oof_lr
    if run_catboost:
        df["oof_catboost"] = oof_cat
    if compute_crossencoder:
        df["oof_crossencoder"] = oof_ce

    df.to_csv(f"{config.output_dir}/df_with_oof_{timestamp}.csv", index=False)

    # Add random to the saved results
    random_baseline = compute_random_baseline(df, config)
    results_dict_full = {"Random": random_baseline, **results_dict}

    df_results_all = pd.concat(list(results_dict_full.values()), ignore_index=True)
    df_results_all.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    # Save fold info
    def convert_to_serializable(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_serializable(v) for v in obj]
        return obj

    with open(f"{config.output_dir}/fold_info_{timestamp}.json", "w") as f:
        json.dump(convert_to_serializable(all_fold_info), f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict_full, all_fold_info


if __name__ == "__main__":
    # Full pipeline with all the improvements
    df, results, fold_info = main(
        filter_union=False,
        compute_crossencoder=True,
        run_catboost=True,
        optuna_trials=100
    )

# OPTUNA BIS

In [ ]:
# ============================================================
# METHODOLOGICALLY STRONGER PIPELINE - V3 FIXED
# ============================================================
"""
Methodologically strengthened OOF pipeline - CORRECTED VERSION

FIXES APPLIED:
1. CatBoost Ranker:
   - Group diagnostics (size, label variance)
   - Filtering out invalid groups (size=1, constant labels)
   - Verification of the group_id ordering

2. Optuna:
   - Inner validation split instead of raw train
   - Avoids overfitting on the optimisation fold

3. Normalisation:
   - Consistent min-max scaling fit on train, applied on test
   - Features harmonised before combination

4. Consistent metrics:
   - Aligned optimisation and reporting
   - Display the gain in #positives (not just %)

MODELS COMPARED:
1. Random (baseline = prevalence)
2. Heuristic (Optuna-optimised with inner validation)
3. Logistic Regression (linear baseline)
4. CatBoostRanker (YetiRank) - CORRECTED
5. Cross-Encoder only (0-shot)
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Callable
from dataclasses import dataclass, field
from collections import defaultdict
import json

import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from catboost import CatBoostRanker, CatBoostClassifier, Pool

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess

    packages = [
        ("optuna", "optuna"),
        ("catboost", "catboost"),
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]

    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    n_folds: int = 5
    random_seed: int = 42

    # Cross-Encoder config
    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    # Optuna config
    optuna_n_trials: int = 100
    optuna_timeout: Optional[int] = None
    optuna_sampler: str = "TPE"
    optuna_inner_val_ratio: float = 0.2  # ratio for inner validation

    # CatBoost config
    catboost_iterations: int = 500
    catboost_learning_rate: float = 0.05
    catboost_depth: int = 6
    catboost_loss: str = "YetiRank"
    catboost_min_group_size: int = 2  # min group size
    catboost_require_mixed_labels: bool = True  # require pos+neg within a group

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.output_dir = f"artifacts/outputs_catboost_optuna_oof"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def dcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        gains = y_sorted / np.log2(np.arange(2, k + 2))
        return float(np.sum(gains))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        dcg = Metrics.dcg_at_k(y_true, scores, k)
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:min(k, len(y_true))]
        k_actual = min(k, len(y_true))
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k_actual + 2))))
        if idcg == 0:
            return 0.0
        return dcg / idcg

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0

    @staticmethod
    def compute_all_metrics(y_true: np.ndarray, scores: np.ndarray, k_values: List[int]) -> Dict:
        """Compute all metrics at once."""
        results = {
            "AP": Metrics.average_precision(y_true, scores),
            "n_samples": len(y_true),
            "n_positives": int(np.sum(y_true))
        }
        for k in k_values:
            results[f"P@{k}"] = Metrics.precision_at_k(y_true, scores, k)
            results[f"R@{k}"] = Metrics.recall_at_k(y_true, scores, k)
            results[f"NDCG@{k}"] = Metrics.ndcg_at_k(y_true, scores, k)
            results[f"count@{k}"] = Metrics.count_at_k(y_true, scores, k)
        return results


# ============================================================
# DATA LOADING & FEATURE ENGINEERING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 3 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]

    for df in [df_main, df_llama, df_qwen, df_mistral]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)
    df = df.drop(columns=["_key"])

    return df


# ============================================================
# DIAGNOSTIC FUNCTIONS
# ============================================================
def diagnose_groups_for_ranking(
    df: pd.DataFrame,
    group_col: str,
    label_col: str,
    verbose: bool = True
) -> Dict:
    """
    Full diagnostic of the groups for ranking.
    Return stats and identify the problems.
    """
    stats = {}

    # Filter the rows with valid labels
    df_valid = df[df[label_col].notna()].copy()

    if len(df_valid) == 0:
        return {"error": "No valid labels"}

    # Per-group stats
    group_stats = df_valid.groupby(group_col).agg(
        size=(label_col, 'count'),
        n_pos=(label_col, 'sum'),
        n_neg=(label_col, lambda x: (x == 0).sum()),
        label_variance=(label_col, 'var')
    ).reset_index()

    group_stats['has_mixed_labels'] = (group_stats['n_pos'] > 0) & (group_stats['n_neg'] > 0)

    # Global statistics
    stats['n_groups_total'] = len(group_stats)
    stats['n_samples_total'] = len(df_valid)

    # Size distribution
    sizes = group_stats['size'].values
    stats['group_size_min'] = int(sizes.min())
    stats['group_size_max'] = int(sizes.max())
    stats['group_size_median'] = float(np.median(sizes))
    stats['group_size_mean'] = float(sizes.mean())

    # Size-1 groups (useless for ranking)
    stats['n_groups_size_1'] = int((sizes == 1).sum())
    stats['pct_groups_size_1'] = 100 * stats['n_groups_size_1'] / stats['n_groups_total']

    # Groups with mixed labels (needed for learning)
    stats['n_groups_mixed_labels'] = int(group_stats['has_mixed_labels'].sum())
    stats['pct_groups_mixed_labels'] = 100 * stats['n_groups_mixed_labels'] / stats['n_groups_total']

    # Valid groups (size >= 2 AND mixed labels)
    valid_groups = group_stats[(group_stats['size'] >= 2) & (group_stats['has_mixed_labels'])]
    stats['n_groups_valid_for_ranking'] = len(valid_groups)
    stats['n_samples_in_valid_groups'] = int(valid_groups['size'].sum())
    stats['pct_groups_valid'] = 100 * stats['n_groups_valid_for_ranking'] / stats['n_groups_total']

    # List of valid groups
    stats['valid_group_ids'] = valid_groups[group_col].tolist()

    if verbose:
        print(f"\n  📊 GROUP DIAGNOSTICS ({label_col}):")
        print(f"     Total: {stats['n_groups_total']} groups, {stats['n_samples_total']} samples")
        print(f"     Size distribution: min={stats['group_size_min']}, median={stats['group_size_median']:.1f}, max={stats['group_size_max']}")
        print(f"     ⚠️  Groups with size=1: {stats['n_groups_size_1']} ({stats['pct_groups_size_1']:.1f}%)")
        print(f"     ⚠️  Groups with mixed labels: {stats['n_groups_mixed_labels']} ({stats['pct_groups_mixed_labels']:.1f}%)")
        print(f"     ✅ Valid for ranking (size≥2 AND mixed): {stats['n_groups_valid_for_ranking']} ({stats['pct_groups_valid']:.1f}%)")
        print(f"     ✅ Samples in valid groups: {stats['n_samples_in_valid_groups']}")

        if stats['pct_groups_valid'] < 50:
            print(f"     ❌ WARNING: Less than 50% valid groups - ranking may not learn properly!")

    return stats


def filter_valid_groups_for_ranking(
    df: pd.DataFrame,
    group_col: str,
    label_col: str,
    min_group_size: int = 2,
    require_mixed_labels: bool = True
) -> pd.DataFrame:
    """
    Filter the DataFrame to keep only the groups valid for ranking.
    """
    df_valid = df[df[label_col].notna()].copy()

    group_stats = df_valid.groupby(group_col).agg(
        size=(label_col, 'count'),
        n_pos=(label_col, 'sum'),
        n_neg=(label_col, lambda x: (x == 0).sum())
    ).reset_index()

    # Validity conditions
    valid_mask = group_stats['size'] >= min_group_size

    if require_mixed_labels:
        valid_mask &= (group_stats['n_pos'] > 0) & (group_stats['n_neg'] > 0)

    valid_groups = set(group_stats.loc[valid_mask, group_col].tolist())

    df_filtered = df_valid[df_valid[group_col].isin(valid_groups)].copy()

    return df_filtered


# ============================================================
# FEATURE NORMALIZATION
# ============================================================
class FeatureNormalizer:
    """
    Normalise features consistently between train and test.
    """
    def __init__(self, feature_cols: List[str], method: str = 'minmax'):
        self.feature_cols = feature_cols
        self.method = method
        self.scalers = {}
        self.is_fitted = False

    def fit(self, df: pd.DataFrame):
        """Fit on the training data."""
        for col in self.feature_cols:
            if col in df.columns:
                if self.method == 'minmax':
                    scaler = MinMaxScaler()
                else:
                    scaler = StandardScaler()

                values = df[col].fillna(0).values.reshape(-1, 1)
                scaler.fit(values)
                self.scalers[col] = scaler

        self.is_fitted = True
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform the data."""
        if not self.is_fitted:
            raise ValueError("Normalizer not fitted. Call fit() first.")

        df_out = df.copy()
        for col in self.feature_cols:
            if col in df_out.columns and col in self.scalers:
                values = df_out[col].fillna(0).values.reshape(-1, 1)
                df_out[col] = self.scalers[col].transform(values).flatten()

        return df_out

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit and transform in a single step."""
        self.fit(df)
        return self.transform(df)


# ============================================================
# OOF TF-IDF & BM25
# ============================================================
def compute_oof_tfidf_bm25(
    df: pd.DataFrame,
    config: Config
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute TF-IDF and BM25 in OOF mode to avoid leakage.
    """
    print("\n  Computing TF-IDF and BM25 in OOF mode...")

    from rank_bm25 import BM25Okapi

    n_samples = len(df)
    oof_tfidf = np.zeros(n_samples)
    oof_bm25 = np.zeros(n_samples)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    stopwords_fr = set([
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords_fr and len(t) > 2]

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"    Fold {fold_idx + 1}/{config.n_folds}...", end=" ")

        df_train = df.iloc[train_idx]
        df_test = df.iloc[test_idx]

        # ============ TF-IDF ============
        train_texts = df_train["text"].fillna("").tolist()
        train_articles = df_train["article_text"].fillna("").tolist()

        vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            stop_words=list(stopwords_fr),
            lowercase=True
        )
        vectorizer.fit(train_texts + train_articles)

        test_texts = df_test["text"].fillna("").tolist()
        test_articles = df_test["article_text"].fillna("").tolist()

        tfidf_texts_test = vectorizer.transform(test_texts)
        tfidf_articles_test = vectorizer.transform(test_articles)

        for i, idx in enumerate(test_idx):
            sim = cosine_similarity(
                tfidf_texts_test[i],
                tfidf_articles_test[i]
            )[0, 0]
            oof_tfidf[idx] = sim

        # ============ BM25 ============
        train_articles_tokenized = [tokenize(a) for a in train_articles]
        train_article_to_idx = {a: i for i, a in enumerate(train_articles)}

        bm25 = BM25Okapi(train_articles_tokenized)

        for i, idx in enumerate(test_idx):
            query = tokenize(test_texts[i])
            target_article = test_articles[i]

            if not query:
                oof_bm25[idx] = 0.0
                continue

            if target_article in train_article_to_idx:
                all_scores = bm25.get_scores(query)
                oof_bm25[idx] = all_scores[train_article_to_idx[target_article]]
            else:
                all_scores = bm25.get_scores(query)
                oof_bm25[idx] = np.mean(all_scores)

        print("✓")

    # Normaliser
    if oof_bm25.max() > 0:
        oof_bm25 = oof_bm25 / oof_bm25.max()

    print(f"    TF-IDF: min={oof_tfidf.min():.3f}, max={oof_tfidf.max():.3f}, mean={oof_tfidf.mean():.3f}")
    print(f"    BM25:   min={oof_bm25.min():.3f}, max={oof_bm25.max():.3f}, mean={oof_bm25.mean():.3f}")

    return oof_tfidf, oof_bm25


# ============================================================
# OOF CROSS-ENCODER
# ============================================================
def compute_oof_crossencoder(df: pd.DataFrame, config: Config) -> np.ndarray:
    """
    Cross-encoder in OOF mode to avoid leakage.
    """
    print("\n" + "="*70)
    print("CROSS-ENCODER (OOF mode)")
    print("="*70)

    from sentence_transformers import CrossEncoder
    import torch

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_test = df.iloc[test_idx]

        texts = df_test["text"].fillna("").tolist()
        articles = df_test["article_text"].fillna("").tolist()
        pairs = [[t, a] for t, a in zip(texts, articles)]

        print(f"      Test: {len(pairs)} pairs")

        scores = []
        batch_size = config.cross_encoder_batch_size

        for i in tqdm(range(0, len(pairs), batch_size),
                     desc=f"      Fold {fold_idx+1}",
                     leave=False):
            batch = pairs[i:i + batch_size]
            batch_scores = model.predict(batch, show_progress_bar=False)
            scores.extend(batch_scores)

        scores = np.array(scores)
        scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

        oof_scores[test_idx] = scores_normalized

        print(f"      Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    print(f"\n  OOF Scores: min={oof_scores.min():.3f}, max={oof_scores.max():.3f}, mean={oof_scores.mean():.3f}")

    return oof_scores


# ============================================================
# PREPARE FEATURES
# ============================================================
def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all features."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION")
    print("="*70)

    df = df.copy()

    # Votes LLM
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)

    # Aggregations
    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"]
    df["vote_union"] = ((df["vote_llama"] == 1) | (df["vote_qwen"] == 1) | (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)
    df["vote_inter3"] = (df["vote_sum"] == 3).astype(int)

    # OOF similarities
    print("  [1/3] TF-IDF + BM25 (OOF mode)...")
    df["sim_tfidf"], df["sim_bm25"] = compute_oof_tfidf_bm25(df, config)

    if compute_crossencoder:
        print("  [2/3] Cross-Encoder (OOF mode)...")
        df["sim_crossencoder"] = compute_oof_crossencoder(df, config)
    else:
        print("  [2/3] Cross-Encoder: DISABLED")
        df["sim_crossencoder"] = 0.0

    # Interaction features
    df["ce_x_votesum"] = df["sim_crossencoder"] * df["vote_sum"]
    df["ce_x_tfidf"] = df["sim_crossencoder"] * df["sim_tfidf"]
    df["bm25_x_votesum"] = df["sim_bm25"] * df["vote_sum"]
    df["llm_disagreement"] = ((df["vote_sum"] > 0) & (df["vote_sum"] < 3)).astype(int)

    return df


# ============================================================
# A. HEURISTIC WITH OPTUNA - CORRECTED
# ============================================================
def compute_heuristic_score(df: pd.DataFrame, weights: Dict[str, float]) -> np.ndarray:
    """Parameterised heuristic score."""
    score = np.zeros(len(df))

    weight_map = {
        "w_llama": "vote_llama",
        "w_qwen": "vote_qwen",
        "w_mistral": "vote_mistral",
        "w_inter2": "vote_inter2",
        "w_inter3": "vote_inter3",
        "w_tfidf": "sim_tfidf",
        "w_bm25": "sim_bm25",
        "w_crossencoder": "sim_crossencoder"
    }

    for w_name, col_name in weight_map.items():
        if weights.get(w_name, 0) != 0 and col_name in df.columns:
            score += weights[w_name] * df[col_name].values

    return score


def optimize_heuristic_optuna_with_inner_val(
    df_train: pd.DataFrame,
    config: Config,
    n_trials: int = 100
) -> Tuple[Dict[str, float], float, float]:
    """
    Bayesian optimisation with INNER VALIDATION SPLIT.

    FIX: We no longer validate on the raw train set but on an inner validation set.
    Retourne: (best_weights, best_inner_val_score, train_score)
    """
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # ========================================
    # INNER SPLIT: train -> train_sub + val_sub
    # ========================================
    groups_train = df_train["decision_id"].values
    unique_groups = df_train["decision_id"].unique()
    n_groups = len(unique_groups)

    # Split at the group level
    n_val_groups = max(1, int(n_groups * config.optuna_inner_val_ratio))
    np.random.seed(config.random_seed)
    val_group_ids = set(np.random.choice(unique_groups, size=n_val_groups, replace=False))

    val_mask = df_train["decision_id"].isin(val_group_ids)
    df_train_sub = df_train[~val_mask].copy()
    df_val_sub = df_train[val_mask].copy()

    print(f"      Inner split: train_sub={len(df_train_sub)}, val_sub={len(df_val_sub)} ({len(val_group_ids)} groups)")

    def objective(trial):
        weights = {
            "w_llama": trial.suggest_float("w_llama", 0.0, 1.0),
            "w_qwen": trial.suggest_float("w_qwen", 0.0, 1.0),
            "w_mistral": trial.suggest_float("w_mistral", 0.0, 1.0),
            "w_inter2": trial.suggest_float("w_inter2", 0.0, 2.0),
            "w_inter3": trial.suggest_float("w_inter3", 0.0, 3.0),
            "w_tfidf": trial.suggest_float("w_tfidf", 0.0, 0.5),
            "w_bm25": trial.suggest_float("w_bm25", 0.0, 0.5),
            "w_crossencoder": trial.suggest_float("w_crossencoder", 0.0, 1.5),
        }

        # ========================================
        # EVALUATE ON VAL_SUB (not train!)
        # ========================================
        scores_val = compute_heuristic_score(df_val_sub, weights)

        aps = []
        for target in config.targets:
            label_col = f"lbl_{target}"
            mask = df_val_sub[label_col].notna()
            if mask.sum() == 0:
                continue
            y = df_val_sub.loc[mask, label_col].values.astype(int)
            s = scores_val[mask.values]
            if np.sum(y) > 0:
                aps.append(Metrics.average_precision(y, s))

        return np.mean(aps) if aps else 0.0

    # Sampler
    if config.optuna_sampler == "CMA-ES":
        sampler = optuna.samplers.CmaEsSampler(seed=config.random_seed)
    else:
        sampler = optuna.samplers.TPESampler(seed=config.random_seed)

    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, timeout=config.optuna_timeout, show_progress_bar=False)

    best_weights = study.best_params
    best_val_score = study.best_value

    # Also compute the score on train_sub to diagnose overfitting
    scores_train = compute_heuristic_score(df_train_sub, best_weights)
    aps_train = []
    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df_train_sub[label_col].notna()
        if mask.sum() > 0:
            y = df_train_sub.loc[mask, label_col].values.astype(int)
            s = scores_train[mask.values]
            if np.sum(y) > 0:
                aps_train.append(Metrics.average_precision(y, s))
    train_score = np.mean(aps_train) if aps_train else 0.0

    return best_weights, best_val_score, train_score


def compute_oof_heuristic_optuna(
    df: pd.DataFrame,
    config: Config
) -> Tuple[np.ndarray, List[Dict]]:
    """OOF with Optuna optimisation + inner validation per fold."""
    print("\n" + "="*70)
    print("HEURISTIC + OPTUNA (with inner validation)")
    print("="*70)

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        print(f"      Outer split: train={len(train_idx)}, test={len(test_idx)}")

        # Optuna with inner validation
        best_weights, best_val_score, train_score = optimize_heuristic_optuna_with_inner_val(
            df_train, config, n_trials=config.optuna_n_trials
        )

        print(f"      Best inner val MAP: {best_val_score:.4f} (train: {train_score:.4f})")

        # Detect potential overfitting
        if train_score > 0 and best_val_score > 0:
            overfit_ratio = train_score / best_val_score
            if overfit_ratio > 1.3:
                print(f"      ⚠️  Possible overfit: train/val ratio = {overfit_ratio:.2f}")

        print(f"      Weights: {json.dumps({k: round(v, 3) for k, v in best_weights.items()})}")

        # Score on test (outer fold)
        scores_test = compute_heuristic_score(df_test, best_weights)
        oof_scores[test_idx] = scores_test

        fold_info.append({
            "fold": fold_idx,
            "inner_val_map": best_val_score,
            "train_map": train_score,
            "weights": best_weights
        })

    return oof_scores, fold_info


# ============================================================
# B. CATBOOST RANKER - CORRECTED
# ============================================================
def compute_oof_catboost_ranker(
    df: pd.DataFrame,
    config: Config,
    target_for_training: str = "Gold"
) -> Tuple[np.ndarray, List[Dict]]:
    """
    CatBoostRanker with YetiRank - CORRECTED VERSION.

    CORRECTIONS:
    1. Group diagnostics before training
    2. Filtering out invalid groups (size=1, constant labels)
    3. Verification of the contiguity of the group_id
    4. Feature normalisation
    """
    print("\n" + "="*70)
    print(f"CATBOOST RANKER ({config.catboost_loss}) - CORRECTED")
    print("="*70)

    from catboost import CatBoostRanker, Pool

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    feature_cols = [
        "vote_llama", "vote_qwen", "vote_mistral",
        "vote_sum", "vote_inter2", "vote_inter3",
        "sim_tfidf", "sim_bm25", "sim_crossencoder",
        "ce_x_votesum", "ce_x_tfidf", "bm25_x_votesum",
        "llm_disagreement"
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]
    print(f"  Features ({len(feature_cols)}): {feature_cols}")

    label_col = f"lbl_{target_for_training}"

    # ========================================
    # DIAGNOSTIC GLOBAL
    # ========================================
    print("\n  === DIAGNOSTIC GLOBAL ===")
    global_stats = diagnose_groups_for_ranking(df, "decision_id", label_col, verbose=True)

    if global_stats.get('pct_groups_valid', 0) < 20:
        print("\n  ❌ CRITICAL: Less than 20% valid groups. CatBoost ranking will likely fail!")
        print("     Consider: (1) more data, (2) different grouping, (3) switch to pointwise model")

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        df_train["_original_idx"] = train_idx
        df_test["_original_idx"] = test_idx

        # ========================================
        # FILTER VALID GROUPS FOR RANKING
        # ========================================
        print(f"      Before filtering: {len(df_train)} samples")

        df_train_valid = filter_valid_groups_for_ranking(
            df_train,
            group_col="decision_id",
            label_col=label_col,
            min_group_size=config.catboost_min_group_size,
            require_mixed_labels=config.catboost_require_mixed_labels
        )

        print(f"      After filtering (valid groups): {len(df_train_valid)} samples")

        if len(df_train_valid) < 100:
            print(f"      [SKIP] Not enough valid data after filtering: {len(df_train_valid)}")
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "skipped_insufficient_data"})
            continue

        # Fold diagnostic
        fold_stats = diagnose_groups_for_ranking(df_train_valid, "decision_id", label_col, verbose=False)
        print(f"      Valid groups for training: {fold_stats['n_groups_valid_for_ranking']}")

        # ========================================
        # SORT BY GROUP_ID (CRITICAL for CatBoost)
        # ========================================
        df_train_valid = df_train_valid.sort_values("decision_id").reset_index(drop=True)

        # Check contiguity
        group_ids = df_train_valid["decision_id"].values
        is_contiguous = all(
            group_ids[i] == group_ids[i-1] or group_ids[i] != group_ids[i+1] if i < len(group_ids)-1 else True
            for i in range(1, len(group_ids))
        )

        # Simpler method: check that each group appears as a contiguous block
        seen_groups = set()
        last_group = None
        is_truly_contiguous = True
        for g in group_ids:
            if g != last_group:
                if g in seen_groups:
                    is_truly_contiguous = False
                    break
                seen_groups.add(g)
                last_group = g

        if not is_truly_contiguous:
            print("      ⚠️  WARNING: Groups are not contiguous after sorting!")
        else:
            print("      ✓ Groups are contiguous")

        # ========================================
        # FEATURE NORMALISATION
        # ========================================
        normalizer = FeatureNormalizer(feature_cols, method='minmax')
        df_train_valid = normalizer.fit_transform(df_train_valid)
        df_test_normalized = normalizer.transform(df_test)

        # Prepare the data
        X_train = df_train_valid[feature_cols].fillna(0).values
        y_train = df_train_valid[label_col].values.astype(int)
        group_ids_train = df_train_valid["decision_id"].values

        X_test = df_test_normalized[feature_cols].fillna(0).values
        original_test_idx = df_test["_original_idx"].values

        n_groups = len(df_train_valid["decision_id"].unique())
        print(f"      Training: {len(X_train)} samples, {n_groups} groups")
        print(f"      Positives: {y_train.sum()} ({100*y_train.mean():.1f}%)")

        # ========================================
        # FINAL GROUP CHECK
        # ========================================
        # Distribution stats within groups
        group_df = pd.DataFrame({
            'group': group_ids_train,
            'label': y_train
        })
        group_agg = group_df.groupby('group').agg(
            size=('label', 'count'),
            sum_labels=('label', 'sum')
        )

        groups_with_pos = (group_agg['sum_labels'] > 0).sum()
        groups_with_neg = (group_agg['sum_labels'] < group_agg['size']).sum()
        groups_mixed = ((group_agg['sum_labels'] > 0) & (group_agg['sum_labels'] < group_agg['size'])).sum()

        print(f"      Groups breakdown: {groups_with_pos} with positives, {groups_with_neg} with negatives, {groups_mixed} mixed")

        # ========================================
        # CATBOOST TRAINING
        # ========================================
        train_pool = Pool(
            data=X_train,
            label=y_train,
            feature_names=feature_cols
        )
        # Drop group_id entirely and use a pointwise model
        model = CatBoostClassifier(  # ou CatBoostRegressor
            iterations=config.catboost_iterations,
            learning_rate=config.catboost_learning_rate,
            depth=config.catboost_depth,
            random_seed=config.random_seed,
            verbose=False,
            task_type="CPU"
        )

        # No need for Pool with group_id
        model.fit(X_train, y_train)

        try:
            model.fit(train_pool)
        except Exception as e:
            print(f"      [ERROR] CatBoost training failed: {e}")
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "error", "error": str(e)})
            continue

        # Prediction
        test_pool = Pool(data=X_test, feature_names=feature_cols)
        scores_test = model.predict(test_pool)

        for i, orig_idx in enumerate(original_test_idx):
            oof_scores[orig_idx] = scores_test[i]

        # Feature importance
        try:
            fi = model.get_feature_importance()
            if fi is not None and len(fi) == len(feature_cols):
                importance = dict(zip(feature_cols, fi))
            else:
                importance = {col: 0.0 for col in feature_cols}
        except Exception:
            importance = {col: 0.0 for col in feature_cols}

        # Check whether all importances are 0
        if all(v == 0.0 for v in importance.values()):
            print("      ⚠️  WARNING: All feature importances are 0.0!")
            print("         This usually means the model didn't learn anything useful.")
            print("         Possible causes: (1) all groups have constant labels, (2) insufficient data")
        else:
            top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]
            print(f"      Top features: {top_features}")

        # Evaluate on train
        train_pred = model.predict(train_pool)
        ap_train = Metrics.average_precision(y_train, train_pred)

        print(f"      Train AP: {ap_train:.4f}")

        fold_info.append({
            "fold": fold_idx,
            "train_ap": ap_train,
            "feature_importance": importance,
            "n_iterations": model.tree_count_,
            "n_valid_groups": n_groups,
            "n_mixed_groups": groups_mixed
        })

    return oof_scores, fold_info


# ============================================================
# C. LOGISTIC REGRESSION (with normalization)
# ============================================================
def compute_oof_lr(
    df: pd.DataFrame,
    config: Config,
    target_for_lr: str = "Gold"
) -> Tuple[np.ndarray, List[Dict]]:
    """Logistic Regression with consistent normalisation."""
    print("\n" + "="*70)
    print("LOGISTIC REGRESSION (baseline)")
    print("="*70)

    n_samples = len(df)
    oof_scores = np.zeros(n_samples)

    feature_cols = [
        "vote_llama", "vote_qwen", "vote_mistral",
        "sim_tfidf", "sim_bm25", "sim_crossencoder"
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]

    label_col = f"lbl_{target_for_lr}"

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")

        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        train_mask = df_train[label_col].notna()
        df_train_labeled = df_train[train_mask]

        X_train = df_train_labeled[feature_cols].fillna(0).values
        y_train = df_train_labeled[label_col].values.astype(int)

        if len(X_train) < 10 or len(np.unique(y_train)) < 2:
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "skipped"})
            continue

        # Normalisation
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)

        lr = LogisticRegression(
            C=1.0, class_weight="balanced",
            max_iter=1000, random_state=config.random_seed
        )
        lr.fit(X_train_scaled, y_train)

        X_test = df_test[feature_cols].fillna(0).values
        X_test_scaled = scaler.transform(X_test)
        proba_test = lr.predict_proba(X_test_scaled)[:, 1]
        oof_scores[test_idx] = proba_test

        proba_train = lr.predict_proba(X_train_scaled)[:, 1]
        ap_train = Metrics.average_precision(y_train, proba_train)

        print(f"      Train: {len(X_train)} | AP: {ap_train:.4f}")
        print(f"      Coefs: {dict(zip(feature_cols, lr.coef_[0].round(3)))}")

        fold_info.append({
            "fold": fold_idx,
            "train_ap": ap_train,
            "coefs": dict(zip(feature_cols, lr.coef_[0]))
        })

    return oof_scores, fold_info


# ============================================================
# RANDOM BASELINE
# ============================================================
def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """
    Compute the random baseline = positive rate (prevalence).
    """
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count@k": int(expected_pos_at_k),
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# EVALUATION & REPORTING
# ============================================================
def evaluate_oof(
    df: pd.DataFrame,
    oof_scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate the OOF scores with extended metrics."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = oof_scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            count_at_k = Metrics.count_at_k(y, s, k)
            expected_random = int(min(k, n_samples) * prevalence)

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count@k": count_at_k,
                "count_vs_random": count_at_k - expected_random,  # gain in #positives
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def print_comparison_table(
    results_dict: Dict[str, pd.DataFrame],
    config: Config,
    df_data: pd.DataFrame = None
):
    """Comparison table with gain in #positives."""
    print("\n" + "="*80)
    print("COMPARISON OF ALL METHODS")
    print("="*80)

    if df_data is not None:
        random_baseline = compute_random_baseline(df_data, config)
        results_dict_full = {"Random": random_baseline, **results_dict}
    else:
        results_dict_full = results_dict

    methods = list(results_dict_full.keys())

    for target in config.targets:
        print(f"\n{'='*80}")
        print(f"TARGET: {target}")
        print(f"{'='*80}")

        # Get prevalence
        df_random = results_dict_full.get("Random")
        prevalence = None
        n_pos_total = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                n_pos_total = df_t_rand["n_positives"].iloc[0]
                print(f"  Prevalence: {prevalence:.3f} ({prevalence*100:.1f}%)")
                print(f"  Total positives: {n_pos_total}")

        # AP Summary
        print("\n  METHOD               | AP     | Δ% vs Random")
        print("  " + "-"*50)
        for method, df_res in results_dict_full.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = (ap - prevalence) / prevalence * 100
                    delta_str = f"+{delta:.0f}%" if delta >= 0 else f"{delta:.0f}%"
                else:
                    delta_str = "-"
                print(f"  {method:<20} | {ap:.4f} | {delta_str}")

        # Detailed table with count gains
        print(f"\n  {'k':<6}", end="")
        for method in methods:
            if method == "Random":
                print(f" | {'Random':^10}", end="")
            else:
                print(f" | {method[:8]:^8} {'Δ#':^5}", end="")
        print()

        print("  " + "-"*100)

        for k in [50, 100, 200, 300, 400, 500]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"

            # Get random expected count
            random_count = 0
            df_rand_k = results_dict_full.get("Random")
            if df_rand_k is not None:
                df_rand_k_t = df_rand_k[(df_rand_k["target"] == target) & (df_rand_k["k"] == k)]
                if len(df_rand_k_t) > 0:
                    random_count = int(df_rand_k_t["count@k"].values[0])

            for method, df_res in results_dict_full.items():
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    if method == "Random":
                        row += f" | {p:.3f} ({random_count:>3})"
                    else:
                        count = int(df_t["count@k"].values[0]) if "count@k" in df_t.columns else int(p * k)
                        delta_count = count - random_count
                        delta_str = f"+{delta_count}" if delta_count >= 0 else f"{delta_count}"
                        row += f" | {p:.3f}   {delta_str:>4}"
                else:
                    row += " | -          "
            print(row)


def print_latex_table(
    results_dict: Dict[str, pd.DataFrame],
    config: Config,
    target: str = "Gold",
    df_data: pd.DataFrame = None
):
    """LaTeX table with deltas in #positives."""
    print(f"\n{'='*70}")
    print(f"LATEX TABLE (Target: {target})")
    print("="*70)

    if df_data is not None:
        random_baseline = compute_random_baseline(df_data, config)
        results_dict_with_random = {"Random": random_baseline, **results_dict}
    else:
        results_dict_with_random = results_dict

    methods = list(results_dict_with_random.keys())

    prevalence = None
    if "Random" in results_dict_with_random:
        df_random = results_dict_with_random["Random"]
        df_random_t = df_random[df_random["target"] == target]
        if len(df_random_t) > 0:
            prevalence = df_random_t["prevalence"].iloc[0]

    n_methods = len(methods)

    print("""
\\begin{table}[htbp]
\\centering
\\caption{Comparison of ranking methods - """ + target + """ (OOF evaluation)}
\\label{tab:ranking_comparison_""" + target.lower() + """}
\\resizebox{\\textwidth}{!}{%
\\begin{tabular}{l""" + "c" * (n_methods * 2 - 1) + """}
\\toprule
""")

    header = "k"
    for method in methods:
        header += f" & {method}"
        if method != "Random":
            header += f" & $\\Delta$\\#"
    header += " \\\\"
    print(header)
    print("\\midrule")

    k_values_display = [50, 100, 200, 300, 400, 500]

    print("\\multicolumn{" + str(n_methods * 2) + "}{l}{\\textit{P@k (count@k)}} \\\\")
    for k in k_values_display:
        if k not in config.k_values:
            continue

        # Get random count for this k
        random_count = 0
        if "Random" in results_dict_with_random:
            df_rand = results_dict_with_random["Random"]
            df_rand_k = df_rand[(df_rand["target"] == target) & (df_rand["k"] == k)]
            if len(df_rand_k) > 0:
                random_count = int(df_rand_k["count@k"].values[0])

        row = f"{k}"
        for method, df_res in results_dict_with_random.items():
            df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
            if len(df_t) > 0:
                p = df_t["P@k"].values[0]
                count = int(df_t["count@k"].values[0]) if "count@k" in df_t.columns else int(p * k)
                row += f" & {p:.3f}"
                if method != "Random":
                    delta = count - random_count
                    row += f" & +{delta}" if delta >= 0 else f" & {delta}"
            else:
                row += " & -"
                if method != "Random":
                    row += " & -"
        row += " \\\\"
        print(row)

    print("\\midrule")

    # MAP row
    row = "MAP"
    for method, df_res in results_dict_with_random.items():
        df_t = df_res[df_res["target"] == target]
        if len(df_t) > 0:
            ap = df_t["AP"].iloc[0]
            row += f" & {ap:.3f}"
            if method != "Random" and prevalence:
                delta = (ap - prevalence) / prevalence * 100 if prevalence > 0 else 0
                row += f" & +{delta:.0f}\\%" if delta >= 0 else f" & {delta:.0f}\\%"
        else:
            row += " & -"
            if method != "Random":
                row += " & -"
    row += " \\\\"
    print(row)

    print("""\\bottomrule
\\end{tabular}%
}
\\end{table}
""")

    if prevalence:
        print(f"\n% Note: Random baseline (prevalence) = {prevalence:.3f} ({prevalence*100:.1f}%)")


def plot_comparison(
    results_dict: Dict[str, pd.DataFrame],
    config: Config,
    output_path: Optional[str] = None,
    df_data: pd.DataFrame = None
):
    """Multi-method visualisation with Random."""

    if df_data is not None:
        random_baseline = compute_random_baseline(df_data, config)
        results_dict_full = {"Random": random_baseline, **results_dict}
    else:
        results_dict_full = results_dict

    n_targets = len(config.targets)

    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))

    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'Heuristic+Optuna': 'magenta',
        'CatBoost': 'red',
        'LR': 'blue',
        'CrossEncoder': 'green',
    }

    linestyles = {
        'Random': '--',
    }

    for i, target in enumerate(config.targets):
        # P@k
        ax = axes[0, i]
        for method, df_res in results_dict_full.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["P@k"],
                       color=colors.get(method, 'gray'),
                       linestyle=linestyles.get(method, '-'),
                       marker='o' if method != 'Random' else None,
                       linewidth=2 if method != 'Random' else 1.5,
                       label=method,
                       alpha=0.7 if method == 'Random' else 1.0)

        ax.set_xlabel("k")
        ax.set_ylabel("P@k")
        ax.set_title(f"{target} - Precision@k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)

        # NDCG@k
        ax = axes[1, i]
        for method, df_res in results_dict_full.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["NDCG@k"],
                       color=colors.get(method, 'gray'),
                       linestyle=linestyles.get(method, '-'),
                       marker='o' if method != 'Random' else None,
                       linewidth=2 if method != 'Random' else 1.5,
                       label=method,
                       alpha=0.7 if method == 'Random' else 1.0)

        ax.set_xlabel("k")
        ax.set_ylabel("NDCG@k")
        ax.set_title(f"{target} - NDCG@k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")

    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(
    filter_union: bool = False,
    compute_crossencoder: bool = True,
    run_catboost: bool = True,
    optuna_trials: int = 100
):
    """
    Main pipeline - CORRECTED VERSION.
    """
    print("="*70)
    print("METHODOLOGICALLY STRONGER PIPELINE - V3 FIXED")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print(f"  CatBoost: {run_catboost}")
    print(f"  Optuna trials: {optuna_trials}")
    print("="*70)

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    config.optuna_n_trials = optuna_trials

    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_methodological_v3{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # 1. Load & prepare data
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    # 2. Filter if needed
    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # 3. Run all methods
    results_dict = {}
    all_fold_info = {}

    # A. Heuristic + Optuna (with inner validation)
    oof_heur, fold_info_heur = compute_oof_heuristic_optuna(df, config)
    results_dict["Heuristic+Optuna"] = evaluate_oof(df, oof_heur, "Heuristic+Optuna", config)
    all_fold_info["heuristic"] = fold_info_heur

    # B. CatBoost Ranker (corrected)
    if run_catboost:
        oof_cat, fold_info_cat = compute_oof_catboost_ranker(df, config)
        results_dict["CatBoost"] = evaluate_oof(df, oof_cat, "CatBoost", config)
        all_fold_info["catboost"] = fold_info_cat

    # C. Logistic Regression
    oof_lr, fold_info_lr = compute_oof_lr(df, config)
    results_dict["LR"] = evaluate_oof(df, oof_lr, "LR", config)
    all_fold_info["lr"] = fold_info_lr

    # D. Cross-Encoder only
    if compute_crossencoder:
        oof_ce = df["sim_crossencoder"].values.copy()
        results_dict["CrossEncoder"] = evaluate_oof(df, oof_ce, "CrossEncoder", config)

    # 4. Comparison & reporting
    print_comparison_table(results_dict, config, df_data=df)

    # 5. Visualization
    plot_comparison(results_dict, config,
                   output_path=f"{config.output_dir}/comparison.png",
                   df_data=df)

    # 6. Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    df["oof_heuristic_optuna"] = oof_heur
    df["oof_lr"] = oof_lr
    if run_catboost:
        df["oof_catboost"] = oof_cat
    if compute_crossencoder:
        df["oof_crossencoder"] = oof_ce

    df.to_csv(f"{config.output_dir}/df_with_oof_{timestamp}.csv", index=False)

    random_baseline = compute_random_baseline(df, config)
    results_dict_full = {"Random": random_baseline, **results_dict}

    df_results_all = pd.concat(list(results_dict_full.values()), ignore_index=True)
    df_results_all.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    def convert_to_serializable(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_serializable(v) for v in obj]
        return obj

    with open(f"{config.output_dir}/fold_info_{timestamp}.json", "w") as f:
        json.dump(convert_to_serializable(all_fold_info), f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict_full, all_fold_info


if __name__ == "__main__":
    df, results, fold_info = main(
        filter_union=False,
        compute_crossencoder=True,
        run_catboost=True,
        optuna_trials=100
    )

# Optuna chati

In [ ]:
# ============================================================
# METHODOLOGICALLY STRONGER PIPELINE - V3 FIXED (FULLY CORRECTED)
# ============================================================
"""
Main fixes applied vs the original script:

1) CatBoost
   - Replacement of CatBoostRanker (group-wise) with CatBoostClassifier (pointwise)
     => aligned with the global evaluation (AP/P@k over the whole dataset).
   - OOF training grouped by decision_id, without group_id ranking.

2) Optuna
   - Group-wise inner validation split kept (no validation on raw train).
   - Objective based on mean AP over {A1,A2,Gold} on val_sub.

3) Normalisation
   - Consistent MinMaxScaler train->test for the methods that combine scores
     (CatBoostClassifier and Heuristic do not strictly require it, but it stabilises).

4) Bugs / robustness
   - LogisticRegression: random_state (not random_seed)
   - BM25: handling of duplicate articles (article_text) via a list-of-indices mapping
   - Random baseline: consistent count@k and Δ#positives (NDCG baseline left as a proxy)

Outputs:
- comparative tables
- plot
- CSV + JSON fold-info exports
"""

import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

from tqdm import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    import subprocess

    packages = [
        ("optuna", "optuna"),
        ("catboost", "catboost"),
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]
    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    n_folds: int = 5
    random_seed: int = 42

    # Cross-Encoder config
    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    # Optuna config
    optuna_n_trials: int = 2000
    optuna_timeout: Optional[int] = None
    optuna_sampler: str = "TPE"
    optuna_inner_val_ratio: float = 0.2

    # CatBoost (pointwise classifier)
    catboost_iterations: int = 800
    catboost_learning_rate: float = 0.05
    catboost_depth: int = 6
    catboost_loss: str = "Logloss"

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.output_dir = f"artifacts/outputs_methodological_v3_fixed"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = int(np.sum(y_true))
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def dcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        gains = y_sorted / np.log2(np.arange(2, k + 2))
        return float(np.sum(gains))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        dcg = Metrics.dcg_at_k(y_true, scores, k)
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:k]
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k + 2))))
        if idcg == 0:
            return 0.0
        return float(dcg / idcg)

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    print("\n" + "=" * 70)
    print("LOADING DATA")
    print("=" * 70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]
    for df in [df_main, df_llama, df_qwen, df_mistral]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)
    df = df.drop(columns=["_key"])

    return df


# ============================================================
# FEATURE NORMALIZATION
# ============================================================
class FeatureNormalizer:
    def __init__(self, feature_cols: List[str], method: str = "minmax"):
        self.feature_cols = feature_cols
        self.method = method
        self.scalers: Dict[str, object] = {}
        self.is_fitted = False

    def fit(self, df: pd.DataFrame):
        for col in self.feature_cols:
            if col not in df.columns:
                continue
            scaler = MinMaxScaler() if self.method == "minmax" else StandardScaler()
            values = df[col].fillna(0).values.reshape(-1, 1)
            scaler.fit(values)
            self.scalers[col] = scaler
        self.is_fitted = True
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if not self.is_fitted:
            raise ValueError("Normalizer not fitted. Call fit() first.")
        df_out = df.copy()
        for col in self.feature_cols:
            if col in df_out.columns and col in self.scalers:
                values = df_out[col].fillna(0).values.reshape(-1, 1)
                df_out[col] = self.scalers[col].transform(values).flatten()
        return df_out

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)


# ============================================================
# OOF TF-IDF & BM25
# ============================================================
def compute_oof_tfidf_bm25(df: pd.DataFrame, config: Config) -> Tuple[np.ndarray, np.ndarray]:
    print("\n  Computing TF-IDF and BM25 in OOF mode...")
    from rank_bm25 import BM25Okapi

    n_samples = len(df)
    oof_tfidf = np.zeros(n_samples, dtype=float)
    oof_bm25 = np.zeros(n_samples, dtype=float)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    stopwords_fr = set([
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r"\b[a-zàâäéèêëïîôùûüç]+\b", str(text).lower())
        return [t for t in tokens if t not in stopwords_fr and len(t) > 2]

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"    Fold {fold_idx + 1}/{config.n_folds}...", end=" ")

        df_train = df.iloc[train_idx]
        df_test = df.iloc[test_idx]

        # TF-IDF
        train_texts = df_train["text"].fillna("").tolist()
        train_articles = df_train["article_text"].fillna("").tolist()

        vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            stop_words=list(stopwords_fr),
            lowercase=True,
        )
        vectorizer.fit(train_texts + train_articles)

        test_texts = df_test["text"].fillna("").tolist()
        test_articles = df_test["article_text"].fillna("").tolist()

        tfidf_texts_test = vectorizer.transform(test_texts)
        tfidf_articles_test = vectorizer.transform(test_articles)

        for i, idx in enumerate(test_idx):
            sim = cosine_similarity(tfidf_texts_test[i], tfidf_articles_test[i])[0, 0]
            oof_tfidf[idx] = float(sim)

        # BM25
        train_articles_tokenized = [tokenize(a) for a in train_articles]
        bm25 = BM25Okapi(train_articles_tokenized)

        # FIX: doublons article_text -> liste d'indices
        art_to_indices: Dict[str, List[int]] = {}
        for j, a in enumerate(train_articles):
            art_to_indices.setdefault(a, []).append(j)

        for i, idx in enumerate(test_idx):
            query = tokenize(test_texts[i])
            target_article = test_articles[i]

            if not query:
                oof_bm25[idx] = 0.0
                continue

            all_scores = bm25.get_scores(query)

            if target_article in art_to_indices:
                inds = art_to_indices[target_article]
                oof_bm25[idx] = float(np.mean([all_scores[j] for j in inds]))
            else:
                # fallback: mean score if the article is absent from the train
                oof_bm25[idx] = float(np.mean(all_scores))

        print("✓")

    if float(oof_bm25.max()) > 0:
        oof_bm25 = oof_bm25 / float(oof_bm25.max())

    print(f"    TF-IDF: min={oof_tfidf.min():.3f}, max={oof_tfidf.max():.3f}, mean={oof_tfidf.mean():.3f}")
    print(f"    BM25:   min={oof_bm25.min():.3f}, max={oof_bm25.max():.3f}, mean={oof_bm25.mean():.3f}")

    return oof_tfidf, oof_bm25


# ============================================================
# OOF CROSS-ENCODER
# ============================================================
def compute_oof_crossencoder(df: pd.DataFrame, config: Config) -> np.ndarray:
    print("\n" + "=" * 70)
    print("CROSS-ENCODER (OOF mode)")
    print("=" * 70)

    from sentence_transformers import CrossEncoder
    import torch

    n_samples = len(df)
    oof_scores = np.zeros(n_samples, dtype=float)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device,
    )

    for fold_idx, (_, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")
        df_test = df.iloc[test_idx]

        texts = df_test["text"].fillna("").tolist()
        articles = df_test["article_text"].fillna("").tolist()
        pairs = [[t, a] for t, a in zip(texts, articles)]
        print(f"      Test: {len(pairs)} pairs")

        scores = []
        bs = config.cross_encoder_batch_size
        for i in tqdm(range(0, len(pairs), bs), desc=f"      Fold {fold_idx+1}", leave=False):
            batch = pairs[i : i + bs]
            batch_scores = model.predict(batch, show_progress_bar=False)
            scores.extend(batch_scores)

        scores = np.array(scores, dtype=float)
        scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

        oof_scores[test_idx] = scores_norm
        print(f"      Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    print(f"\n  OOF Scores: min={oof_scores.min():.3f}, max={oof_scores.max():.3f}, mean={oof_scores.mean():.3f}")
    return oof_scores


# ============================================================
# PREPARE FEATURES
# ============================================================
def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    print("\n" + "=" * 70)
    print("FEATURE PREPARATION")
    print("=" * 70)

    df = df.copy()

    # Votes LLM
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)

    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"]
    df["vote_union"] = ((df["vote_sum"] >= 1)).astype(int)
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)
    df["vote_inter3"] = (df["vote_sum"] == 3).astype(int)

    # OOF similarities
    print("  [1/3] TF-IDF + BM25 (OOF mode)...")
    df["sim_tfidf"], df["sim_bm25"] = compute_oof_tfidf_bm25(df, config)

    if compute_crossencoder:
        print("  [2/3] Cross-Encoder (OOF mode)...")
        df["sim_crossencoder"] = compute_oof_crossencoder(df, config)
    else:
        print("  [2/3] Cross-Encoder: DISABLED")
        df["sim_crossencoder"] = 0.0

    # Interactions
    df["ce_x_votesum"] = df["sim_crossencoder"] * df["vote_sum"]
    df["ce_x_tfidf"] = df["sim_crossencoder"] * df["sim_tfidf"]
    df["bm25_x_votesum"] = df["sim_bm25"] * df["vote_sum"]
    df["llm_disagreement"] = ((df["vote_sum"] > 0) & (df["vote_sum"] < 3)).astype(int)

    return df


# ============================================================
# HEURISTIC (Optuna + inner val)
# ============================================================
def compute_heuristic_score(df: pd.DataFrame, weights: Dict[str, float]) -> np.ndarray:
    score = np.zeros(len(df), dtype=float)
    weight_map = {
        "w_llama": "vote_llama",
        "w_qwen": "vote_qwen",
        "w_mistral": "vote_mistral",
        "w_inter2": "vote_inter2",
        "w_inter3": "vote_inter3",
        "w_tfidf": "sim_tfidf",
        "w_bm25": "sim_bm25",
        "w_crossencoder": "sim_crossencoder",
    }
    for w_name, col in weight_map.items():
        w = float(weights.get(w_name, 0.0))
        if w != 0.0 and col in df.columns:
            score += w * df[col].fillna(0).values
    return score


def normalize_weights(weights: Dict[str, float], eps: float = 1e-12) -> Dict[str, float]:
    s = sum(abs(v) for v in weights.values())
    if s < eps:
        return weights
    return {k: float(v) / s for k, v in weights.items()}


def optimize_heuristic_optuna_with_inner_val(
    df_train: pd.DataFrame,
    config: Config,
    n_trials: int,
) -> Tuple[Dict[str, float], float, float]:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    unique_groups = df_train["decision_id"].unique()
    n_groups = len(unique_groups)
    n_val_groups = max(1, int(n_groups * config.optuna_inner_val_ratio))

    rng = np.random.RandomState(config.random_seed)
    val_group_ids = set(rng.choice(unique_groups, size=n_val_groups, replace=False))

    val_mask = df_train["decision_id"].isin(val_group_ids)
    df_train_sub = df_train[~val_mask].copy()
    df_val_sub = df_train[val_mask].copy()

    print(f"      Inner split: train_sub={len(df_train_sub)}, val_sub={len(df_val_sub)} ({len(val_group_ids)} groups)")

    def objective(trial):
      grid_01 = [round(x, 2) for x in np.linspace(0.0, 1.0, 11)]  # 0.0..1.0 step 0.1
      grid_inter2 = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5, 1.7, 2.0]
      grid_inter3 = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5, 1.7, 2.0, 2.2, 2.5, 2.7, 3.0]
      grid_ce = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6]

      weights = {
          "w_llama": trial.suggest_categorical("w_llama", grid_01),
          "w_qwen": trial.suggest_categorical("w_qwen", grid_01),
          "w_mistral": trial.suggest_categorical("w_mistral", grid_01),
          "w_inter2": trial.suggest_categorical("w_inter2", grid_inter2),
          "w_inter3": trial.suggest_categorical("w_inter3", grid_inter3),
          "w_tfidf": trial.suggest_categorical("w_tfidf", grid_01),
          "w_bm25": trial.suggest_categorical("w_bm25", grid_01),
          "w_crossencoder": trial.suggest_categorical("w_crossencoder", grid_ce),
      }

      weights = normalize_weights(weights)  # <<< key
      scores_val = compute_heuristic_score(df_val_sub, weights)

      aps = []
      for target in config.targets:
          label_col = f"lbl_{target}"
          mask = df_val_sub[label_col].notna()
          if mask.sum() == 0:
              continue
          y = df_val_sub.loc[mask, label_col].values.astype(int)
          s = scores_val[mask.values]
          if int(np.sum(y)) > 0:
              aps.append(Metrics.average_precision(y, s))

      return float(np.mean(aps)) if aps else 0.0

    sampler = optuna.samplers.TPESampler(seed=config.random_seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, timeout=config.optuna_timeout, show_progress_bar=False)

    best_weights = study.best_params
    best_val_score = float(study.best_value)

    # Train-sub score (diagnostic)
    scores_train = compute_heuristic_score(df_train_sub, best_weights)
    aps_train = []
    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df_train_sub[label_col].notna()
        if mask.sum() == 0:
            continue
        y = df_train_sub.loc[mask, label_col].values.astype(int)
        s = scores_train[mask.values]
        if int(np.sum(y)) > 0:
            aps_train.append(Metrics.average_precision(y, s))
    train_score = float(np.mean(aps_train)) if aps_train else 0.0

    return best_weights, best_val_score, train_score


def compute_oof_heuristic_optuna(df: pd.DataFrame, config: Config) -> Tuple[np.ndarray, List[Dict]]:
    print("\n" + "=" * 70)
    print("HEURISTIC + OPTUNA (inner validation)")
    print("=" * 70)

    n_samples = len(df)
    oof_scores = np.zeros(n_samples, dtype=float)

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []
    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")
        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        print(f"      Outer split: train={len(train_idx)}, test={len(test_idx)}")

        best_weights, best_val_score, train_score = optimize_heuristic_optuna_with_inner_val(
            df_train, config, n_trials=config.optuna_n_trials
        )

        print(f"      Best inner val MAP: {best_val_score:.4f} (train_sub: {train_score:.4f})")
        if train_score > 0 and best_val_score > 0 and (train_score / best_val_score) > 1.3:
            print(f"      ⚠️  Possible overfit: train/val ratio={train_score/best_val_score:.2f}")

        print(f"      Weights: {json.dumps({k: round(v, 3) for k, v in best_weights.items()})}")

        oof_scores[test_idx] = compute_heuristic_score(df_test, best_weights)

        fold_info.append({
            "fold": fold_idx,
            "inner_val_map": best_val_score,
            "train_sub_map": train_score,
            "weights": best_weights,
        })

    return oof_scores, fold_info


# ============================================================
# CATBOOST (POINTWISE) - OOF, GROUPED SPLITS
# ============================================================
def compute_oof_catboost_pointwise(
    df: pd.DataFrame,
    config: Config,
    target_for_training: str = "Gold",
) -> Tuple[np.ndarray, List[Dict]]:
    """
    CatBoostClassifier OOF, splits grouped by decision_id.
    IMPORTANT: pointwise model (aligned with global AP/P@k metrics).
    """
    print("\n" + "=" * 70)
    print("CATBOOST (pointwise classifier) - OOF")
    print("=" * 70)

    from catboost import CatBoostClassifier

    n_samples = len(df)
    oof_scores = np.zeros(n_samples, dtype=float)

    feature_cols = [
        "vote_llama", "vote_qwen", "vote_mistral",
        "vote_sum", "vote_inter2", "vote_inter3",
        "sim_tfidf", "sim_bm25", "sim_crossencoder",
        "ce_x_votesum", "ce_x_tfidf", "bm25_x_votesum",
        "llm_disagreement",
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]
    print(f"  Features ({len(feature_cols)}): {feature_cols}")

    label_col = f"lbl_{target_for_training}"

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []
    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")
        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        # Train only on labelled rows for target_for_training
        train_mask = df_train[label_col].notna()
        df_train_labeled = df_train[train_mask].copy()

        if len(df_train_labeled) < 50 or df_train_labeled[label_col].nunique() < 2:
            print(f"      [SKIP] Not enough labeled data or single class for {label_col}.")
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "skipped"})
            continue

        # Min-max normalisation (stabilises, optional for CatBoost)
        normalizer = FeatureNormalizer(feature_cols, method="minmax")
        df_train_labeled = normalizer.fit_transform(df_train_labeled)
        df_test_norm = normalizer.transform(df_test)

        X_train = df_train_labeled[feature_cols].fillna(0).values
        y_train = df_train_labeled[label_col].values.astype(int)

        X_test = df_test_norm[feature_cols].fillna(0).values

        model = CatBoostClassifier(
            loss_function=config.catboost_loss,
            iterations=config.catboost_iterations,
            learning_rate=config.catboost_learning_rate,
            depth=config.catboost_depth,
            random_seed=config.random_seed,
            verbose=False,
            task_type="CPU",
        )
        model.fit(X_train, y_train)

        proba_test = model.predict_proba(X_test)[:, 1]
        oof_scores[test_idx] = proba_test

        proba_train = model.predict_proba(X_train)[:, 1]
        ap_train = Metrics.average_precision(y_train, proba_train)
        print(f"      Train labeled: {len(X_train)} | AP(train): {ap_train:.4f}")

        try:
            fi = model.get_feature_importance()
            top = sorted(zip(feature_cols, fi), key=lambda x: x[1], reverse=True)[:5]
            print(f"      Top features: {[(k, round(v, 3)) for k, v in top]}")
            importance = dict(zip(feature_cols, [float(x) for x in fi]))
        except Exception:
            importance = {c: 0.0 for c in feature_cols}

        fold_info.append({
            "fold": fold_idx,
            "train_ap": ap_train,
            "feature_importance": importance,
            "n_train_labeled": int(len(X_train)),
        })

    return oof_scores, fold_info


# ============================================================
# LOGISTIC REGRESSION (OOF)
# ============================================================
def compute_oof_lr(
    df: pd.DataFrame,
    config: Config,
    target_for_lr: str = "Gold",
) -> Tuple[np.ndarray, List[Dict]]:
    print("\n" + "=" * 70)
    print("LOGISTIC REGRESSION (baseline) - OOF")
    print("=" * 70)

    n_samples = len(df)
    oof_scores = np.zeros(n_samples, dtype=float)

    feature_cols = [
        "vote_llama", "vote_qwen", "vote_mistral",
        "sim_tfidf", "sim_bm25", "sim_crossencoder",
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]
    label_col = f"lbl_{target_for_lr}"

    groups = df["decision_id"].values
    gkf = GroupKFold(n_splits=config.n_folds)

    fold_info = []
    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        print(f"\n  --- FOLD {fold_idx + 1}/{config.n_folds} ---")
        df_train = df.iloc[train_idx].copy()
        df_test = df.iloc[test_idx].copy()

        train_mask = df_train[label_col].notna()
        df_train_labeled = df_train[train_mask].copy()

        X_train = df_train_labeled[feature_cols].fillna(0).values
        y_train = df_train_labeled[label_col].values.astype(int)

        if len(X_train) < 50 or len(np.unique(y_train)) < 2:
            print("      [SKIP] Not enough labeled data or single class.")
            oof_scores[test_idx] = 0.5
            fold_info.append({"fold": fold_idx, "status": "skipped"})
            continue

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)

        lr = LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=1000,
            random_state=config.random_seed,  # FIX
        )
        lr.fit(X_train_scaled, y_train)

        X_test = df_test[feature_cols].fillna(0).values
        X_test_scaled = scaler.transform(X_test)
        proba_test = lr.predict_proba(X_test_scaled)[:, 1]
        oof_scores[test_idx] = proba_test

        proba_train = lr.predict_proba(X_train_scaled)[:, 1]
        ap_train = Metrics.average_precision(y_train, proba_train)
        print(f"      Train labeled: {len(X_train)} | AP(train): {ap_train:.4f}")
        print(f"      Coefs: {dict(zip(feature_cols, lr.coef_[0].round(3)))}")

        fold_info.append({
            "fold": fold_idx,
            "train_ap": ap_train,
            "coefs": dict(zip(feature_cols, [float(x) for x in lr.coef_[0]])),
            "n_train_labeled": int(len(X_train)),
        })

    return oof_scores, fold_info


# ============================================================
# RANDOM BASELINE
# ============================================================
def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    results = []
    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples if n_samples > 0 else 0.0

        for k in config.k_values:
            k_eff = min(k, n_samples)
            expected_pos_at_k = k_eff * prevalence
            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": (expected_pos_at_k / n_pos) if n_pos > 0 else 0.0,
                # Exact expected random NDCG is non-trivial; we keep a consistent proxy (prevalence)
                "NDCG@k": prevalence,
                "count@k": int(round(expected_pos_at_k)),
                "count_vs_random": 0,
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence,
            })
    return pd.DataFrame(results)


# ============================================================
# EVALUATION
# ============================================================
def evaluate_oof(df: pd.DataFrame, oof_scores: np.ndarray, method_name: str, config: Config) -> pd.DataFrame:
    results = []
    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = oof_scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples if n_samples > 0 else 0.0
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            k_eff = min(k, n_samples)
            count_at_k = Metrics.count_at_k(y, s, k_eff)
            expected_random = int(round(k_eff * prevalence))

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k_eff),
                "R@k": Metrics.recall_at_k(y, s, k_eff),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k_eff),
                "count@k": int(count_at_k),
                "count_vs_random": int(count_at_k - expected_random),
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence,
            })

    return pd.DataFrame(results)


# ============================================================
# REPORTING
# ============================================================
def print_comparison_table(results_dict: Dict[str, pd.DataFrame], config: Config, df_data: pd.DataFrame):
    print("\n" + "=" * 80)
    print("COMPARISON OF ALL METHODS")
    print("=" * 80)

    random_baseline = compute_random_baseline(df_data, config)
    results_dict_full = {"Random": random_baseline, **results_dict}

    methods = list(results_dict_full.keys())

    for target in config.targets:
        print(f"\n{'='*80}")
        print(f"TARGET: {target}")
        print(f"{'='*80}")

        df_rand = results_dict_full["Random"]
        df_t_rand = df_rand[df_rand["target"] == target]
        if len(df_t_rand) == 0:
            continue

        prevalence = float(df_t_rand["prevalence"].iloc[0])
        n_pos_total = int(df_t_rand["n_positives"].iloc[0])
        print(f"  Prevalence: {prevalence:.3f} ({prevalence*100:.1f}%)")
        print(f"  Total positives: {n_pos_total}")

        print("\n  METHOD               | AP     | ΔAP (abs)")
        print("  " + "-" * 50)
        for method, df_res in results_dict_full.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) == 0:
                continue
            ap = float(df_t["AP"].iloc[0])
            if method == "Random":
                delta_str = "-"
            else:
                delta_str = f"{(ap - prevalence):+.4f}"
            print(f"  {method:<20} | {ap:.4f} | {delta_str}")

        # Detailed table P@k + Δ#positives
        ks_show = [10, 50, 100, 200, 300, 500, 800, 1000]
        ks_show = [k for k in ks_show if k in config.k_values]

        print(f"\n  {'k':<6}", end="")
        for method in methods:
            if method == "Random":
                print(f" | {'Random':^14}", end="")
            else:
                print(f" | {method[:10]:^10} {'Δ#':^3}", end="")
        print()
        print("  " + "-" * 110)

        for k in ks_show:
            row = f"  {k:<6}"

            df_rand_k = df_t_rand[df_t_rand["k"] == k]
            random_count = int(df_rand_k["count@k"].values[0]) if len(df_rand_k) else 0
            random_p = float(df_rand_k["P@k"].values[0]) if len(df_rand_k) else 0.0
            row += f" | {random_p:.3f} ({random_count:>4})"

            for method, df_res in results_dict_full.items():
                if method == "Random":
                    continue
                df_tk = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_tk) == 0:
                    row += " | -"
                    continue
                p = float(df_tk["P@k"].values[0])
                delta_count = int(df_tk["count_vs_random"].values[0])
                row += f" | {p:.3f} {delta_count:>+3d}"
            print(row)


def plot_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: str, df_data: pd.DataFrame):
    random_baseline = compute_random_baseline(df_data, config)
    results_full = {"Random": random_baseline, **results_dict}

    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    for i, target in enumerate(config.targets):
        # P@k
        ax = axes[0, i]
        for method, df_res in results_full.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) == 0:
                continue
            ax.plot(df_t["k"], df_t["P@k"], marker="o", linewidth=2 if method != "Random" else 1.5, alpha=0.7 if method == "Random" else 1.0, label=method)
        ax.set_title(f"{target} - Precision@k")
        ax.set_xlabel("k")
        ax.set_ylabel("P@k")
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)
        ax.legend(fontsize=7, loc="best")

        # NDCG@k
        ax = axes[1, i]
        for method, df_res in results_full.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) == 0:
                continue
            ax.plot(df_t["k"], df_t["NDCG@k"], marker="o", linewidth=2 if method != "Random" else 1.5, alpha=0.7 if method == "Random" else 1.0, label=method)
        ax.set_title(f"{target} - NDCG@k")
        ax.set_xlabel("k")
        ax.set_ylabel("NDCG@k")
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)
        ax.legend(fontsize=7, loc="best")

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(
    filter_union: bool = False,
    compute_crossencoder: bool = True,
    run_catboost: bool = True,
    optuna_trials: int = 100,
):
    print("=" * 70)
    print("METHODOLOGICALLY STRONGER PIPELINE - V3 FIXED (FULLY CORRECTED)")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print(f"  CatBoost (pointwise): {run_catboost}")
    print(f"  Optuna trials: {optuna_trials}")
    print("=" * 70)

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    config.optuna_n_trials = optuna_trials

    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_methodological_v3_fixed{suffix}"
    os.makedirs(config.output_dir, exist_ok=True)

    np.random.seed(config.random_seed)

    # Load & features
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    # Filter union if requested
    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    results_dict: Dict[str, pd.DataFrame] = {}
    all_fold_info: Dict[str, object] = {}

    # A) Heuristic+Optuna
    oof_heur, fold_info_heur = compute_oof_heuristic_optuna(df, config)
    results_dict["Heuristic+Optuna"] = evaluate_oof(df, oof_heur, "Heuristic+Optuna", config)
    all_fold_info["heuristic"] = fold_info_heur

    # B) CatBoost pointwise
    if run_catboost:
        oof_cat, fold_info_cat = compute_oof_catboost_pointwise(df, config, target_for_training="Gold")
        results_dict["CatBoost"] = evaluate_oof(df, oof_cat, "CatBoost", config)
        all_fold_info["catboost"] = fold_info_cat

    # C) LR
    oof_lr, fold_info_lr = compute_oof_lr(df, config, target_for_lr="Gold")
    results_dict["LR"] = evaluate_oof(df, oof_lr, "LR", config)
    all_fold_info["lr"] = fold_info_lr

    # D) Cross-Encoder seul
    if compute_crossencoder:
        oof_ce = df["sim_crossencoder"].values.copy()
        results_dict["CrossEncoder"] = evaluate_oof(df, oof_ce, "CrossEncoder", config)

    # Reporting
    print_comparison_table(results_dict, config, df_data=df)
    plot_comparison(
        results_dict,
        config,
        output_path=f"{config.output_dir}/comparison.png",
        df_data=df,
    )

    # Save outputs
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    df["oof_heuristic_optuna"] = oof_heur
    df["oof_lr"] = oof_lr
    if run_catboost:
        df["oof_catboost"] = oof_cat
    if compute_crossencoder:
        df["oof_crossencoder"] = oof_ce

    df.to_csv(f"{config.output_dir}/df_with_oof_{timestamp}.csv", index=False)

    random_baseline = compute_random_baseline(df, config)
    results_full = {"Random": random_baseline, **results_dict}
    df_results_all = pd.concat(list(results_full.values()), ignore_index=True)
    df_results_all.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    def convert(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        if isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        if isinstance(obj, dict):
            return {k: convert(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [convert(v) for v in obj]
        return obj

    with open(f"{config.output_dir}/fold_info_{timestamp}.json", "w") as f:
        json.dump(convert(all_fold_info), f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")
    return df, results_full, all_fold_info


if __name__ == "__main__":
    df, results, fold_info = main(
        filter_union=False,
        compute_crossencoder=True,
        run_catboost=True,
        optuna_trials=100,
    )


# Unsupervised

In [ ]:
# ============================================================
# FULLY UNSUPERVISED PIPELINE - NO LABELS USED
# ============================================================
"""
Fully unsupervised pipeline for legal ranking:

METHODS COMPARED (without using the labels):
1. Random (baseline = random ordering)
2. TF-IDF only
3. BM25 only
4. Cross-Encoder only (0-shot)
5. LLM Union (at least 1 LLM votes yes)
6. LLM Intersection (2+ LLMs vote yes)
7. Fixed heuristic (predefined weights, no optimisation)

IMPORTANT: We evaluate on the labels AFTERWARDS, but we never use them
to build the ranking scores.
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
import json

import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess

    packages = [
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]

    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    n_folds: int = 5
    random_seed: int = 42

    # Cross-Encoder config
    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.output_dir = f"artifacts/outputs_unsupervised"


# ============================================================
# METRICS (for evaluation only, not for training)
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def dcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        gains = y_sorted / np.log2(np.arange(2, k + 2))
        return float(np.sum(gains))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        dcg = Metrics.dcg_at_k(y_true, scores, k)
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:min(k, len(y_true))]
        k_actual = min(k, len(y_true))
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k_actual + 2))))
        if idcg == 0:
            return 0.0
        return dcg / idcg

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 3 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]

    for df in [df_main, df_llama, df_qwen, df_mistral]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    # Keep the labels only for evaluation
    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)
    df = df.drop(columns=["_key"])

    return df


# ============================================================
# UNSUPERVISED FEATURES (NO LABELS USED)
# ============================================================
def compute_tfidf_similarity(df: pd.DataFrame) -> np.ndarray:
    """TF-IDF similarity (computed over the whole dataset since unsupervised)."""
    stopwords_fr = [
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ]

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2,
        max_df=0.95, stop_words=stopwords_fr, lowercase=True
    )
    vectorizer.fit(texts + articles)

    tfidf_texts = vectorizer.transform(texts)
    tfidf_articles = vectorizer.transform(articles)

    return np.array([
        cosine_similarity(tfidf_texts[i], tfidf_articles[i])[0, 0]
        for i in range(len(texts))
    ])


def compute_bm25_scores(df: pd.DataFrame) -> np.ndarray:
    """BM25 scores (computed over the whole dataset since unsupervised)."""
    from rank_bm25 import BM25Okapi

    stopwords = set(["le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "à"])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]

    articles = df["article_text"].fillna("").tolist()
    texts = df["text"].fillna("").tolist()

    tokenized_articles = [tokenize(a) for a in articles]
    bm25 = BM25Okapi(tokenized_articles)

    scores = []
    for i, text in enumerate(texts):
        query = tokenize(text)
        if query:
            all_scores = bm25.get_scores(query)
            scores.append(all_scores[i])
        else:
            scores.append(0.0)

    scores = np.array(scores)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder 0-shot (no fine-tuning, therefore unsupervised)."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all UNSUPERVISED features (no use of the labels)."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION (UNSUPERVISED)")
    print("="*70)

    df = df.copy()

    # LLM votes (0-shot, therefore unsupervised)
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)

    # Aggregations
    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"]
    df["vote_union"] = ((df["vote_llama"] == 1) | (df["vote_qwen"] == 1) | (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)
    df["vote_inter3"] = (df["vote_sum"] == 3).astype(int)

    # Similarities
    print("  [1/3] TF-IDF...")
    df["sim_tfidf"] = compute_tfidf_similarity(df)

    print("  [2/3] BM25...")
    df["sim_bm25"] = compute_bm25_scores(df)

    if compute_crossencoder:
        print("  [3/3] Cross-Encoder...")
        df["sim_crossencoder"] = compute_cross_encoder_scores(df, config)
    else:
        df["sim_crossencoder"] = 0.0

    return df


# ============================================================
# UNSUPERVISED RANKING METHODS
# ============================================================
def compute_random_scores(df: pd.DataFrame, seed: int = 42) -> np.ndarray:
    """Baseline: random scores."""
    np.random.seed(seed)
    return np.random.rand(len(df))


def compute_tfidf_only(df: pd.DataFrame) -> np.ndarray:
    """Ranking based only on TF-IDF."""
    return df["sim_tfidf"].values


def compute_bm25_only(df: pd.DataFrame) -> np.ndarray:
    """Ranking based only on BM25."""
    return df["sim_bm25"].values


def compute_crossencoder_only(df: pd.DataFrame) -> np.ndarray:
    """Ranking based only on Cross-Encoder."""
    return df["sim_crossencoder"].values


def compute_llm_union(df: pd.DataFrame) -> np.ndarray:
    """Ranking: 1 if at least 1 LLM votes yes, 0 otherwise."""
    return df["vote_union"].values.astype(float)


def compute_llm_inter2(df: pd.DataFrame) -> np.ndarray:
    """Ranking: 1 if 2+ LLMs vote yes, 0 otherwise."""
    return df["vote_inter2"].values.astype(float)


def compute_llm_inter3(df: pd.DataFrame) -> np.ndarray:
    """Ranking: 1 if 3 LLMs vote yes, 0 otherwise."""
    return df["vote_inter3"].values.astype(float)


def compute_heuristic_fixed(df: pd.DataFrame) -> np.ndarray:
    """
    Heuristic with FIXED weights (no optimisation on labels).
    Weights chosen a priori based on intuition:
    - LLMs: medium importance (0.3 each)
    - Intersections: bonus if agreement
    - Similarities: secondary importance
    """
    weights = {
        "vote_llama": 0.3,
        "vote_qwen": 0.3,
        "vote_mistral": 0.3,
        "vote_inter2": 0.5,
        "vote_inter3": 1.0,
        "sim_tfidf": 0.2,
        "sim_bm25": 0.2,
        "sim_crossencoder": 0.4,
    }

    score = np.zeros(len(df))
    for feature, weight in weights.items():
        if feature in df.columns:
            score += weight * df[feature].values

    return score


# ============================================================
# EVALUATION (labels used ONLY here, not for training)
# ============================================================
def evaluate_method(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate a ranking with all metrics."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        # Global metrics (computed once)
        ap = Metrics.average_precision(y, s)
        rr = Metrics.reciprocal_rank(y, s)

        for k in config.k_values:
            count_k = Metrics.count_at_k(y, s, k)
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "F1@k": Metrics.f1_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": count_k,
                "total_positives": n_pos,
                "count_ratio": f"{count_k}/{n_pos}",
                # Global metrics (repeated for each k)
                "AP": ap,
                "RR": rr,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """
    Compute the random baseline = positive rate (prevalence).
    """
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count": int(expected_pos_at_k),
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# REPORTING
# ============================================================
def print_comparison_table(
    results_dict: Dict[str, pd.DataFrame],
    config: Config
):
    """Formatted comparison table."""
    print("\n" + "="*120)
    print("COMPARISON OF UNSUPERVISED METHODS")
    print("="*120)

    methods = list(results_dict.keys())

    for target in config.targets:
        print(f"\n{'='*120}")
        print(f"TARGET: {target}")
        print(f"{'='*120}")

        # Get prevalence
        df_random = results_dict.get("Random")
        prevalence = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")

        # AP Summary - ALL methods
        print("\n  METHOD               | AP     | Δ vs Random")
        print("  " + "-"*45)
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = (ap - prevalence) / prevalence * 100
                    delta_str = f"+{delta:.0f}%" if delta >= 0 else f"{delta:.0f}%"
                else:
                    delta_str = "-"
                print(f"  {method:<20} | {ap:.4f} | {delta_str}")

        # ========================================
        # Show ALL methods
        # ========================================
        print(f"\n  {'k':<6}", end="")
        for method in methods:
            print(f" | {method[:12]:<12}", end="")
        print()

        print("  " + "-"*(7 + 15 * len(methods)))

        for k in [10, 50, 100, 200, 300, 500]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in methods:
                df_res = results_dict[method]
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    row += f" | {p:.3f}       "
                else:
                    row += " | -           "
            print(row)


def plot_comparison(
    results_dict: Dict[str, pd.DataFrame],
    config: Config,
    output_path: Optional[str] = None
):
    """Multi-method visualisation."""
    n_targets = len(config.targets)

    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))

    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'TF-IDF': 'blue',
        'BM25': 'cyan',
        'CrossEncoder': 'green',
        'LLM_Union': 'orange',
        'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple',
        'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        # P@k
        ax = axes[0, i]
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["P@k"],
                       color=colors.get(method, 'gray'),
                       marker='o' if method != 'Random' else '.',
                       linewidth=2 if method != 'Random' else 1,
                       label=method,
                       alpha=0.7 if method == 'Random' else 1.0)

        ax.set_xlabel("k")
        ax.set_ylabel("P@k")
        ax.set_title(f"{target} - Precision@k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)

        # NDCG@k
        ax = axes[1, i]
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["NDCG@k"],
                       color=colors.get(method, 'gray'),
                       marker='o' if method != 'Random' else '.',
                       linewidth=2 if method != 'Random' else 1,
                       label=method,
                       alpha=0.7 if method == 'Random' else 1.0)

        ax.set_xlabel("k")
        ax.set_ylabel("NDCG@k")
        ax.set_title(f"{target} - NDCG@k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")

    plt.show()


# ============================================================
# DETAILED ANALYSIS OF RETRIEVED POSITIVES
# ============================================================
def analyze_positive_coverage(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """
    Detailed analysis: how many positives retrieved at each k vs chance.
    """
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))

        if n_pos == 0:
            continue

        prevalence = n_pos / n_samples

        # Ranking order
        order = np.argsort(s)[::-1]
        y_sorted = y[order]

        for k in config.k_values:
            k_actual = min(k, n_samples)
            positives_at_k = int(np.sum(y_sorted[:k_actual]))

            # Chance = k * prevalence
            random_expected = k_actual * prevalence
            delta_vs_random = positives_at_k - random_expected

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "positives_retrieved": positives_at_k,
                "total_positives": n_pos,
                "coverage": positives_at_k / n_pos,
                "remaining": n_pos - positives_at_k,
                "random_expected": random_expected,
                "delta_vs_random": delta_vs_random,
                "n_samples": n_samples,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)

def analyze_llm_intersection_coverage(df: pd.DataFrame) -> Dict:
    """
    Coverage analysis by LLM intersections.
    """
    stats = {
        "inter3": int((df["vote_inter3"] == 1).sum()),
        "inter2": int((df["vote_inter2"] == 1).sum()),
        "union": int((df["vote_union"] == 1).sum()),
        "no_vote": int((df["vote_union"] == 0).sum()),
        "total": int(len(df))
    }

    # Per annotator
    for target in ["A1", "A2", "Gold"]:
        label_col = f"lbl_{target}"
        if label_col not in df.columns:
            continue

        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        df_target = df[mask].copy()
        y = df_target[label_col].astype(int)

        stats[f"{target}_total"] = int(len(y))
        stats[f"{target}_positives"] = int(y.sum())

        # How many positives in each category
        stats[f"{target}_pos_inter3"] = int(y[df_target["vote_inter3"] == 1].sum())
        stats[f"{target}_pos_inter2"] = int(y[df_target["vote_inter2"] == 1].sum())
        stats[f"{target}_pos_union"] = int(y[df_target["vote_union"] == 1].sum())
        stats[f"{target}_pos_no_vote"] = int(y[df_target["vote_union"] == 0].sum())

    return stats

def print_llm_intersection_analysis(stats: Dict):
    """
    Print the LLM intersection analysis.
    """
    print("\n" + "="*70)
    print("LLM INTERSECTION ANALYSIS")
    print("="*70)

    print(f"\n  Full dataset: {stats['total']} entries")
    print(f"    - 3 LLMs agree (inter3):  {stats['inter3']:>6} ({stats['inter3']/stats['total']*100:>5.1f}%)")
    print(f"    - 2+ LLMs agree (inter2):  {stats['inter2']:>6} ({stats['inter2']/stats['total']*100:>5.1f}%)")
    print(f"    - 1+ LLM agrees (union):    {stats['union']:>6} ({stats['union']/stats['total']*100:>5.1f}%)")
    print(f"    - No LLM agrees:         {stats['no_vote']:>6} ({stats['no_vote']/stats['total']*100:>5.1f}%)")

    for target in ["A1", "A2", "Gold"]:
        if f"{target}_total" not in stats:
            continue

        n_total = stats[f"{target}_total"]
        n_pos = stats[f"{target}_positives"]

        if n_pos == 0:
            continue

        print(f"\n  {target}: {n_pos} positives out of {n_total} ({n_pos/n_total*100:.1f}%)")

        pos_inter3 = stats[f"{target}_pos_inter3"]
        pos_inter2 = stats[f"{target}_pos_inter2"]
        pos_union = stats[f"{target}_pos_union"]
        pos_no_vote = stats[f"{target}_pos_no_vote"]

        print(f"    - In inter3:    {pos_inter3:>4} ({pos_inter3/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter2:    {pos_inter2:>4} ({pos_inter2/n_pos*100:>5.1f}% of positives)")
        print(f"    - In union:     {pos_union:>4} ({pos_union/n_pos*100:>5.1f}% of positives)")
        print(f"    - Outside union:     {pos_no_vote:>4} ({pos_no_vote/n_pos*100:>5.1f}% of positives)")


def print_coverage_table(
    coverage_dict: Dict[str, pd.DataFrame],
    config: Config
):
    """
    Coverage table: how many positives retrieved vs chance.
    """
    print("\n" + "="*140)
    print("POSITIVE COVERAGE PER METHOD (vs CHANCE)")
    print("="*140)

    methods = list(coverage_dict.keys())

    for target in config.targets:
        print(f"\n{'='*140}")
        print(f"TARGET: {target}")

        # Total positives
        df_first = next(iter(coverage_dict.values()))
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue

        total_pos = df_t["total_positives"].iloc[0]
        n_samples = df_t["n_samples"].iloc[0]
        prevalence = df_t["prevalence"].iloc[0]

        print(f"  Total positives: {total_pos} / {n_samples} (prevalence: {prevalence:.3f})")
        print(f"{'='*140}")

        # Header
        print(f"\n  {'k':<6}", end="")
        for method in methods:
            print(f" | {method[:10]:<16}", end="")
        print()

        print(f"  {'':6}", end="")
        for _ in methods:
            print(f" | Retr (Δ chance)", end="")
        print()

        print("  " + "-"*(7 + 19 * len(methods)))

        # Rows
        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue

            row = f"  {k:<6}"
            for method in methods:
                df_cov = coverage_dict[method]
                df_tk = df_cov[(df_cov["target"] == target) & (df_cov["k"] == k)]
                if len(df_tk) > 0:
                    retrieved = int(df_tk["positives_retrieved"].values[0])
                    random_exp = df_tk["random_expected"].values[0]
                    delta = retrieved - random_exp

                    row += f" | {retrieved:>3} ({delta:+5.1f}) "
                else:
                    row += " | -               "
            print(row)


def plot_coverage_curves(
    coverage_dict: Dict[str, pd.DataFrame],
    config: Config,
    output_path: Optional[str] = None
):
    """
    Coverage curves: positives retrieved vs k (with random baseline).
    """
    n_targets = len(config.targets)

    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))

    if n_targets == 1:
        axes = [axes]

    colors = {
        'Random': 'black',
        'TF-IDF': 'blue',
        'BM25': 'cyan',
        'CrossEncoder': 'green',
        'LLM_Union': 'orange',
        'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple',
        'Heuristic': 'magenta',
    }

    for i, target in enumerate(config.targets):
        ax = axes[i]

        # First draw the Random line (baseline)
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            # Draw the chance-expected curve
            ax.plot(df_t_first["k"], df_t_first["random_expected"],
                   color='black',
                   linestyle='--',
                   linewidth=1.5,
                   label='Random (expected)',
                   alpha=0.7,
                   zorder=1)  # In the background

        # Then the other methods
        for method, df_cov in coverage_dict.items():
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["positives_retrieved"],
                       color=colors.get(method, 'gray'),
                       marker='o',
                       linewidth=2,
                       label=method,
                       zorder=2)  # In the foreground

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Number of positives retrieved")
        ax.set_title(f"{target} - Positives retrieved as a function of k")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")

    plt.show()

def plot_gain_vs_random(
    coverage_dict: Dict[str, pd.DataFrame],
    config: Config,
    output_path: Optional[str] = None
):
    """
    Plot of relative gain over chance.
    """
    n_targets = len(config.targets)

    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))

    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue',
        'BM25': 'cyan',
        'CrossEncoder': 'green',
        'LLM_Union': 'orange',
        'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple',
        'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        ax = axes[i]

        # Reference line at 0
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

        for method, df_cov in coverage_dict.items():
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["delta_vs_random"],
                       color=colors.get(method, 'gray'),
                       marker='o',
                       linewidth=2,
                       label=method)

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Gain vs Random (number of positives)")
        ax.set_title(f"{target} - Gain over chance")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")

    plt.show()

# ============================================================
# MAIN
# ============================================================
def main(
    filter_union: bool = False,
    compute_crossencoder: bool = True
):
    """
    Fully unsupervised main pipeline with coverage analysis.
    """
    print("="*70)
    print("FULLY UNSUPERVISED RANKING PIPELINE")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print("  ⚠️  NO LABELS USED FOR RANKING (evaluation only)")
    print("="*70)

    # Install dependencies
    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()

    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_unsupervised{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # 1. Load & prepare data
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    # 2. Filter if needed
    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # LLM intersection analysis
    llm_stats = analyze_llm_intersection_coverage(df)
    print_llm_intersection_analysis(llm_stats)

    # 3. Run all unsupervised methods
    print("\n" + "="*70)
    print("COMPUTING UNSUPERVISED RANKINGS")
    print("="*70)

    results_dict = {}
    coverage_dict = {}

    # Random baseline
    print("\n  [1/8] Random baseline...")
    results_dict["Random"] = compute_random_baseline(df, config)

    # TF-IDF only
    print("  [2/8] TF-IDF only...")
    scores_tfidf = compute_tfidf_only(df)
    results_dict["TF-IDF"] = evaluate_method(df, scores_tfidf, "TF-IDF", config)
    coverage_dict["TF-IDF"] = analyze_positive_coverage(df, scores_tfidf, "TF-IDF", config)

    # BM25 only
    print("  [3/8] BM25 only...")
    scores_bm25 = compute_bm25_only(df)
    results_dict["BM25"] = evaluate_method(df, scores_bm25, "BM25", config)
    coverage_dict["BM25"] = analyze_positive_coverage(df, scores_bm25, "BM25", config)

    # Cross-Encoder only
    if compute_crossencoder:
        print("  [4/8] Cross-Encoder only...")
        scores_ce = compute_crossencoder_only(df)
        results_dict["CrossEncoder"] = evaluate_method(df, scores_ce, "CrossEncoder", config)
        coverage_dict["CrossEncoder"] = analyze_positive_coverage(df, scores_ce, "CrossEncoder", config)

    # LLM Union
    print("  [5/8] LLM Union...")
    scores_union = compute_llm_union(df)
    results_dict["LLM_Union"] = evaluate_method(df, scores_union, "LLM_Union", config)
    coverage_dict["LLM_Union"] = analyze_positive_coverage(df, scores_union, "LLM_Union", config)

    # LLM Intersection 2+
    print("  [6/8] LLM Intersection 2+...")
    scores_inter2 = compute_llm_inter2(df)
    results_dict["LLM_Inter2"] = evaluate_method(df, scores_inter2, "LLM_Inter2", config)
    coverage_dict["LLM_Inter2"] = analyze_positive_coverage(df, scores_inter2, "LLM_Inter2", config)

    # LLM Intersection 3
    print("  [7/8] LLM Intersection 3...")
    scores_inter3 = compute_llm_inter3(df)
    results_dict["LLM_Inter3"] = evaluate_method(df, scores_inter3, "LLM_Inter3", config)
    coverage_dict["LLM_Inter3"] = analyze_positive_coverage(df, scores_inter3, "LLM_Inter3", config)

    # Heuristic fixe
    print("  [8/8] Heuristic (fixed weights)...")
    scores_heur = compute_heuristic_fixed(df)
    results_dict["Heuristic"] = evaluate_method(df, scores_heur, "Heuristic", config)
    coverage_dict["Heuristic"] = analyze_positive_coverage(df, scores_heur, "Heuristic", config)

    # 4. Reporting
    print_comparison_table(results_dict, config)

    # Coverage tables
    print_coverage_table(coverage_dict, config)

    # 5. Visualization
    plot_comparison(results_dict, config,
                   output_path=f"{config.output_dir}/comparison_unsupervised.png")

    # Coverage curves (with Random)
    plot_coverage_curves(coverage_dict, config,
                        output_path=f"{config.output_dir}/coverage_curves.png")

    # Gain vs Random plot
    plot_gain_vs_random(coverage_dict, config,
                       output_path=f"{config.output_dir}/gain_vs_random.png")

    # 6. Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    df["score_tfidf"] = scores_tfidf
    df["score_bm25"] = scores_bm25
    df["score_union"] = scores_union
    df["score_inter2"] = scores_inter2
    df["score_inter3"] = scores_inter3
    df["score_heuristic"] = scores_heur
    if compute_crossencoder:
        df["score_ce"] = scores_ce

    df.to_csv(f"{config.output_dir}/df_with_scores_{timestamp}.csv", index=False)

    df_results_all = pd.concat(list(results_dict.values()), ignore_index=True)
    df_results_all.to_csv(f"{config.output_dir}/results_unsupervised_{timestamp}.csv", index=False)

    # Save coverage stats
    df_coverage_all = pd.concat(list(coverage_dict.values()), ignore_index=True)
    df_coverage_all.to_csv(f"{config.output_dir}/coverage_analysis_{timestamp}.csv", index=False)

    with open(f"{config.output_dir}/llm_intersection_stats_{timestamp}.json", "w") as f:
        json.dump(llm_stats, f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict, coverage_dict, llm_stats


if __name__ == "__main__":
    # Fully unsupervised pipeline
    df, results, coverage, llm_stats = main(
        filter_union=False,
        compute_crossencoder=True
    )


# FINAL UNSUPERVISED

In [ ]:
# ============================================================
# FULLY UNSUPERVISED PIPELINE - NO LABELS USED
# ============================================================
"""
Fully unsupervised pipeline for legal ranking.

TIE HANDLING (TIE-BREAK):
====================================
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3) produce
binary scores (0 or 1), creating many ties. To break them:

  Score final = Score_LLM + ε × Score_CrossEncoder

where ε = 1e-6, small enough for the LLM score to dominate.

Concretely:
- LLM_Union: the 820 docs with score=1 are ordered among themselves by CrossEncoder
- LLM_Inter2: the 523 docs with score=1 are ordered among themselves by CrossEncoder
- LLM_Inter3: the 195 docs with score=1 are ordered among themselves by CrossEncoder

The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) have no
significant ties and use a direct sort.

Reference: Voorhees (2000), TREC evaluation methodology
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import json

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess
    packages = [
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]
    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    random_seed: int = 42

    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.output_dir = f"artifacts/outputs_unsupervised"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        dcg = float(np.sum(y_sorted / np.log2(np.arange(2, k + 2))))

        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:k]
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k + 2))))

        return dcg / idcg if idcg > 0 else 0.0

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 3 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]
    for df in [df_main, df_llama, df_qwen, df_mistral]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)
    df = df.drop(columns=["_key"])

    return df


# ============================================================
# FEATURES
# ============================================================
def compute_tfidf_similarity(df: pd.DataFrame) -> np.ndarray:
    """TF-IDF similarity."""
    stopwords_fr = [
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ]

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2,
        max_df=0.95, stop_words=stopwords_fr, lowercase=True
    )
    vectorizer.fit(texts + articles)

    tfidf_texts = vectorizer.transform(texts)
    tfidf_articles = vectorizer.transform(articles)

    return np.array([
        cosine_similarity(tfidf_texts[i], tfidf_articles[i])[0, 0]
        for i in range(len(texts))
    ])


def compute_bm25_scores(df: pd.DataFrame) -> np.ndarray:
    """BM25 scores."""
    from rank_bm25 import BM25Okapi

    stopwords = set(["le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "à"])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]

    articles = df["article_text"].fillna("").tolist()
    texts = df["text"].fillna("").tolist()

    tokenized_articles = [tokenize(a) for a in articles]
    bm25 = BM25Okapi(tokenized_articles)

    scores = []
    for i, text in enumerate(texts):
        query = tokenize(text)
        if query:
            all_scores = bm25.get_scores(query)
            scores.append(all_scores[i])
        else:
            scores.append(0.0)

    scores = np.array(scores)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder 0-shot."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all UNSUPERVISED features."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION (UNSUPERVISED)")
    print("="*70)

    df = df.copy()

    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)

    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"]
    df["vote_union"] = ((df["vote_llama"] == 1) | (df["vote_qwen"] == 1) | (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)
    df["vote_inter3"] = (df["vote_sum"] == 3).astype(int)

    print("  [1/3] TF-IDF...")
    df["sim_tfidf"] = compute_tfidf_similarity(df)

    print("  [2/3] BM25...")
    df["sim_bm25"] = compute_bm25_scores(df)

    if compute_crossencoder:
        print("  [3/3] Cross-Encoder...")
        df["sim_crossencoder"] = compute_cross_encoder_scores(df, config)
    else:
        df["sim_crossencoder"] = 0.0

    return df


# ============================================================
# RANKING SCORES (with integrated tie-break)
# ============================================================
def compute_all_ranking_scores(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    """
    Compute the ranking scores for all methods.

    TIE-BREAK HANDLING:
    - Continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic): direct scores
    - Binary methods (LLM_*): score = vote + ε × CrossEncoder to break ties
    """
    scores = {}

    # Continuous methods - no tie-break needed
    scores["TF-IDF"] = df["sim_tfidf"].values
    scores["BM25"] = df["sim_bm25"].values
    scores["CrossEncoder"] = df["sim_crossencoder"].values

    # Heuristic - weighted combination (near-unique scores)
    weights = {
        "vote_llama": 0.3, "vote_qwen": 0.3, "vote_mistral": 0.3,
        "vote_inter2": 0.5, "vote_inter3": 1.0,
        "sim_tfidf": 0.2, "sim_bm25": 0.2, "sim_crossencoder": 0.4,
    }
    heuristic = np.zeros(len(df))
    for feature, weight in weights.items():
        if feature in df.columns:
            heuristic += weight * df[feature].values
    scores["Heuristic"] = heuristic

    # Binary methods - tie-break by CrossEncoder
    # Score = vote + ε × CrossEncoder_normalized
    eps = 1e-6
    ce_norm = df["sim_crossencoder"].values
    if ce_norm.max() > ce_norm.min():
        ce_norm = (ce_norm - ce_norm.min()) / (ce_norm.max() - ce_norm.min())

    scores["LLM_Union"] = df["vote_union"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter2"] = df["vote_inter2"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter3"] = df["vote_inter3"].values.astype(float) + eps * ce_norm

    return scores


# ============================================================
# EVALUATION
# ============================================================
def evaluate_method(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate a ranking."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": Metrics.count_at_k(y, s, k),
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """Random baseline = prevalence."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count": int(expected_pos_at_k),
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# COVERAGE ANALYSIS
# ============================================================
def analyze_positive_coverage(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Analysis: how many positives retrieved at each k."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))

        if n_pos == 0:
            continue

        prevalence = n_pos / n_samples
        order = np.argsort(s)[::-1]
        y_sorted = y[order]

        for k in config.k_values:
            k_actual = min(k, n_samples)
            positives_at_k = int(np.sum(y_sorted[:k_actual]))
            random_expected = k_actual * prevalence

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "positives_retrieved": positives_at_k,
                "total_positives": n_pos,
                "coverage": positives_at_k / n_pos,
                "random_expected": random_expected,
                "delta_vs_random": positives_at_k - random_expected,
                "n_samples": n_samples,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def analyze_llm_intersection_coverage(df: pd.DataFrame) -> Dict:
    """Coverage analysis by LLM intersections."""
    stats = {
        "inter3": int((df["vote_inter3"] == 1).sum()),
        "inter2": int((df["vote_inter2"] == 1).sum()),
        "union": int((df["vote_union"] == 1).sum()),
        "no_vote": int((df["vote_union"] == 0).sum()),
        "total": int(len(df))
    }

    for target in ["A1", "A2", "Gold"]:
        label_col = f"lbl_{target}"
        if label_col not in df.columns:
            continue

        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        df_target = df[mask].copy()
        y = df_target[label_col].astype(int)

        stats[f"{target}_total"] = int(len(y))
        stats[f"{target}_positives"] = int(y.sum())
        stats[f"{target}_pos_inter3"] = int(y[df_target["vote_inter3"] == 1].sum())
        stats[f"{target}_pos_inter2"] = int(y[df_target["vote_inter2"] == 1].sum())
        stats[f"{target}_pos_union"] = int(y[df_target["vote_union"] == 1].sum())
        stats[f"{target}_pos_no_vote"] = int(y[df_target["vote_union"] == 0].sum())

    return stats


# ============================================================
# REPORTING
# ============================================================
def print_llm_intersection_analysis(stats: Dict):
    """Print the LLM intersection analysis."""
    print("\n" + "="*70)
    print("LLM INTERSECTION ANALYSIS")
    print("="*70)

    print(f"\n  Full dataset: {stats['total']} entries")
    print(f"    - 3 LLMs agree (inter3):  {stats['inter3']:>6} ({stats['inter3']/stats['total']*100:>5.1f}%)")
    print(f"    - 2+ LLMs agree (inter2): {stats['inter2']:>6} ({stats['inter2']/stats['total']*100:>5.1f}%)")
    print(f"    - 1+ LLM agrees (union):   {stats['union']:>6} ({stats['union']/stats['total']*100:>5.1f}%)")
    print(f"    - No LLM agrees:        {stats['no_vote']:>6} ({stats['no_vote']/stats['total']*100:>5.1f}%)")

    for target in ["A1", "A2", "Gold"]:
        if f"{target}_total" not in stats:
            continue

        n_total = stats[f"{target}_total"]
        n_pos = stats[f"{target}_positives"]

        if n_pos == 0:
            continue

        print(f"\n  {target}: {n_pos} positives out of {n_total} ({n_pos/n_total*100:.1f}%)")
        print(f"    - In inter3:  {stats[f'{target}_pos_inter3']:>4} ({stats[f'{target}_pos_inter3']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter2:  {stats[f'{target}_pos_inter2']:>4} ({stats[f'{target}_pos_inter2']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In union:   {stats[f'{target}_pos_union']:>4} ({stats[f'{target}_pos_union']/n_pos*100:>5.1f}% of positives)")
        print(f"    - Outside union:   {stats[f'{target}_pos_no_vote']:>4} ({stats[f'{target}_pos_no_vote']/n_pos*100:>5.1f}% of positives)")


def print_comparison_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Comparison table."""
    print("\n" + "="*120)
    print("COMPARISON OF UNSUPERVISED METHODS")
    print("="*120)

    methods = list(results_dict.keys())

    for target in config.targets:
        print(f"\n{'='*120}")
        print(f"TARGET: {target}")
        print(f"{'='*120}")

        df_random = results_dict.get("Random")
        prevalence = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")

        print("\n  METHOD               | AP     | Δ vs Random")
        print("  " + "-"*45)
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = ap - prevalence
                    delta_str = f"+{delta:.3f}" if delta >= 0 else f"{delta:.3f}"
                else:
                    delta_str = "-"
                print(f"  {method:<20} | {ap:.4f} | {delta_str}")

        print(f"\n  {'k':<6}", end="")
        for method in methods:
            print(f" | {method[:12]:<12}", end="")
        print()
        print("  " + "-"*(7 + 15 * len(methods)))

        for k in [10, 50, 100, 200, 300, 500]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in methods:
                df_res = results_dict[method]
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    row += f" | {p:.3f}       "
                else:
                    row += " | -           "
            print(row)


def print_coverage_table(coverage_dict: Dict[str, pd.DataFrame], config: Config):
    """Coverage table."""
    print("\n" + "="*140)
    print("POSITIVE COVERAGE PER METHOD")
    print("="*140)

    methods = list(coverage_dict.keys())

    for target in config.targets:
        print(f"\n{'='*140}")
        print(f"TARGET: {target}")

        df_first = next(iter(coverage_dict.values()))
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue

        total_pos = df_t["total_positives"].iloc[0]
        n_samples = df_t["n_samples"].iloc[0]
        prevalence = df_t["prevalence"].iloc[0]

        print(f"  Total positives: {total_pos} / {n_samples} (prevalence: {prevalence:.3f})")
        print(f"{'='*140}")

        print(f"\n  {'k':<6}", end="")
        for method in methods:
            print(f" | {method[:12]:<14}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in methods:
            print(f" | Retr / Total ", end="")
        print()
        print("  " + "-"*(7 + 17 * len(methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in methods:
                df_cov = coverage_dict[method]
                df_tk = df_cov[(df_cov["target"] == target) & (df_cov["k"] == k)]
                if len(df_tk) > 0:
                    retrieved = int(df_tk["positives_retrieved"].values[0])
                    total = int(df_tk["total_positives"].values[0])
                    row += f" | {retrieved:>4} / {total:<4}  "
                else:
                    row += " | -              "
            print(row)


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualisation."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple', 'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method, df_res in results_dict.items():
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric}")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_coverage_curves(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Coverage curves."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        ax = axes[i]
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            ax.plot(df_t_first["k"], df_t_first["random_expected"],
                   color='black', linestyle='--', linewidth=1.5,
                   label='Random (expected)', alpha=0.7)

        for method, df_cov in coverage_dict.items():
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["positives_retrieved"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k")
        ax.set_ylabel("Positives retrieved")
        ax.set_title(f"{target}")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_gain_vs_random(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of relative gain over chance."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        ax = axes[i]
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

        for method, df_cov in coverage_dict.items():
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["delta_vs_random"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Gain vs Random (number of positives)")
        ax.set_title(f"{target} - Gain over chance")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(filter_union: bool = False, compute_crossencoder: bool = True):
    """Main pipeline."""
    print("="*70)
    print("FULLY UNSUPERVISED RANKING PIPELINE")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print("="*70)
    print("""
TIE HANDLING (TIE-BREAK):
------------------------------------
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3) produce
0/1 scores creating ties. To break them deterministically:

  Score_final = Score_LLM + ε × Score_CrossEncoder  (ε = 1e-6)

Documents with the same LLM vote are ordered by their CrossEncoder score.
The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) have no
significant ties and use a direct sort.
""")

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_unsupervised{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # Load & prepare
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # LLM intersection analysis
    llm_stats = analyze_llm_intersection_coverage(df)
    print_llm_intersection_analysis(llm_stats)

    # Compute all scores
    print("\n" + "="*70)
    print("COMPUTING RANKINGS")
    print("="*70)

    all_scores = compute_all_ranking_scores(df)

    results_dict = {}
    coverage_dict = {}

    print("\n  [1/8] Random baseline...")
    results_dict["Random"] = compute_random_baseline(df, config)

    for i, (method_name, scores) in enumerate(all_scores.items(), 2):
        print(f"  [{i}/8] {method_name}...")
        results_dict[method_name] = evaluate_method(df, scores, method_name, config)
        coverage_dict[method_name] = analyze_positive_coverage(df, scores, method_name, config)

    # Reporting
    print_comparison_table(results_dict, config)
    print_coverage_table(coverage_dict, config)

    # Plots
    plot_comparison(results_dict, config, f"{config.output_dir}/comparison.png")
    plot_coverage_curves(coverage_dict, config, f"{config.output_dir}/coverage.png")
    plot_gain_vs_random(coverage_dict, config, f"{config.output_dir}/gain_vs_random.png")

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    for name, scores in all_scores.items():
        df[f"score_{name.lower().replace('-', '_')}"] = scores

    df.to_csv(f"{config.output_dir}/df_with_scores_{timestamp}.csv", index=False)

    df_results = pd.concat(list(results_dict.values()), ignore_index=True)
    df_results.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    df_coverage = pd.concat(list(coverage_dict.values()), ignore_index=True)
    df_coverage.to_csv(f"{config.output_dir}/coverage_{timestamp}.csv", index=False)

    with open(f"{config.output_dir}/llm_stats_{timestamp}.json", "w") as f:
        json.dump(llm_stats, f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict, coverage_dict, llm_stats


if __name__ == "__main__":
    df, results, coverage, llm_stats = main(filter_union=False, compute_crossencoder=True)

# BIS

In [ ]:
# ============================================================
# FULLY UNSUPERVISED PIPELINE - NO LABELS USED
# ============================================================
"""
Fully unsupervised pipeline for legal ranking.

TIE HANDLING (TIE-BREAK):
====================================
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3) produce
binary scores (0 or 1), creating many ties. To break them:

  Score final = Score_LLM + ε × Score_CrossEncoder

where ε = 1e-6, small enough for the LLM score to dominate.

Concretely:
- LLM_Union: the 820 docs with score=1 are ordered among themselves by CrossEncoder
- LLM_Inter2: the 523 docs with score=1 are ordered among themselves by CrossEncoder
- LLM_Inter3: the 195 docs with score=1 are ordered among themselves by CrossEncoder

The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) have no
significant ties and use a direct sort.

Reference: Voorhees (2000), TREC evaluation methodology
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import json

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess
    packages = [
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]
    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    random_seed: int = 42

    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.output_dir = f"artifacts/outputs_unsupervised"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        dcg = float(np.sum(y_sorted / np.log2(np.arange(2, k + 2))))

        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:k]
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k + 2))))

        return dcg / idcg if idcg > 0 else 0.0

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 3 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]
    for df in [df_main, df_llama, df_qwen, df_mistral]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)
    df = df.drop(columns=["_key"])

    return df


# ============================================================
# FEATURES
# ============================================================
def compute_tfidf_similarity(df: pd.DataFrame) -> np.ndarray:
    """TF-IDF similarity."""
    stopwords_fr = [
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ]

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2,
        max_df=0.95, stop_words=stopwords_fr, lowercase=True
    )
    vectorizer.fit(texts + articles)

    tfidf_texts = vectorizer.transform(texts)
    tfidf_articles = vectorizer.transform(articles)

    return np.array([
        cosine_similarity(tfidf_texts[i], tfidf_articles[i])[0, 0]
        for i in range(len(texts))
    ])


def compute_bm25_scores(df: pd.DataFrame) -> np.ndarray:
    """BM25 scores."""
    from rank_bm25 import BM25Okapi

    stopwords = set(["le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "à"])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]

    articles = df["article_text"].fillna("").tolist()
    texts = df["text"].fillna("").tolist()

    tokenized_articles = [tokenize(a) for a in articles]
    bm25 = BM25Okapi(tokenized_articles)

    scores = []
    for i, text in enumerate(texts):
        query = tokenize(text)
        if query:
            all_scores = bm25.get_scores(query)
            scores.append(all_scores[i])
        else:
            scores.append(0.0)

    scores = np.array(scores)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder 0-shot."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all UNSUPERVISED features."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION (UNSUPERVISED)")
    print("="*70)

    df = df.copy()

    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)

    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"]
    df["vote_union"] = ((df["vote_llama"] == 1) | (df["vote_qwen"] == 1) | (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)
    df["vote_inter3"] = (df["vote_sum"] == 3).astype(int)

    print("  [1/3] TF-IDF...")
    df["sim_tfidf"] = compute_tfidf_similarity(df)

    print("  [2/3] BM25...")
    df["sim_bm25"] = compute_bm25_scores(df)

    if compute_crossencoder:
        print("  [3/3] Cross-Encoder...")
        df["sim_crossencoder"] = compute_cross_encoder_scores(df, config)
    else:
        df["sim_crossencoder"] = 0.0

    return df


# ============================================================
# RANKING SCORES (with integrated tie-break)
# ============================================================
def compute_all_ranking_scores(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    """
    Compute the ranking scores for all methods.

    TIE-BREAK HANDLING:
    - Continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic): direct scores
    - Binary methods (LLM_*): score = vote + ε × CrossEncoder to break ties
    """
    scores = {}

    # Continuous methods - no tie-break needed
    scores["TF-IDF"] = df["sim_tfidf"].values
    scores["BM25"] = df["sim_bm25"].values
    scores["CrossEncoder"] = df["sim_crossencoder"].values

    # Heuristic - weighted combination (near-unique scores)
    weights = {
        "vote_llama": 0.3, "vote_qwen": 0.3, "vote_mistral": 0.3,
        "vote_inter2": 0.5, "vote_inter3": 1.0,
        "sim_tfidf": 0.2, "sim_bm25": 0.2, "sim_crossencoder": 0.4,
    }
    heuristic = np.zeros(len(df))
    for feature, weight in weights.items():
        if feature in df.columns:
            heuristic += weight * df[feature].values
    scores["Heuristic"] = heuristic

    # Binary methods - tie-break by CrossEncoder
    # Score = vote + ε × CrossEncoder_normalized
    eps = 1e-6
    ce_norm = df["sim_crossencoder"].values
    if ce_norm.max() > ce_norm.min():
        ce_norm = (ce_norm - ce_norm.min()) / (ce_norm.max() - ce_norm.min())

    scores["LLM_Union"] = df["vote_union"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter2"] = df["vote_inter2"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter3"] = df["vote_inter3"].values.astype(float) + eps * ce_norm

    return scores


# ============================================================
# EVALUATION (MODIFIED - with count and ratio)
# ============================================================
def evaluate_method(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate a ranking with the number of retrieved yes."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            count_k = Metrics.count_at_k(y, s, k)
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": count_k,
                "total_positives": n_pos,
                "count_ratio": f"{count_k}/{n_pos}",
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """Random baseline = prevalence."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count": int(expected_pos_at_k),
                "total_positives": n_pos,
                "count_ratio": f"{int(expected_pos_at_k)}/{n_pos}",
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# COVERAGE ANALYSIS
# ============================================================
def analyze_positive_coverage(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Analysis: how many positives retrieved at each k."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))

        if n_pos == 0:
            continue

        prevalence = n_pos / n_samples
        order = np.argsort(s)[::-1]
        y_sorted = y[order]

        for k in config.k_values:
            k_actual = min(k, n_samples)
            positives_at_k = int(np.sum(y_sorted[:k_actual]))
            random_expected = k_actual * prevalence

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "positives_retrieved": positives_at_k,
                "total_positives": n_pos,
                "coverage": positives_at_k / n_pos,
                "count_ratio": f"{positives_at_k}/{n_pos}",
                "random_expected": random_expected,
                "delta_vs_random": positives_at_k - random_expected,
                "n_samples": n_samples,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def analyze_llm_intersection_coverage(df: pd.DataFrame) -> Dict:
    """Coverage analysis by LLM intersections."""
    stats = {
        "inter3": int((df["vote_inter3"] == 1).sum()),
        "inter2": int((df["vote_inter2"] == 1).sum()),
        "union": int((df["vote_union"] == 1).sum()),
        "no_vote": int((df["vote_union"] == 0).sum()),
        "total": int(len(df))
    }

    for target in ["A1", "A2", "Gold"]:
        label_col = f"lbl_{target}"
        if label_col not in df.columns:
            continue

        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        df_target = df[mask].copy()
        y = df_target[label_col].astype(int)

        stats[f"{target}_total"] = int(len(y))
        stats[f"{target}_positives"] = int(y.sum())
        stats[f"{target}_pos_inter3"] = int(y[df_target["vote_inter3"] == 1].sum())
        stats[f"{target}_pos_inter2"] = int(y[df_target["vote_inter2"] == 1].sum())
        stats[f"{target}_pos_union"] = int(y[df_target["vote_union"] == 1].sum())
        stats[f"{target}_pos_no_vote"] = int(y[df_target["vote_union"] == 0].sum())

    return stats


# ============================================================
# REPORTING (MODIFIED - with count/total)
# ============================================================
def print_llm_intersection_analysis(stats: Dict):
    """Print the LLM intersection analysis."""
    print("\n" + "="*70)
    print("LLM INTERSECTION ANALYSIS")
    print("="*70)

    print(f"\n  Full dataset: {stats['total']} entries")
    print(f"    - 3 LLMs agree (inter3):  {stats['inter3']:>6} ({stats['inter3']/stats['total']*100:>5.1f}%)")
    print(f"    - 2+ LLMs agree (inter2): {stats['inter2']:>6} ({stats['inter2']/stats['total']*100:>5.1f}%)")
    print(f"    - 1+ LLM agrees (union):   {stats['union']:>6} ({stats['union']/stats['total']*100:>5.1f}%)")
    print(f"    - No LLM agrees:        {stats['no_vote']:>6} ({stats['no_vote']/stats['total']*100:>5.1f}%)")

    for target in ["A1", "A2", "Gold"]:
        if f"{target}_total" not in stats:
            continue

        n_total = stats[f"{target}_total"]
        n_pos = stats[f"{target}_positives"]

        if n_pos == 0:
            continue

        print(f"\n  {target}: {n_pos} positives out of {n_total} ({n_pos/n_total*100:.1f}%)")
        print(f"    - In inter3:  {stats[f'{target}_pos_inter3']:>4}/{n_pos} ({stats[f'{target}_pos_inter3']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter2:  {stats[f'{target}_pos_inter2']:>4}/{n_pos} ({stats[f'{target}_pos_inter2']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In union:   {stats[f'{target}_pos_union']:>4}/{n_pos} ({stats[f'{target}_pos_union']/n_pos*100:>5.1f}% of positives)")
        print(f"    - Outside union:   {stats[f'{target}_pos_no_vote']:>4}/{n_pos} ({stats[f'{target}_pos_no_vote']/n_pos*100:>5.1f}% of positives)")


def print_comparison_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Comparison table with count/total."""
    print("\n" + "="*140)
    print("COMPARISON OF UNSUPERVISED METHODS (# yes retrieved / total # yes)")
    print("="*140)

    methods = list(results_dict.keys())

    for target in config.targets:
        print(f"\n{'='*140}")
        print(f"TARGET: {target}")
        print(f"{'='*140}")

        df_random = results_dict.get("Random")
        prevalence = None
        n_pos_total = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                n_pos_total = df_t_rand["total_positives"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")
                print(f"  Total number of YES: {n_pos_total}")

        print("\n  METHOD               | AP     | Δ vs Random")
        print("  " + "-"*45)
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = ap - prevalence
                    delta_str = f"+{delta:.3f}" if delta >= 0 else f"{delta:.3f}"
                else:
                    delta_str = "-"
                print(f"  {method:<20} | {ap:.4f} | {delta_str}")

        # Header with the methods
        print(f"\n  {'k':<6}", end="")
        for method in methods:
            print(f" | {method[:14]:<14}", end="")
        print()

        # Sub-header for P@k and count
        print(f"  {'':6}", end="")
        for _ in methods:
            print(f" | P@k   (yes/tot)", end="")
        print()
        print("  " + "-"*(7 + 18 * len(methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in methods:
                df_res = results_dict[method]
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    count = df_t["count"].values[0]
                    total = df_t["total_positives"].values[0]
                    row += f" | {p:.3f} ({count:>3}/{total:<3})"
                else:
                    row += " | -               "
            print(row)


def print_coverage_table(coverage_dict: Dict[str, pd.DataFrame], config: Config):
    """Coverage table with count/total."""
    print("\n" + "="*160)
    print("POSITIVE COVERAGE PER METHOD (yes retrieved / total yes)")
    print("="*160)

    methods = list(coverage_dict.keys())

    for target in config.targets:
        print(f"\n{'='*160}")
        print(f"TARGET: {target}")

        df_first = next(iter(coverage_dict.values()))
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue

        total_pos = df_t["total_positives"].iloc[0]
        n_samples = df_t["n_samples"].iloc[0]
        prevalence = df_t["prevalence"].iloc[0]

        print(f"  Total positives (YES): {total_pos} / {n_samples} (prevalence: {prevalence:.3f})")
        print(f"{'='*160}")

        print(f"\n  {'k':<6}", end="")
        for method in methods:
            print(f" | {method[:16]:<16}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in methods:
            print(f" | yes/tot (R@k)   ", end="")
        print()
        print("  " + "-"*(7 + 19 * len(methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in methods:
                df_cov = coverage_dict[method]
                df_tk = df_cov[(df_cov["target"] == target) & (df_cov["k"] == k)]
                if len(df_tk) > 0:
                    retrieved = int(df_tk["positives_retrieved"].values[0])
                    total = int(df_tk["total_positives"].values[0])
                    recall = df_tk["coverage"].values[0]
                    row += f" | {retrieved:>3}/{total:<3} ({recall:.2f}) "
                else:
                    row += " | -                "
            print(row)


def print_detailed_count_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Detailed table with only the counts (yes retrieved / total yes)."""
    print("\n" + "="*140)
    print("DETAIL: NUMBER OF YES RETRIEVED / TOTAL NUMBER OF YES")
    print("="*140)

    methods = [m for m in results_dict.keys() if m != "Random"]

    for target in config.targets:
        print(f"\n{'='*140}")
        print(f"TARGET: {target}")
        print(f"{'='*140}")

        # Get the total number of positives
        df_first = results_dict[methods[0]]
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue
        total_pos = df_t["total_positives"].iloc[0]
        print(f"  Total number of YES: {total_pos}")

        # Header
        print(f"\n  {'k':<6} | {'Random':<12}", end="")
        for method in methods:
            print(f" | {method[:12]:<12}", end="")
        print()
        print("  " + "-"*(7 + 15 + 15 * len(methods)))

        for k in config.k_values:
            row = f"  {k:<6}"

            # Random
            df_rand = results_dict["Random"]
            df_tk = df_rand[(df_rand["target"] == target) & (df_rand["k"] == k)]
            if len(df_tk) > 0:
                count = int(df_tk["count"].values[0])
                row += f" | {count:>4}/{total_pos:<4}   "
            else:
                row += " | -            "

            # Other methods
            for method in methods:
                df_res = results_dict[method]
                df_tk = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_tk) > 0:
                    count = int(df_tk["count"].values[0])
                    row += f" | {count:>4}/{total_pos:<4}  "
                else:
                    row += " | -            "
            print(row)


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualisation."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple', 'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method, df_res in results_dict.items():
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric}")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_coverage_curves(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Coverage curves."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        ax = axes[i]
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            ax.plot(df_t_first["k"], df_t_first["random_expected"],
                   color='black', linestyle='--', linewidth=1.5,
                   label='Random (expected)', alpha=0.7)

            # Horizontal line for the total positives
            total_pos = df_t_first["total_positives"].iloc[0]
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        for method, df_cov in coverage_dict.items():
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["positives_retrieved"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k")
        ax.set_ylabel("YES retrieved")
        ax.set_title(f"{target} - # of YES retrieved")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_gain_vs_random(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of relative gain over chance."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        ax = axes[i]
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

        for method, df_cov in coverage_dict.items():
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["delta_vs_random"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Gain vs Random (number of YES)")
        ax.set_title(f"{target} - Gain over chance")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_count_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of the number of retrieved YES per method."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple', 'Heuristic': 'magenta'
    }

    for i, target in enumerate(config.targets):
        ax = axes[i]

        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["count"],
                       color=colors.get(method, 'gray'),
                       marker='o' if method != 'Random' else '.',
                       linewidth=2 if method != 'Random' else 1,
                       linestyle='--' if method == 'Random' else '-',
                       label=method, alpha=0.7 if method == 'Random' else 1.0)

                # Horizontal line for the total
                if method != 'Random':
                    total_pos = df_t["total_positives"].iloc[0]

        # Add the total line
        ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                  label=f'Total YES ({total_pos})', alpha=0.5)

        ax.set_xlabel("k")
        ax.set_ylabel("Number of YES retrieved")
        ax.set_title(f"{target} - YES retrieved vs k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(filter_union: bool = False, compute_crossencoder: bool = True):
    """Main pipeline."""
    print("="*70)
    print("FULLY UNSUPERVISED RANKING PIPELINE")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print("="*70)
    print("""
TIE HANDLING (TIE-BREAK):
------------------------------------
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3) produce
0/1 scores creating ties. To break them deterministically:

  Score_final = Score_LLM + ε × Score_CrossEncoder  (ε = 1e-6)

Documents with the same LLM vote are ordered by their CrossEncoder score.
The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) have no
significant ties and use a direct sort.
""")

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_unsupervised{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # Load & prepare
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # LLM intersection analysis
    llm_stats = analyze_llm_intersection_coverage(df)
    print_llm_intersection_analysis(llm_stats)

    # Compute all scores
    print("\n" + "="*70)
    print("COMPUTING RANKINGS")
    print("="*70)

    all_scores = compute_all_ranking_scores(df)

    results_dict = {}
    coverage_dict = {}

    print("\n  [1/8] Random baseline...")
    results_dict["Random"] = compute_random_baseline(df, config)

    for i, (method_name, scores) in enumerate(all_scores.items(), 2):
        print(f"  [{i}/8] {method_name}...")
        results_dict[method_name] = evaluate_method(df, scores, method_name, config)
        coverage_dict[method_name] = analyze_positive_coverage(df, scores, method_name, config)

    # Reporting
    print_comparison_table(results_dict, config)
    print_coverage_table(coverage_dict, config)
    print_detailed_count_table(results_dict, config)

    # Plots
    plot_comparison(results_dict, config, f"{config.output_dir}/comparison.png")
    plot_coverage_curves(coverage_dict, config, f"{config.output_dir}/coverage.png")
    plot_gain_vs_random(coverage_dict, config, f"{config.output_dir}/gain_vs_random.png")
    plot_count_comparison(results_dict, config, f"{config.output_dir}/count_comparison.png")

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    for name, scores in all_scores.items():
        df[f"score_{name.lower().replace('-', '_')}"] = scores

    df.to_csv(f"{config.output_dir}/df_with_scores_{timestamp}.csv", index=False)

    df_results = pd.concat(list(results_dict.values()), ignore_index=True)
    df_results.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    df_coverage = pd.concat(list(coverage_dict.values()), ignore_index=True)
    df_coverage.to_csv(f"{config.output_dir}/coverage_{timestamp}.csv", index=False)

    with open(f"{config.output_dir}/llm_stats_{timestamp}.json", "w") as f:
        json.dump(llm_stats, f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict, coverage_dict, llm_stats


if __name__ == "__main__":
    df, results, coverage, llm_stats = main(filter_union=False, compute_crossencoder=True)

# 4 LLMs

In [ ]:
# ============================================================
# FULLY UNSUPERVISED PIPELINE - NO LABELS USED
# 4 LLMs VERSION: Llama, Qwen, Mistral, Qwen32B
# ============================================================
"""
Fully unsupervised pipeline for legal ranking.

TIE HANDLING (TIE-BREAK):
====================================
The binary methods (LLM_Union, LLM_Inter*, etc.) produce
binary scores (0 or 1), creating many ties. To break them:

  Score final = Score_LLM + ε × Score_CrossEncoder

where ε = 1e-6, small enough for the LLM score to dominate.

COMBINATIONS TESTED (4 LLMs):
==============================
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3_all: the 4 combinations of 3 LLMs:
    * Inter3_LQM: Llama + Qwen + Mistral
    * Inter3_LQQ32: Llama + Qwen + Qwen32B
    * Inter3_LMQ32: Llama + Mistral + Qwen32B
    * Inter3_QMQ32: Qwen + Mistral + Qwen32B
- Inter3: at least 3 of the 4 LLMs say yes
- Inter4: all 4 LLMs say yes

Reference: Voorhees (2000), TREC evaluation methodology
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import json
from itertools import combinations

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess
    packages = [
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]
    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    qwen32b_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    random_seed: int = 42

    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.qwen32b_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-32b.xlsx"
        self.output_dir = f"artifacts/outputs_unsupervised_4llm"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        dcg = float(np.sum(y_sorted / np.log2(np.arange(2, k + 2))))

        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:k]
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k + 2))))

        return dcg / idcg if idcg > 0 else 0.0

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 4 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA (4 LLMs)")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)
    df_qwen32b = pd.read_excel(config.qwen32b_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)} | Qwen32B: {len(df_qwen32b)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]
    for df in [df_main, df_llama, df_qwen, df_mistral, df_qwen32b]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    # Llama
    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    # Qwen (7B)
    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    # Mistral
    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    # Qwen 32B
    pred_col_q32 = None
    for col in df_qwen32b.columns:
        if "pred" in col.lower() and col not in key_cols:
            pred_col_q32 = col
            break

    if pred_col_q32 is None:
        for col in df_qwen32b.columns:
            if col not in key_cols and col != "_key":
                vals = df_qwen32b[col].dropna().unique()
                if set(vals).issubset({0, 1, "0", "1", "oui", "non", "Oui", "Non"}):
                    pred_col_q32 = col
                    break

    if pred_col_q32:
        df_qwen32b = df_qwen32b.rename(columns={pred_col_q32: "Qwen32B_Pred"})
    else:
        df_qwen32b["Qwen32B_Pred"] = 0
        print("  WARNING: Could not find prediction column for Qwen32B, defaulting to 0")

    qwen32b_cols = df_qwen32b[["_key", "Qwen32B_Pred"]].copy()

    # Merge all
    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")
    df = df.merge(qwen32b_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)
    df = df.drop(columns=["_key"])

    return df


# ============================================================
# FEATURES
# ============================================================
def compute_tfidf_similarity(df: pd.DataFrame) -> np.ndarray:
    """TF-IDF similarity."""
    stopwords_fr = [
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ]

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2,
        max_df=0.95, stop_words=stopwords_fr, lowercase=True
    )
    vectorizer.fit(texts + articles)

    tfidf_texts = vectorizer.transform(texts)
    tfidf_articles = vectorizer.transform(articles)

    return np.array([
        cosine_similarity(tfidf_texts[i], tfidf_articles[i])[0, 0]
        for i in range(len(texts))
    ])


def compute_bm25_scores(df: pd.DataFrame) -> np.ndarray:
    """BM25 scores."""
    from rank_bm25 import BM25Okapi

    stopwords = set(["le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "à"])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]

    articles = df["article_text"].fillna("").tolist()
    texts = df["text"].fillna("").tolist()

    tokenized_articles = [tokenize(a) for a in articles]
    bm25 = BM25Okapi(tokenized_articles)

    scores = []
    for i, text in enumerate(texts):
        query = tokenize(text)
        if query:
            all_scores = bm25.get_scores(query)
            scores.append(all_scores[i])
        else:
            scores.append(0.0)

    scores = np.array(scores)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder 0-shot."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all UNSUPERVISED features with 4 LLMs."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION (UNSUPERVISED - 4 LLMs)")
    print("="*70)

    df = df.copy()

    # Individual votes of the 4 LLMs
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)
    df["vote_qwen32b"] = (df["Qwen32B_Pred"] == 1).astype(int)

    # Sum of votes (0-4)
    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"] + df["vote_qwen32b"]

    # Union: at least 1 LLM says yes
    df["vote_union"] = (df["vote_sum"] >= 1).astype(int)

    # Inter2: at least 2 LLMs say yes
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)

    # Inter3: at least 3 of the 4 LLMs say yes
    df["vote_inter3"] = (df["vote_sum"] >= 3).astype(int)

    # Inter4: all 4 LLMs say yes
    df["vote_inter4"] = (df["vote_sum"] == 4).astype(int)

    # The 4 specific combinations of 3 LLMs (exact intersection of 3 specific LLMs)
    # L = Llama, Q = Qwen, M = Mistral, Q32 = Qwen32B
    df["vote_inter3_LQM"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) & (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter3_LQQ32"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_LMQ32"] = ((df["vote_llama"] == 1) & (df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_QMQ32"] = ((df["vote_qwen"] == 1) & (df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)

    print("  [1/3] TF-IDF...")
    df["sim_tfidf"] = compute_tfidf_similarity(df)

    print("  [2/3] BM25...")
    df["sim_bm25"] = compute_bm25_scores(df)

    if compute_crossencoder:
        print("  [3/3] Cross-Encoder...")
        df["sim_crossencoder"] = compute_cross_encoder_scores(df, config)
    else:
        df["sim_crossencoder"] = 0.0

    return df


# ============================================================
# RANKING SCORES (with integrated tie-break)
# ============================================================
def compute_all_ranking_scores(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    """
    Compute the ranking scores for all methods.

    TIE-BREAK HANDLING:
    - Continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic): direct scores
    - Binary methods (LLM_*): score = vote + ε × CrossEncoder to break ties
    """
    scores = {}

    # Continuous methods - no tie-break needed
    scores["TF-IDF"] = df["sim_tfidf"].values
    scores["BM25"] = df["sim_bm25"].values
    scores["CrossEncoder"] = df["sim_crossencoder"].values

    # Heuristic - weighted combination (near-unique scores) - updated for 4 LLMs
    weights = {
        "vote_llama": 0.25, "vote_qwen": 0.25, "vote_mistral": 0.25, "vote_qwen32b": 0.25,
        "vote_inter2": 0.3, "vote_inter3": 0.5, "vote_inter4": 1.0,
        "sim_tfidf": 0.2, "sim_bm25": 0.2, "sim_crossencoder": 0.4,
    }
    heuristic = np.zeros(len(df))
    for feature, weight in weights.items():
        if feature in df.columns:
            heuristic += weight * df[feature].values
    scores["Heuristic"] = heuristic

    # Binary methods - tie-break by CrossEncoder
    # Score = vote + ε × CrossEncoder_normalized
    eps = 1e-6
    ce_norm = df["sim_crossencoder"].values
    if ce_norm.max() > ce_norm.min():
        ce_norm = (ce_norm - ce_norm.min()) / (ce_norm.max() - ce_norm.min())

    # Main methods
    scores["LLM_Union"] = df["vote_union"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter2"] = df["vote_inter2"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter3"] = df["vote_inter3"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter4"] = df["vote_inter4"].values.astype(float) + eps * ce_norm

    # The 4 specific combinations of 3 LLMs
    scores["Inter3_LQM"] = df["vote_inter3_LQM"].values.astype(float) + eps * ce_norm
    scores["Inter3_LQQ32"] = df["vote_inter3_LQQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_LMQ32"] = df["vote_inter3_LMQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_QMQ32"] = df["vote_inter3_QMQ32"].values.astype(float) + eps * ce_norm

    return scores


# ============================================================
# EVALUATION (with count and ratio)
# ============================================================
def evaluate_method(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate a ranking with the number of retrieved yes."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            count_k = Metrics.count_at_k(y, s, k)
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": count_k,
                "total_positives": n_pos,
                "count_ratio": f"{count_k}/{n_pos}",
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """Random baseline = prevalence."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count": int(expected_pos_at_k),
                "total_positives": n_pos,
                "count_ratio": f"{int(expected_pos_at_k)}/{n_pos}",
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# COVERAGE ANALYSIS
# ============================================================
def analyze_positive_coverage(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Analysis: how many positives retrieved at each k."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))

        if n_pos == 0:
            continue

        prevalence = n_pos / n_samples
        order = np.argsort(s)[::-1]
        y_sorted = y[order]

        for k in config.k_values:
            k_actual = min(k, n_samples)
            positives_at_k = int(np.sum(y_sorted[:k_actual]))
            random_expected = k_actual * prevalence

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "positives_retrieved": positives_at_k,
                "total_positives": n_pos,
                "coverage": positives_at_k / n_pos,
                "count_ratio": f"{positives_at_k}/{n_pos}",
                "random_expected": random_expected,
                "delta_vs_random": positives_at_k - random_expected,
                "n_samples": n_samples,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def analyze_llm_intersection_coverage(df: pd.DataFrame) -> Dict:
    """Coverage analysis by LLM intersections (4 LLMs)."""
    stats = {
        "inter4": int((df["vote_inter4"] == 1).sum()),
        "inter3": int((df["vote_inter3"] == 1).sum()),
        "inter3_LQM": int((df["vote_inter3_LQM"] == 1).sum()),
        "inter3_LQQ32": int((df["vote_inter3_LQQ32"] == 1).sum()),
        "inter3_LMQ32": int((df["vote_inter3_LMQ32"] == 1).sum()),
        "inter3_QMQ32": int((df["vote_inter3_QMQ32"] == 1).sum()),
        "inter2": int((df["vote_inter2"] == 1).sum()),
        "union": int((df["vote_union"] == 1).sum()),
        "no_vote": int((df["vote_union"] == 0).sum()),
        "total": int(len(df))
    }

    # Per-LLM stats
    stats["llama_yes"] = int((df["vote_llama"] == 1).sum())
    stats["qwen_yes"] = int((df["vote_qwen"] == 1).sum())
    stats["mistral_yes"] = int((df["vote_mistral"] == 1).sum())
    stats["qwen32b_yes"] = int((df["vote_qwen32b"] == 1).sum())

    for target in ["A1", "A2", "Gold"]:
        label_col = f"lbl_{target}"
        if label_col not in df.columns:
            continue

        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        df_target = df[mask].copy()
        y = df_target[label_col].astype(int)

        stats[f"{target}_total"] = int(len(y))
        stats[f"{target}_positives"] = int(y.sum())
        stats[f"{target}_pos_inter4"] = int(y[df_target["vote_inter4"] == 1].sum())
        stats[f"{target}_pos_inter3"] = int(y[df_target["vote_inter3"] == 1].sum())
        stats[f"{target}_pos_inter3_LQM"] = int(y[df_target["vote_inter3_LQM"] == 1].sum())
        stats[f"{target}_pos_inter3_LQQ32"] = int(y[df_target["vote_inter3_LQQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_LMQ32"] = int(y[df_target["vote_inter3_LMQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_QMQ32"] = int(y[df_target["vote_inter3_QMQ32"] == 1].sum())
        stats[f"{target}_pos_inter2"] = int(y[df_target["vote_inter2"] == 1].sum())
        stats[f"{target}_pos_union"] = int(y[df_target["vote_union"] == 1].sum())
        stats[f"{target}_pos_no_vote"] = int(y[df_target["vote_union"] == 0].sum())

    return stats


# ============================================================
# REPORTING (with count/total)
# ============================================================
def print_llm_intersection_analysis(stats: Dict):
    """Print the LLM intersection analysis (4 LLMs)."""
    print("\n" + "="*70)
    print("LLM INTERSECTION ANALYSIS (4 LLMs)")
    print("="*70)

    print(f"\n  Full dataset: {stats['total']} entries")
    print(f"\n  Individual votes:")
    print(f"    - Llama:   {stats['llama_yes']:>6} ({stats['llama_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen:    {stats['qwen_yes']:>6} ({stats['qwen_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Mistral: {stats['mistral_yes']:>6} ({stats['mistral_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen32B: {stats['qwen32b_yes']:>6} ({stats['qwen32b_yes']/stats['total']*100:>5.1f}%)")

    print(f"\n  Intersections:")
    print(f"    - 4 LLMs agree (inter4):  {stats['inter4']:>6} ({stats['inter4']/stats['total']*100:>5.1f}%)")
    print(f"    - 3+ LLMs agree (inter3): {stats['inter3']:>6} ({stats['inter3']/stats['total']*100:>5.1f}%)")
    print(f"    - 2+ LLMs agree (inter2): {stats['inter2']:>6} ({stats['inter2']/stats['total']*100:>5.1f}%)")
    print(f"    - 1+ LLM agrees (union):   {stats['union']:>6} ({stats['union']/stats['total']*100:>5.1f}%)")
    print(f"    - No LLM agrees:        {stats['no_vote']:>6} ({stats['no_vote']/stats['total']*100:>5.1f}%)")

    print(f"\n  Specific combinations of 3 LLMs:")
    print(f"    - Llama+Qwen+Mistral (LQM):     {stats['inter3_LQM']:>6} ({stats['inter3_LQM']/stats['total']*100:>5.1f}%)")
    print(f"    - Llama+Qwen+Qwen32B (LQQ32):   {stats['inter3_LQQ32']:>6} ({stats['inter3_LQQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - Llama+Mistral+Qwen32B (LMQ32):{stats['inter3_LMQ32']:>6} ({stats['inter3_LMQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen+Mistral+Qwen32B (QMQ32): {stats['inter3_QMQ32']:>6} ({stats['inter3_QMQ32']/stats['total']*100:>5.1f}%)")

    for target in ["A1", "A2", "Gold"]:
        if f"{target}_total" not in stats:
            continue

        n_total = stats[f"{target}_total"]
        n_pos = stats[f"{target}_positives"]

        if n_pos == 0:
            continue

        print(f"\n  {target}: {n_pos} positives out of {n_total} ({n_pos/n_total*100:.1f}%)")
        print(f"    - In inter4:     {stats[f'{target}_pos_inter4']:>4}/{n_pos} ({stats[f'{target}_pos_inter4']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter3:     {stats[f'{target}_pos_inter3']:>4}/{n_pos} ({stats[f'{target}_pos_inter3']/n_pos*100:>5.1f}% of positives)")
        print(f"      * LQM:           {stats[f'{target}_pos_inter3_LQM']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LQM']/n_pos*100:>5.1f}%)")
        print(f"      * LQQ32:         {stats[f'{target}_pos_inter3_LQQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LQQ32']/n_pos*100:>5.1f}%)")
        print(f"      * LMQ32:         {stats[f'{target}_pos_inter3_LMQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LMQ32']/n_pos*100:>5.1f}%)")
        print(f"      * QMQ32:         {stats[f'{target}_pos_inter3_QMQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_QMQ32']/n_pos*100:>5.1f}%)")
        print(f"    - In inter2:     {stats[f'{target}_pos_inter2']:>4}/{n_pos} ({stats[f'{target}_pos_inter2']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In union:      {stats[f'{target}_pos_union']:>4}/{n_pos} ({stats[f'{target}_pos_union']/n_pos*100:>5.1f}% of positives)")
        print(f"    - Outside union:      {stats[f'{target}_pos_no_vote']:>4}/{n_pos} ({stats[f'{target}_pos_no_vote']/n_pos*100:>5.1f}% of positives)")


def print_comparison_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Comparison table with count/total."""
    print("\n" + "="*160)
    print("COMPARISON OF UNSUPERVISED METHODS (4 LLMs) - # yes retrieved / total # yes")
    print("="*160)

    # Main methods only for the first table
    main_methods = ["Random", "TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                   "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4"]
    main_methods = [m for m in main_methods if m in results_dict]

    for target in config.targets:
        print(f"\n{'='*160}")
        print(f"TARGET: {target}")
        print(f"{'='*160}")

        df_random = results_dict.get("Random")
        prevalence = None
        n_pos_total = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                n_pos_total = df_t_rand["total_positives"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")
                print(f"  Total number of YES: {n_pos_total}")

        # Table AP
        print("\n  METHOD                | AP     | Δ vs Random")
        print("  " + "-"*50)
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = ap - prevalence
                    delta_str = f"+{delta:.3f}" if delta >= 0 else f"{delta:.3f}"
                else:
                    delta_str = "-"
                print(f"  {method:<22} | {ap:.4f} | {delta_str}")

        # P@k table for the main methods
        print(f"\n  --- P@k for the main methods (yes/tot) ---")
        print(f"  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:14]:<14}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | P@k   (yes/tot)", end="")
        print()
        print("  " + "-"*(7 + 18 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_res = results_dict[method]
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    count = df_t["count"].values[0]
                    total = df_t["total_positives"].values[0]
                    row += f" | {p:.3f} ({count:>3}/{total:<3})"
                else:
                    row += " | -               "
            print(row)

        # P@k table for the combinations of 3 LLMs
        inter3_methods = ["Inter3_LQM", "Inter3_LQQ32", "Inter3_LMQ32", "Inter3_QMQ32"]
        inter3_methods = [m for m in inter3_methods if m in results_dict]

        if inter3_methods:
            print(f"\n  --- P@k for combinations of 3 LLMs (yes/tot) ---")
            print(f"  {'k':<6}", end="")
            for method in inter3_methods:
                print(f" | {method:<16}", end="")
            print()
            print(f"  {'':6}", end="")
            for _ in inter3_methods:
                print(f" | P@k   (yes/tot) ", end="")
            print()
            print("  " + "-"*(7 + 19 * len(inter3_methods)))

            for k in [10, 50, 100, 200, 300, 500, 1000]:
                if k not in config.k_values:
                    continue
                row = f"  {k:<6}"
                for method in inter3_methods:
                    df_res = results_dict[method]
                    df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                    if len(df_t) > 0:
                        p = df_t["P@k"].values[0]
                        count = df_t["count"].values[0]
                        total = df_t["total_positives"].values[0]
                        row += f" | {p:.3f} ({count:>3}/{total:<3}) "
                    else:
                        row += " | -                "
                print(row)


def print_coverage_table(coverage_dict: Dict[str, pd.DataFrame], config: Config):
    """Coverage table with count/total."""
    print("\n" + "="*180)
    print("POSITIVE COVERAGE PER METHOD (4 LLMs) - yes retrieved / total yes")
    print("="*180)

    # Main methods
    main_methods = ["TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                   "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4"]
    main_methods = [m for m in main_methods if m in coverage_dict]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")

        df_first = next(iter(coverage_dict.values()))
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue

        total_pos = df_t["total_positives"].iloc[0]
        n_samples = df_t["n_samples"].iloc[0]
        prevalence = df_t["prevalence"].iloc[0]

        print(f"  Total positives (YES): {total_pos} / {n_samples} (prevalence: {prevalence:.3f})")
        print(f"{'='*180}")

        print(f"\n  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:16]:<16}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | yes/tot (R@k)   ", end="")
        print()
        print("  " + "-"*(7 + 19 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_cov = coverage_dict[method]
                df_tk = df_cov[(df_cov["target"] == target) & (df_cov["k"] == k)]
                if len(df_tk) > 0:
                    retrieved = int(df_tk["positives_retrieved"].values[0])
                    total = int(df_tk["total_positives"].values[0])
                    recall = df_tk["coverage"].values[0]
                    row += f" | {retrieved:>3}/{total:<3} ({recall:.2f}) "
                else:
                    row += " | -                "
            print(row)


def print_detailed_count_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Detailed table with only the counts (yes retrieved / total yes)."""
    print("\n" + "="*180)
    print("DETAIL: NUMBER OF YES RETRIEVED / TOTAL NUMBER OF YES (4 LLMs)")
    print("="*180)

    methods = [m for m in results_dict.keys() if m != "Random"]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")
        print(f"{'='*180}")

        # Get the total number of positives
        df_first = results_dict[methods[0]]
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue
        total_pos = df_t["total_positives"].iloc[0]
        print(f"  Total number of YES: {total_pos}")

        # Header
        print(f"\n  {'k':<6} | {'Random':<12}", end="")
        for method in methods:
            print(f" | {method[:12]:<12}", end="")
        print()
        print("  " + "-"*(7 + 15 + 15 * len(methods)))

        for k in config.k_values:
            row = f"  {k:<6}"

            # Random
            df_rand = results_dict["Random"]
            df_tk = df_rand[(df_rand["target"] == target) & (df_rand["k"] == k)]
            if len(df_tk) > 0:
                count = int(df_tk["count"].values[0])
                row += f" | {count:>4}/{total_pos:<4}   "
            else:
                row += " | -            "

            # Other methods
            for method in methods:
                df_res = results_dict[method]
                df_tk = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_tk) > 0:
                    count = int(df_tk["count"].values[0])
                    row += f" | {count:>4}/{total_pos:<4}  "
                else:
                    row += " | -            "
            print(row)


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualisation."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in main_methods:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric}")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_inter3_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualisation of the 4 combinations of 3 LLMs."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'Inter3_LQM': 'blue',
        'Inter3_LQQ32': 'green',
        'Inter3_LMQ32': 'red',
        'Inter3_QMQ32': 'purple',
        'LLM_Inter3': 'orange',
        'LLM_Inter4': 'brown'
    }

    methods_to_plot = ['Random', 'Inter3_LQM', 'Inter3_LQQ32', 'Inter3_LMQ32',
                       'Inter3_QMQ32', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in methods_to_plot:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric} (Inter3 Combinations)")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_coverage_curves(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Coverage curves."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            ax.plot(df_t_first["k"], df_t_first["random_expected"],
                   color='black', linestyle='--', linewidth=1.5,
                   label='Random (expected)', alpha=0.7)

            # Horizontal line for the total positives
            total_pos = df_t_first["total_positives"].iloc[0]
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["positives_retrieved"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k")
        ax.set_ylabel("YES retrieved")
        ax.set_title(f"{target} - # of YES retrieved")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_gain_vs_random(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of relative gain over chance."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["delta_vs_random"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Gain vs Random (number of YES)")
        ax.set_title(f"{target} - Gain over chance")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_count_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of the number of retrieved YES per method."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        total_pos = None

        for method in main_methods:
            if method not in results_dict:
                continue
            df_res = results_dict[method]
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["count"],
                       color=colors.get(method, 'gray'),
                       marker='o' if method != 'Random' else '.',
                       linewidth=2 if method != 'Random' else 1,
                       linestyle='--' if method == 'Random' else '-',
                       label=method, alpha=0.7 if method == 'Random' else 1.0)

                if method != 'Random' and total_pos is None:
                    total_pos = df_t["total_positives"].iloc[0]

        # Add the total line
        if total_pos is not None:
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        ax.set_xlabel("k")
        ax.set_ylabel("Number of YES retrieved")
        ax.set_title(f"{target} - YES retrieved vs k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(filter_union: bool = False, compute_crossencoder: bool = True):
    """Main pipeline."""
    print("="*70)
    print("FULLY UNSUPERVISED RANKING PIPELINE (4 LLMs)")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print("="*70)
    print("""
4 LLMs: Llama, Qwen, Mistral, Qwen32B

COMBINATIONS TESTED:
---------------------
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3: at least 3 of the 4 LLMs say yes
- Inter4: all 4 LLMs say yes

Specific combinations of 3:
- Inter3_LQM: Llama + Qwen + Mistral
- Inter3_LQQ32: Llama + Qwen + Qwen32B
- Inter3_LMQ32: Llama + Mistral + Qwen32B
- Inter3_QMQ32: Qwen + Mistral + Qwen32B

TIE HANDLING (TIE-BREAK):
------------------------------------
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3, LLM_Inter4)
produce 0/1 scores creating ties. To break them
deterministically:

  Score_final = Score_LLM + ε × Score_CrossEncoder  (ε = 1e-6)

Documents with the same LLM vote are ordered by their CrossEncoder score.
The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) have no
significant ties and use a direct sort.
""")

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_unsupervised_4llm{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # Load & prepare
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # LLM intersection analysis
    llm_stats = analyze_llm_intersection_coverage(df)
    print_llm_intersection_analysis(llm_stats)

    # Compute all scores
    print("\n" + "="*70)
    print("COMPUTING RANKINGS")
    print("="*70)

    all_scores = compute_all_ranking_scores(df)

    results_dict = {}
    coverage_dict = {}

    n_methods = len(all_scores) + 1  # +1 for Random
    print(f"\n  [1/{n_methods}] Random baseline...")
    results_dict["Random"] = compute_random_baseline(df, config)

    for i, (method_name, scores) in enumerate(all_scores.items(), 2):
        print(f"  [{i}/{n_methods}] {method_name}...")
        results_dict[method_name] = evaluate_method(df, scores, method_name, config)
        coverage_dict[method_name] = analyze_positive_coverage(df, scores, method_name, config)

    # Reporting
    print_comparison_table(results_dict, config)
    print_coverage_table(coverage_dict, config)
    print_detailed_count_table(results_dict, config)

    # Plots
    plot_comparison(results_dict, config, f"{config.output_dir}/comparison_main.png")
    plot_inter3_comparison(results_dict, config, f"{config.output_dir}/comparison_inter3.png")
    plot_coverage_curves(coverage_dict, config, f"{config.output_dir}/coverage.png")
    plot_gain_vs_random(coverage_dict, config, f"{config.output_dir}/gain_vs_random.png")
    plot_count_comparison(results_dict, config, f"{config.output_dir}/count_comparison.png")

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    for name, scores in all_scores.items():
        df[f"score_{name.lower().replace('-', '_')}"] = scores

    df.to_csv(f"{config.output_dir}/df_with_scores_{timestamp}.csv", index=False)

    df_results = pd.concat(list(results_dict.values()), ignore_index=True)
    df_results.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    df_coverage = pd.concat(list(coverage_dict.values()), ignore_index=True)
    df_coverage.to_csv(f"{config.output_dir}/coverage_{timestamp}.csv", index=False)

    with open(f"{config.output_dir}/llm_stats_{timestamp}.json", "w") as f:
        json.dump(llm_stats, f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict, coverage_dict, llm_stats


if __name__ == "__main__":
    df, results, coverage, llm_stats = main(filter_union=False, compute_crossencoder=True)

# with agree disagree

In [ ]:
# ============================================================
# FULLY UNSUPERVISED PIPELINE - NO LABELS USED
# 4 LLMs VERSION: Llama, Qwen, Mistral, Qwen32B
# ============================================================
"""
Fully unsupervised pipeline for legal ranking.

TIE HANDLING (TIE-BREAK):
====================================
The binary methods (LLM_Union, LLM_Inter*, etc.) produce
binary scores (0 or 1), creating many ties. To break them:

  Score final = Score_LLM + ε × Score_CrossEncoder

where ε = 1e-6, small enough for the LLM score to dominate.

COMBINATIONS TESTED (4 LLMs):
==============================
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3_all: the 4 combinations of 3 LLMs:
    * Inter3_LQM: Llama + Qwen + Mistral
    * Inter3_LQQ32: Llama + Qwen + Qwen32B
    * Inter3_LMQ32: Llama + Mistral + Qwen32B
    * Inter3_QMQ32: Qwen + Mistral + Qwen32B
- Inter3: at least 3 of the 4 LLMs say yes
- Inter4: all 4 LLMs say yes

Reference: Voorhees (2000), TREC evaluation methodology
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import json
from itertools import combinations

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
def install_dependencies():
    """Install the required dependencies."""
    import subprocess
    packages = [
        ("rank_bm25", "rank-bm25"),
        ("sentence_transformers", "sentence-transformers"),
    ]
    for module_name, pip_name in packages:
        try:
            __import__(module_name)
        except ImportError:
            print(f"  Installing {pip_name}...")
            subprocess.run(["pip", "install", pip_name, "-q"], check=True)


# ============================================================
# CONFIGURATION
# ============================================================
@dataclass
class Config:
    base_path: str = "."  # repo root; derived/heavy outputs -> artifacts/ (not shipped -- see DATA.md)
    main_dataset: str = ""
    llama_predictions: str = ""
    qwen_predictions: str = ""
    mistral_predictions: str = ""
    qwen32b_predictions: str = ""
    output_dir: str = ""

    targets: List[str] = field(default_factory=lambda: ["A1", "A2", "Gold"])
    # Targets split by agreement/disagreement
    targets_by_agreement: List[str] = field(default_factory=lambda: [
        "Gold_Agree", "Gold_Disagree",
        "A1_Agree", "A1_Disagree",
        "A2_Agree", "A2_Disagree"
    ])
    k_values: List[int] = field(default_factory=lambda: [10, 50, 100, 150, 200, 250, 300, 400, 500, 600, 800, 1000])
    random_seed: int = 42

    cross_encoder_model: str = "BAAI/bge-reranker-v2-m3"
    cross_encoder_batch_size: int = 32
    cross_encoder_max_length: int = 512

    def __post_init__(self):
        self.main_dataset = f"{self.base_path}/DATA/outputs/benchmark.csv"
        self.llama_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/llama3.1-8b.xlsx"
        self.qwen_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-7b.xlsx"
        self.mistral_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/mistral-nemo-12b.xlsx"
        self.qwen32b_predictions = f"{self.base_path}/DATA/outputs/predictions/zeroshot_llms/qwen2.5-32b.xlsx"
        self.output_dir = f"artifacts/outputs_unsupervised_4llm"


# ============================================================
# METRICS
# ============================================================
class Metrics:
    @staticmethod
    def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.mean(y_true[order][:k]))

    @staticmethod
    def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        total_pos = np.sum(y_true)
        if total_pos == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return float(np.sum(y_true[order][:k]) / total_pos)

    @staticmethod
    def count_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> int:
        if len(y_true) == 0 or k == 0:
            return 0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        return int(np.sum(y_true[order][:k]))

    @staticmethod
    def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
        if len(y_true) == 0 or k == 0:
            return 0.0
        k = min(k, len(y_true))
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order][:k]
        dcg = float(np.sum(y_sorted / np.log2(np.arange(2, k + 2))))

        ideal_order = np.argsort(y_true)[::-1]
        y_ideal = y_true[ideal_order][:k]
        idcg = float(np.sum(y_ideal / np.log2(np.arange(2, k + 2))))

        return dcg / idcg if idcg > 0 else 0.0

    @staticmethod
    def average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
        if len(y_true) == 0 or np.sum(y_true) == 0:
            return 0.0
        order = np.argsort(scores)[::-1]
        y_sorted = y_true[order]
        precisions = []
        n_pos_seen = 0
        for i, y in enumerate(y_sorted):
            if y == 1:
                n_pos_seen += 1
                precisions.append(n_pos_seen / (i + 1))
        return float(np.mean(precisions)) if precisions else 0.0


# ============================================================
# DATA LOADING
# ============================================================
def clean_label(x) -> Optional[int]:
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes", "true"]:
        return 1
    if s in ["non", "0", "no", "false"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan


def load_data(config: Config) -> pd.DataFrame:
    """Load and merge the data with 4 LLMs."""
    print("\n" + "="*70)
    print("LOADING DATA (4 LLMs)")
    print("="*70)

    df_main = pd.read_csv(config.main_dataset)
    df_llama = pd.read_excel(config.llama_predictions)
    df_qwen = pd.read_excel(config.qwen_predictions)
    df_mistral = pd.read_excel(config.mistral_predictions)
    df_qwen32b = pd.read_excel(config.qwen32b_predictions)

    print(f"  Main: {len(df_main)} | Llama: {len(df_llama)} | Qwen: {len(df_qwen)} | Mistral: {len(df_mistral)} | Qwen32B: {len(df_qwen32b)}")

    key_cols = ["decision_id", "chunk_id", "pred_art"]
    for df in [df_main, df_llama, df_qwen, df_mistral, df_qwen32b]:
        df["_key"] = df[key_cols].astype(str).agg("_".join, axis=1)

    # Llama
    llama_cols = df_llama[["_key", "Llama_Pred"]].copy()

    # Qwen (7B)
    if "Qwen_Pred" not in df_qwen.columns:
        df_qwen = df_qwen.rename(columns={"Llama_Pred": "Qwen_Pred"})
    qwen_cols = df_qwen[["_key", "Qwen_Pred"]].copy()

    # Mistral
    if "Mistral_Pred" not in df_mistral.columns:
        pred_col = [c for c in df_mistral.columns if "Pred" in c]
        if pred_col:
            df_mistral = df_mistral.rename(columns={pred_col[0]: "Mistral_Pred"})
        else:
            df_mistral["Mistral_Pred"] = 0
    mistral_cols = df_mistral[["_key", "Mistral_Pred"]].copy()

    # Qwen 32B
    pred_col_q32 = None
    for col in df_qwen32b.columns:
        if "pred" in col.lower() and col not in key_cols:
            pred_col_q32 = col
            break

    if pred_col_q32 is None:
        for col in df_qwen32b.columns:
            if col not in key_cols and col != "_key":
                vals = df_qwen32b[col].dropna().unique()
                if set(vals).issubset({0, 1, "0", "1", "oui", "non", "Oui", "Non"}):
                    pred_col_q32 = col
                    break

    if pred_col_q32:
        df_qwen32b = df_qwen32b.rename(columns={pred_col_q32: "Qwen32B_Pred"})
    else:
        df_qwen32b["Qwen32B_Pred"] = 0
        print("  WARNING: Could not find prediction column for Qwen32B, defaulting to 0")

    qwen32b_cols = df_qwen32b[["_key", "Qwen32B_Pred"]].copy()

    # Merge all
    df = df_main.merge(llama_cols, on="_key", how="left")
    df = df.merge(qwen_cols, on="_key", how="left")
    df = df.merge(mistral_cols, on="_key", how="left")
    df = df.merge(qwen32b_cols, on="_key", how="left")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def compute_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if pd.notna(t) and pd.notna(a) and t == a:
            return t
        if pd.notna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(compute_gold, axis=1)

    # Create labels for the agree/disagree cases
    # Agree = A2 and A1 agree (same value, both non-NaN)
    # Disagree = A2 and A1 disagree (different values, both non-NaN)
    df["is_agree"] = (df["lbl_A2"] == df["lbl_A1"]) & df["lbl_A2"].notna() & df["lbl_A1"].notna()
    df["is_disagree"] = (df["lbl_A2"] != df["lbl_A1"]) & df["lbl_A2"].notna() & df["lbl_A1"].notna()

    # Labels filtered by agreement/disagreement
    df["lbl_Gold_Agree"] = df.apply(lambda r: r["lbl_Gold"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_Gold_Disagree"] = df.apply(lambda r: r["lbl_Gold"] if r["is_disagree"] else np.nan, axis=1)
    df["lbl_A1_Agree"] = df.apply(lambda r: r["lbl_A1"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_A1_Disagree"] = df.apply(lambda r: r["lbl_A1"] if r["is_disagree"] else np.nan, axis=1)
    df["lbl_A2_Agree"] = df.apply(lambda r: r["lbl_A2"] if r["is_agree"] else np.nan, axis=1)
    df["lbl_A2_Disagree"] = df.apply(lambda r: r["lbl_A2"] if r["is_disagree"] else np.nan, axis=1)

    # Stats on agree/disagree
    n_agree = df["is_agree"].sum()
    n_disagree = df["is_disagree"].sum()
    print(f"\n  Agreement/Disagreement between annotators:")
    print(f"    - Agree (A2 == A1):   {n_agree} ({n_agree/len(df)*100:.1f}%)")
    print(f"    - Disagree (A2 != A1): {n_disagree} ({n_disagree/len(df)*100:.1f}%)")

    df = df.drop(columns=["_key"])

    return df


# ============================================================
# FEATURES
# ============================================================
def compute_tfidf_similarity(df: pd.DataFrame) -> np.ndarray:
    """TF-IDF similarity."""
    stopwords_fr = [
        "le", "la", "les", "l", "de", "du", "des", "d", "un", "une",
        "et", "en", "à", "au", "aux", "ce", "cette", "ces", "qui",
        "que", "qu", "quoi", "dont", "où", "ou", "ne", "pas", "plus",
        "par", "pour", "avec", "sans", "sous", "sur", "dans", "entre",
        "est", "sont", "a", "ont", "être", "avoir", "fait", "été"
    ]

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()

    vectorizer = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), min_df=2,
        max_df=0.95, stop_words=stopwords_fr, lowercase=True
    )
    vectorizer.fit(texts + articles)

    tfidf_texts = vectorizer.transform(texts)
    tfidf_articles = vectorizer.transform(articles)

    return np.array([
        cosine_similarity(tfidf_texts[i], tfidf_articles[i])[0, 0]
        for i in range(len(texts))
    ])


def compute_bm25_scores(df: pd.DataFrame) -> np.ndarray:
    """BM25 scores."""
    from rank_bm25 import BM25Okapi

    stopwords = set(["le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "à"])

    def tokenize(text):
        if not text or pd.isna(text):
            return []
        tokens = re.findall(r'\b[a-zàâäéèêëïîôùûüç]+\b', str(text).lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]

    articles = df["article_text"].fillna("").tolist()
    texts = df["text"].fillna("").tolist()

    tokenized_articles = [tokenize(a) for a in articles]
    bm25 = BM25Okapi(tokenized_articles)

    scores = []
    for i, text in enumerate(texts):
        query = tokenize(text)
        if query:
            all_scores = bm25.get_scores(query)
            scores.append(all_scores[i])
        else:
            scores.append(0.0)

    scores = np.array(scores)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores


def compute_cross_encoder_scores(df: pd.DataFrame, config: Config) -> np.ndarray:
    """Cross-encoder 0-shot."""
    print("\n  Computing Cross-Encoder scores...")

    from sentence_transformers import CrossEncoder
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"    Device: {device}")

    model = CrossEncoder(
        config.cross_encoder_model,
        max_length=config.cross_encoder_max_length,
        device=device
    )

    texts = df["text"].fillna("").tolist()
    articles = df["article_text"].fillna("").tolist()
    pairs = [[t, a] for t, a in zip(texts, articles)]

    scores = []
    batch_size = config.cross_encoder_batch_size

    for i in tqdm(range(0, len(pairs), batch_size), desc="    Cross-encoder"):
        batch = pairs[i:i + batch_size]
        batch_scores = model.predict(batch, show_progress_bar=False)
        scores.extend(batch_scores)

    scores = np.array(scores)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    print(f"    Scores: min={scores.min():.3f}, max={scores.max():.3f}, mean={scores.mean():.3f}")

    return scores_normalized


def prepare_features(df: pd.DataFrame, config: Config, compute_crossencoder: bool = True) -> pd.DataFrame:
    """Prepare all UNSUPERVISED features with 4 LLMs."""
    print("\n" + "="*70)
    print("FEATURE PREPARATION (UNSUPERVISED - 4 LLMs)")
    print("="*70)

    df = df.copy()

    # Individual votes of the 4 LLMs
    df["vote_llama"] = (df["Llama_Pred"] == 1).astype(int)
    df["vote_qwen"] = (df["Qwen_Pred"] == 1).astype(int)
    df["vote_mistral"] = (df["Mistral_Pred"] == 1).astype(int)
    df["vote_qwen32b"] = (df["Qwen32B_Pred"] == 1).astype(int)

    # Sum of votes (0-4)
    df["vote_sum"] = df["vote_llama"] + df["vote_qwen"] + df["vote_mistral"] + df["vote_qwen32b"]

    # Union: at least 1 LLM says yes
    df["vote_union"] = (df["vote_sum"] >= 1).astype(int)

    # Inter2: at least 2 LLMs say yes
    df["vote_inter2"] = (df["vote_sum"] >= 2).astype(int)

    # Inter3: at least 3 of the 4 LLMs say yes
    df["vote_inter3"] = (df["vote_sum"] >= 3).astype(int)

    # Inter4: all 4 LLMs say yes
    df["vote_inter4"] = (df["vote_sum"] == 4).astype(int)

    # The 4 specific combinations of 3 LLMs (exact intersection of 3 specific LLMs)
    # L = Llama, Q = Qwen, M = Mistral, Q32 = Qwen32B
    df["vote_inter3_LQM"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) & (df["vote_mistral"] == 1)).astype(int)
    df["vote_inter3_LQQ32"] = ((df["vote_llama"] == 1) & (df["vote_qwen"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_LMQ32"] = ((df["vote_llama"] == 1) & (df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)
    df["vote_inter3_QMQ32"] = ((df["vote_qwen"] == 1) & (df["vote_mistral"] == 1) & (df["vote_qwen32b"] == 1)).astype(int)

    print("  [1/3] TF-IDF...")
    df["sim_tfidf"] = compute_tfidf_similarity(df)

    print("  [2/3] BM25...")
    df["sim_bm25"] = compute_bm25_scores(df)

    if compute_crossencoder:
        print("  [3/3] Cross-Encoder...")
        df["sim_crossencoder"] = compute_cross_encoder_scores(df, config)
    else:
        df["sim_crossencoder"] = 0.0

    return df


# ============================================================
# RANKING SCORES (with integrated tie-break)
# ============================================================
def compute_all_ranking_scores(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    """
    Compute the ranking scores for all methods.

    TIE-BREAK HANDLING:
    - Continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic): direct scores
    - Binary methods (LLM_*): score = vote + ε × CrossEncoder to break ties
    """
    scores = {}

    # Continuous methods - no tie-break needed
    scores["TF-IDF"] = df["sim_tfidf"].values
    scores["BM25"] = df["sim_bm25"].values
    scores["CrossEncoder"] = df["sim_crossencoder"].values

    # Heuristic - weighted combination (near-unique scores) - updated for 4 LLMs
    weights = {
        "vote_llama": 0.25, "vote_qwen": 0.25, "vote_mistral": 0.25, "vote_qwen32b": 0.25,
        "vote_inter2": 0.3, "vote_inter3": 0.5, "vote_inter4": 1.0,
        "sim_tfidf": 0.2, "sim_bm25": 0.2, "sim_crossencoder": 0.4,
    }
    heuristic = np.zeros(len(df))
    for feature, weight in weights.items():
        if feature in df.columns:
            heuristic += weight * df[feature].values
    scores["Heuristic"] = heuristic

    # Binary methods - tie-break by CrossEncoder
    # Score = vote + ε × CrossEncoder_normalized
    eps = 1e-6
    ce_norm = df["sim_crossencoder"].values
    if ce_norm.max() > ce_norm.min():
        ce_norm = (ce_norm - ce_norm.min()) / (ce_norm.max() - ce_norm.min())

    # Main methods
    scores["LLM_Union"] = df["vote_union"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter2"] = df["vote_inter2"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter3"] = df["vote_inter3"].values.astype(float) + eps * ce_norm
    scores["LLM_Inter4"] = df["vote_inter4"].values.astype(float) + eps * ce_norm

    # The 4 specific combinations of 3 LLMs
    scores["Inter3_LQM"] = df["vote_inter3_LQM"].values.astype(float) + eps * ce_norm
    scores["Inter3_LQQ32"] = df["vote_inter3_LQQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_LMQ32"] = df["vote_inter3_LMQ32"].values.astype(float) + eps * ce_norm
    scores["Inter3_QMQ32"] = df["vote_inter3_QMQ32"].values.astype(float) + eps * ce_norm

    return scores


# ============================================================
# EVALUATION (with count and ratio)
# ============================================================
def evaluate_method(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Evaluate a ranking with the number of retrieved yes."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples
        ap = Metrics.average_precision(y, s)

        for k in config.k_values:
            count_k = Metrics.count_at_k(y, s, k)
            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "P@k": Metrics.precision_at_k(y, s, k),
                "R@k": Metrics.recall_at_k(y, s, k),
                "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                "count": count_k,
                "total_positives": n_pos,
                "count_ratio": f"{count_k}/{n_pos}",
                "AP": ap,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def compute_random_baseline(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """Random baseline = prevalence."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        n_samples = len(y)
        n_pos = int(np.sum(y))
        prevalence = n_pos / n_samples

        for k in config.k_values:
            expected_pos_at_k = min(k, n_samples) * prevalence
            recall_at_k = min(expected_pos_at_k, n_pos) / n_pos if n_pos > 0 else 0

            results.append({
                "method": "Random",
                "target": target,
                "k": k,
                "P@k": prevalence,
                "R@k": recall_at_k,
                "NDCG@k": prevalence,
                "count": int(expected_pos_at_k),
                "total_positives": n_pos,
                "count_ratio": f"{int(expected_pos_at_k)}/{n_pos}",
                "AP": prevalence,
                "n_samples": n_samples,
                "n_positives": n_pos,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


# ============================================================
# COVERAGE ANALYSIS
# ============================================================
def analyze_positive_coverage(
    df: pd.DataFrame,
    scores: np.ndarray,
    method_name: str,
    config: Config
) -> pd.DataFrame:
    """Analysis: how many positives retrieved at each k."""
    results = []

    for target in config.targets:
        label_col = f"lbl_{target}"
        mask = df[label_col].notna()

        if mask.sum() == 0:
            continue

        y = df.loc[mask, label_col].values.astype(int)
        s = scores[mask.values]

        n_samples = len(y)
        n_pos = int(np.sum(y))

        if n_pos == 0:
            continue

        prevalence = n_pos / n_samples
        order = np.argsort(s)[::-1]
        y_sorted = y[order]

        for k in config.k_values:
            k_actual = min(k, n_samples)
            positives_at_k = int(np.sum(y_sorted[:k_actual]))
            random_expected = k_actual * prevalence

            results.append({
                "method": method_name,
                "target": target,
                "k": k,
                "positives_retrieved": positives_at_k,
                "total_positives": n_pos,
                "coverage": positives_at_k / n_pos,
                "count_ratio": f"{positives_at_k}/{n_pos}",
                "random_expected": random_expected,
                "delta_vs_random": positives_at_k - random_expected,
                "n_samples": n_samples,
                "prevalence": prevalence
            })

    return pd.DataFrame(results)


def analyze_llm_intersection_coverage(df: pd.DataFrame) -> Dict:
    """Coverage analysis by LLM intersections (4 LLMs)."""
    stats = {
        "inter4": int((df["vote_inter4"] == 1).sum()),
        "inter3": int((df["vote_inter3"] == 1).sum()),
        "inter3_LQM": int((df["vote_inter3_LQM"] == 1).sum()),
        "inter3_LQQ32": int((df["vote_inter3_LQQ32"] == 1).sum()),
        "inter3_LMQ32": int((df["vote_inter3_LMQ32"] == 1).sum()),
        "inter3_QMQ32": int((df["vote_inter3_QMQ32"] == 1).sum()),
        "inter2": int((df["vote_inter2"] == 1).sum()),
        "union": int((df["vote_union"] == 1).sum()),
        "no_vote": int((df["vote_union"] == 0).sum()),
        "total": int(len(df))
    }

    # Per-LLM stats
    stats["llama_yes"] = int((df["vote_llama"] == 1).sum())
    stats["qwen_yes"] = int((df["vote_qwen"] == 1).sum())
    stats["mistral_yes"] = int((df["vote_mistral"] == 1).sum())
    stats["qwen32b_yes"] = int((df["vote_qwen32b"] == 1).sum())

    for target in ["A1", "A2", "Gold"]:
        label_col = f"lbl_{target}"
        if label_col not in df.columns:
            continue

        mask = df[label_col].notna()
        if mask.sum() == 0:
            continue

        df_target = df[mask].copy()
        y = df_target[label_col].astype(int)

        stats[f"{target}_total"] = int(len(y))
        stats[f"{target}_positives"] = int(y.sum())
        stats[f"{target}_pos_inter4"] = int(y[df_target["vote_inter4"] == 1].sum())
        stats[f"{target}_pos_inter3"] = int(y[df_target["vote_inter3"] == 1].sum())
        stats[f"{target}_pos_inter3_LQM"] = int(y[df_target["vote_inter3_LQM"] == 1].sum())
        stats[f"{target}_pos_inter3_LQQ32"] = int(y[df_target["vote_inter3_LQQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_LMQ32"] = int(y[df_target["vote_inter3_LMQ32"] == 1].sum())
        stats[f"{target}_pos_inter3_QMQ32"] = int(y[df_target["vote_inter3_QMQ32"] == 1].sum())
        stats[f"{target}_pos_inter2"] = int(y[df_target["vote_inter2"] == 1].sum())
        stats[f"{target}_pos_union"] = int(y[df_target["vote_union"] == 1].sum())
        stats[f"{target}_pos_no_vote"] = int(y[df_target["vote_union"] == 0].sum())

    return stats


def analyze_agree_disagree_performance(
    df: pd.DataFrame,
    all_scores: Dict[str, np.ndarray],
    config: Config
) -> pd.DataFrame:
    """Analyse performance on agree vs disagree cases."""
    results = []

    # Targets to analyse
    targets_agreement = ["Gold_Agree", "Gold_Disagree", "A1_Agree", "A1_Disagree",
                         "A2_Agree", "A2_Disagree"]

    for method_name, scores in all_scores.items():
        for target in targets_agreement:
            label_col = f"lbl_{target}"
            if label_col not in df.columns:
                continue

            mask = df[label_col].notna()
            if mask.sum() == 0:
                continue

            y = df.loc[mask, label_col].values.astype(int)
            s = scores[mask.values]

            n_samples = len(y)
            n_pos = int(np.sum(y))
            if n_samples == 0:
                continue

            prevalence = n_pos / n_samples
            ap = Metrics.average_precision(y, s)

            # Extract base_target and agreement_type
            parts = target.rsplit('_', 1)
            base_target = parts[0]
            agreement_type = parts[1] if len(parts) > 1 else "All"

            for k in config.k_values:
                count_k = Metrics.count_at_k(y, s, k)
                results.append({
                    "method": method_name,
                    "target": target,
                    "base_target": base_target,
                    "agreement": agreement_type,
                    "k": k,
                    "P@k": Metrics.precision_at_k(y, s, k),
                    "R@k": Metrics.recall_at_k(y, s, k),
                    "NDCG@k": Metrics.ndcg_at_k(y, s, k),
                    "count": count_k,
                    "total_positives": n_pos,
                    "count_ratio": f"{count_k}/{n_pos}",
                    "AP": ap,
                    "n_samples": n_samples,
                    "prevalence": prevalence
                })

    return pd.DataFrame(results)


# ============================================================
# REPORTING (with count/total)
# ============================================================
def print_llm_intersection_analysis(stats: Dict):
    """Print the LLM intersection analysis (4 LLMs)."""
    print("\n" + "="*70)
    print("LLM INTERSECTION ANALYSIS (4 LLMs)")
    print("="*70)

    print(f"\n  Full dataset: {stats['total']} entries")
    print(f"\n  Individual votes:")
    print(f"    - Llama:   {stats['llama_yes']:>6} ({stats['llama_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen:    {stats['qwen_yes']:>6} ({stats['qwen_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Mistral: {stats['mistral_yes']:>6} ({stats['mistral_yes']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen32B: {stats['qwen32b_yes']:>6} ({stats['qwen32b_yes']/stats['total']*100:>5.1f}%)")

    print(f"\n  Intersections:")
    print(f"    - 4 LLMs agree (inter4):  {stats['inter4']:>6} ({stats['inter4']/stats['total']*100:>5.1f}%)")
    print(f"    - 3+ LLMs agree (inter3): {stats['inter3']:>6} ({stats['inter3']/stats['total']*100:>5.1f}%)")
    print(f"    - 2+ LLMs agree (inter2): {stats['inter2']:>6} ({stats['inter2']/stats['total']*100:>5.1f}%)")
    print(f"    - 1+ LLM agrees (union):   {stats['union']:>6} ({stats['union']/stats['total']*100:>5.1f}%)")
    print(f"    - No LLM agrees:        {stats['no_vote']:>6} ({stats['no_vote']/stats['total']*100:>5.1f}%)")

    print(f"\n  Specific combinations of 3 LLMs:")
    print(f"    - Llama+Qwen+Mistral (LQM):     {stats['inter3_LQM']:>6} ({stats['inter3_LQM']/stats['total']*100:>5.1f}%)")
    print(f"    - Llama+Qwen+Qwen32B (LQQ32):   {stats['inter3_LQQ32']:>6} ({stats['inter3_LQQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - Llama+Mistral+Qwen32B (LMQ32):{stats['inter3_LMQ32']:>6} ({stats['inter3_LMQ32']/stats['total']*100:>5.1f}%)")
    print(f"    - Qwen+Mistral+Qwen32B (QMQ32): {stats['inter3_QMQ32']:>6} ({stats['inter3_QMQ32']/stats['total']*100:>5.1f}%)")

    for target in ["A1", "A2", "Gold"]:
        if f"{target}_total" not in stats:
            continue

        n_total = stats[f"{target}_total"]
        n_pos = stats[f"{target}_positives"]

        if n_pos == 0:
            continue

        print(f"\n  {target}: {n_pos} positives out of {n_total} ({n_pos/n_total*100:.1f}%)")
        print(f"    - In inter4:     {stats[f'{target}_pos_inter4']:>4}/{n_pos} ({stats[f'{target}_pos_inter4']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In inter3:     {stats[f'{target}_pos_inter3']:>4}/{n_pos} ({stats[f'{target}_pos_inter3']/n_pos*100:>5.1f}% of positives)")
        print(f"      * LQM:           {stats[f'{target}_pos_inter3_LQM']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LQM']/n_pos*100:>5.1f}%)")
        print(f"      * LQQ32:         {stats[f'{target}_pos_inter3_LQQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LQQ32']/n_pos*100:>5.1f}%)")
        print(f"      * LMQ32:         {stats[f'{target}_pos_inter3_LMQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_LMQ32']/n_pos*100:>5.1f}%)")
        print(f"      * QMQ32:         {stats[f'{target}_pos_inter3_QMQ32']:>4}/{n_pos} ({stats[f'{target}_pos_inter3_QMQ32']/n_pos*100:>5.1f}%)")
        print(f"    - In inter2:     {stats[f'{target}_pos_inter2']:>4}/{n_pos} ({stats[f'{target}_pos_inter2']/n_pos*100:>5.1f}% of positives)")
        print(f"    - In union:      {stats[f'{target}_pos_union']:>4}/{n_pos} ({stats[f'{target}_pos_union']/n_pos*100:>5.1f}% of positives)")
        print(f"    - Outside union:      {stats[f'{target}_pos_no_vote']:>4}/{n_pos} ({stats[f'{target}_pos_no_vote']/n_pos*100:>5.1f}% of positives)")


def print_comparison_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Comparison table with count/total."""
    print("\n" + "="*160)
    print("COMPARISON OF UNSUPERVISED METHODS (4 LLMs) - # yes retrieved / total # yes")
    print("="*160)

    # Main methods only for the first table
    main_methods = ["Random", "TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                   "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4"]
    main_methods = [m for m in main_methods if m in results_dict]

    for target in config.targets:
        print(f"\n{'='*160}")
        print(f"TARGET: {target}")
        print(f"{'='*160}")

        df_random = results_dict.get("Random")
        prevalence = None
        n_pos_total = None
        if df_random is not None:
            df_t_rand = df_random[df_random["target"] == target]
            if len(df_t_rand) > 0:
                prevalence = df_t_rand["prevalence"].iloc[0]
                n_pos_total = df_t_rand["total_positives"].iloc[0]
                print(f"  Prevalence (yes rate): {prevalence:.3f} ({prevalence*100:.1f}%)")
                print(f"  Total number of YES: {n_pos_total}")

        # Table AP
        print("\n  METHOD                | AP     | Δ vs Random")
        print("  " + "-"*50)
        for method, df_res in results_dict.items():
            df_t = df_res[df_res["target"] == target]
            if len(df_t) > 0:
                ap = df_t["AP"].iloc[0]
                if prevalence and method != "Random":
                    delta = ap - prevalence
                    delta_str = f"+{delta:.3f}" if delta >= 0 else f"{delta:.3f}"
                else:
                    delta_str = "-"
                print(f"  {method:<22} | {ap:.4f} | {delta_str}")

        # P@k table for the main methods
        print(f"\n  --- P@k for the main methods (yes/tot) ---")
        print(f"  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:14]:<14}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | P@k   (yes/tot)", end="")
        print()
        print("  " + "-"*(7 + 18 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_res = results_dict[method]
                df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_t) > 0:
                    p = df_t["P@k"].values[0]
                    count = df_t["count"].values[0]
                    total = df_t["total_positives"].values[0]
                    row += f" | {p:.3f} ({count:>3}/{total:<3})"
                else:
                    row += " | -               "
            print(row)

        # P@k table for the combinations of 3 LLMs
        inter3_methods = ["Inter3_LQM", "Inter3_LQQ32", "Inter3_LMQ32", "Inter3_QMQ32"]
        inter3_methods = [m for m in inter3_methods if m in results_dict]

        if inter3_methods:
            print(f"\n  --- P@k for combinations of 3 LLMs (yes/tot) ---")
            print(f"  {'k':<6}", end="")
            for method in inter3_methods:
                print(f" | {method:<16}", end="")
            print()
            print(f"  {'':6}", end="")
            for _ in inter3_methods:
                print(f" | P@k   (yes/tot) ", end="")
            print()
            print("  " + "-"*(7 + 19 * len(inter3_methods)))

            for k in [10, 50, 100, 200, 300, 500, 1000]:
                if k not in config.k_values:
                    continue
                row = f"  {k:<6}"
                for method in inter3_methods:
                    df_res = results_dict[method]
                    df_t = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                    if len(df_t) > 0:
                        p = df_t["P@k"].values[0]
                        count = df_t["count"].values[0]
                        total = df_t["total_positives"].values[0]
                        row += f" | {p:.3f} ({count:>3}/{total:<3}) "
                    else:
                        row += " | -                "
                print(row)


def print_coverage_table(coverage_dict: Dict[str, pd.DataFrame], config: Config):
    """Coverage table with count/total."""
    print("\n" + "="*180)
    print("POSITIVE COVERAGE PER METHOD (4 LLMs) - yes retrieved / total yes")
    print("="*180)

    # Main methods
    main_methods = ["TF-IDF", "BM25", "CrossEncoder", "Heuristic",
                   "LLM_Union", "LLM_Inter2", "LLM_Inter3", "LLM_Inter4"]
    main_methods = [m for m in main_methods if m in coverage_dict]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")

        df_first = next(iter(coverage_dict.values()))
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue

        total_pos = df_t["total_positives"].iloc[0]
        n_samples = df_t["n_samples"].iloc[0]
        prevalence = df_t["prevalence"].iloc[0]

        print(f"  Total positives (YES): {total_pos} / {n_samples} (prevalence: {prevalence:.3f})")
        print(f"{'='*180}")

        print(f"\n  {'k':<6}", end="")
        for method in main_methods:
            print(f" | {method[:16]:<16}", end="")
        print()
        print(f"  {'':6}", end="")
        for _ in main_methods:
            print(f" | yes/tot (R@k)   ", end="")
        print()
        print("  " + "-"*(7 + 19 * len(main_methods)))

        for k in [10, 50, 100, 200, 300, 500, 1000]:
            if k not in config.k_values:
                continue
            row = f"  {k:<6}"
            for method in main_methods:
                df_cov = coverage_dict[method]
                df_tk = df_cov[(df_cov["target"] == target) & (df_cov["k"] == k)]
                if len(df_tk) > 0:
                    retrieved = int(df_tk["positives_retrieved"].values[0])
                    total = int(df_tk["total_positives"].values[0])
                    recall = df_tk["coverage"].values[0]
                    row += f" | {retrieved:>3}/{total:<3} ({recall:.2f}) "
                else:
                    row += " | -                "
            print(row)


def print_detailed_count_table(results_dict: Dict[str, pd.DataFrame], config: Config):
    """Detailed table with only the counts (yes retrieved / total yes)."""
    print("\n" + "="*180)
    print("DETAIL: NUMBER OF YES RETRIEVED / TOTAL NUMBER OF YES (4 LLMs)")
    print("="*180)

    methods = [m for m in results_dict.keys() if m != "Random"]

    for target in config.targets:
        print(f"\n{'='*180}")
        print(f"TARGET: {target}")
        print(f"{'='*180}")

        # Get the total number of positives
        df_first = results_dict[methods[0]]
        df_t = df_first[df_first["target"] == target]
        if len(df_t) == 0:
            continue
        total_pos = df_t["total_positives"].iloc[0]
        print(f"  Total number of YES: {total_pos}")

        # Header
        print(f"\n  {'k':<6} | {'Random':<12}", end="")
        for method in methods:
            print(f" | {method[:12]:<12}", end="")
        print()
        print("  " + "-"*(7 + 15 + 15 * len(methods)))

        for k in config.k_values:
            row = f"  {k:<6}"

            # Random
            df_rand = results_dict["Random"]
            df_tk = df_rand[(df_rand["target"] == target) & (df_rand["k"] == k)]
            if len(df_tk) > 0:
                count = int(df_tk["count"].values[0])
                row += f" | {count:>4}/{total_pos:<4}   "
            else:
                row += " | -            "

            # Other methods
            for method in methods:
                df_res = results_dict[method]
                df_tk = df_res[(df_res["target"] == target) & (df_res["k"] == k)]
                if len(df_tk) > 0:
                    count = int(df_tk["count"].values[0])
                    row += f" | {count:>4}/{total_pos:<4}  "
                else:
                    row += " | -            "
            print(row)


def print_agree_disagree_comparison(df_agree_disagree: pd.DataFrame, config: Config):
    """Print the performance comparison on agree vs disagree cases."""
    print("\n" + "="*180)
    print("AGREE vs DISAGREE COMPARISON - Do the methods perform differently on the hard cases?")
    print("="*180)

    if df_agree_disagree.empty:
        print("  No data available.")
        return

    # Focus on Gold (the most relevant)
    for base_target in ["Gold", "A1", "A2"]:
        df_base = df_agree_disagree[df_agree_disagree["base_target"] == base_target]
        if df_base.empty:
            continue

        print(f"\n{'='*180}")
        print(f"TARGET: {base_target}")
        print(f"{'='*180}")

        # Basic stats
        df_agree = df_base[df_base["agreement"] == "Agree"]
        df_disagree = df_base[df_base["agreement"] == "Disagree"]

        if not df_agree.empty:
            n_agree = df_agree["n_samples"].iloc[0]
            n_pos_agree = df_agree["total_positives"].iloc[0]
            prev_agree = df_agree["prevalence"].iloc[0]
            print(f"  AGREE:    {n_agree} samples, {n_pos_agree} positives (prevalence: {prev_agree:.3f})")

        if not df_disagree.empty:
            n_disagree = df_disagree["n_samples"].iloc[0]
            n_pos_disagree = df_disagree["total_positives"].iloc[0]
            prev_disagree = df_disagree["prevalence"].iloc[0]
            print(f"  DISAGREE: {n_disagree} samples, {n_pos_disagree} positives (prevalence: {prev_disagree:.3f})")

        # AP comparison per method
        print(f"\n  --- Average Precision (AP) ---")
        print(f"  {'METHOD':<22} | {'AP Agree':<10} | {'AP Disagree':<12} | {'Δ (Agree-Disagree)':<18} | {'Δ vs Random Agree':<18} | {'Δ vs Random Disagree':<20}")
        print("  " + "-"*120)

        methods = df_base["method"].unique()

        # Compute the prevalences (random baseline)
        random_ap_agree = prev_agree if not df_agree.empty else 0
        random_ap_disagree = prev_disagree if not df_disagree.empty else 0

        for method in methods:
            df_m_agree = df_agree[df_agree["method"] == method]
            df_m_disagree = df_disagree[df_disagree["method"] == method]

            ap_agree = df_m_agree["AP"].iloc[0] if not df_m_agree.empty else np.nan
            ap_disagree = df_m_disagree["AP"].iloc[0] if not df_m_disagree.empty else np.nan

            if pd.notna(ap_agree) and pd.notna(ap_disagree):
                delta = ap_agree - ap_disagree
                delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
                delta_rand_agree = ap_agree - random_ap_agree
                delta_rand_disagree = ap_disagree - random_ap_disagree
                delta_rand_agree_str = f"+{delta_rand_agree:.4f}" if delta_rand_agree >= 0 else f"{delta_rand_agree:.4f}"
                delta_rand_disagree_str = f"+{delta_rand_disagree:.4f}" if delta_rand_disagree >= 0 else f"{delta_rand_disagree:.4f}"
            else:
                delta_str = "-"
                delta_rand_agree_str = "-"
                delta_rand_disagree_str = "-"

            ap_agree_str = f"{ap_agree:.4f}" if pd.notna(ap_agree) else "-"
            ap_disagree_str = f"{ap_disagree:.4f}" if pd.notna(ap_disagree) else "-"

            print(f"  {method:<22} | {ap_agree_str:<10} | {ap_disagree_str:<12} | {delta_str:<18} | {delta_rand_agree_str:<18} | {delta_rand_disagree_str:<20}")

        # P@k table for a few important k
        print(f"\n  --- P@k Comparison (Agree vs Disagree) ---")
        key_methods = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2", "CrossEncoder"]
        key_methods = [m for m in key_methods if m in methods]

        for method in key_methods:
            print(f"\n  {method}:")
            print(f"    {'k':<6} | {'P@k Agree (yes/tot)':<22} | {'P@k Disagree (yes/tot)':<24} | {'Δ P@k':<10}")
            print("    " + "-"*75)

            for k in [10, 50, 100, 200, 300, 500]:
                if k not in config.k_values:
                    continue

                df_m_agree_k = df_agree[(df_agree["method"] == method) & (df_agree["k"] == k)]
                df_m_disagree_k = df_disagree[(df_disagree["method"] == method) & (df_disagree["k"] == k)]

                if not df_m_agree_k.empty and not df_m_disagree_k.empty:
                    p_agree = df_m_agree_k["P@k"].values[0]
                    p_disagree = df_m_disagree_k["P@k"].values[0]
                    c_agree = df_m_agree_k["count"].values[0]
                    c_disagree = df_m_disagree_k["count"].values[0]
                    t_agree = df_m_agree_k["total_positives"].values[0]
                    t_disagree = df_m_disagree_k["total_positives"].values[0]
                    delta_p = p_agree - p_disagree
                    delta_str = f"+{delta_p:.3f}" if delta_p >= 0 else f"{delta_p:.3f}"

                    print(f"    {k:<6} | {p_agree:.3f} ({c_agree:>3}/{t_agree:<3})         | {p_disagree:.3f} ({c_disagree:>3}/{t_disagree:<3})           | {delta_str}")
                else:
                    print(f"    {k:<6} | -                      | -                        | -")


def plot_agree_disagree_comparison(df_agree_disagree: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot comparing performance on agree vs disagree."""
    if df_agree_disagree.empty:
        return

    # Focus on Gold
    df_gold = df_agree_disagree[df_agree_disagree["base_target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2",
                       "LLM_Union", "CrossEncoder", "BM25", "TF-IDF"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter4': 'brown', 'LLM_Inter3': 'purple',
        'LLM_Inter2': 'red', 'LLM_Union': 'orange', 'CrossEncoder': 'green',
        'BM25': 'cyan', 'TF-IDF': 'blue'
    }

    # Plot P@k for Agree
    ax = axes[0]
    df_agree = df_gold[df_gold["agreement"] == "Agree"]
    for method in methods_to_plot:
        df_m = df_agree[df_agree["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["P@k"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    # Add random baseline (prevalence)
    if not df_agree.empty:
        prev = df_agree["prevalence"].iloc[0]
        ax.axhline(y=prev, color='black', linestyle='--', linewidth=1,
                  label=f'Random ({prev:.3f})', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("P@k")
    ax.set_title("Gold - AGREE cases (annotators agree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.0)

    # Plot P@k for Disagree
    ax = axes[1]
    df_disagree = df_gold[df_gold["agreement"] == "Disagree"]
    for method in methods_to_plot:
        df_m = df_disagree[df_disagree["method"] == method].sort_values("k")
        if not df_m.empty:
            ax.plot(df_m["k"], df_m["P@k"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    # Add random baseline (prevalence)
    if not df_disagree.empty:
        prev = df_disagree["prevalence"].iloc[0]
        ax.axhline(y=prev, color='black', linestyle='--', linewidth=1,
                  label=f'Random ({prev:.3f})', alpha=0.7)

    ax.set_xlabel("k")
    ax.set_ylabel("P@k")
    ax.set_title("Gold - DISAGREE cases (annotators disagree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_agree_vs_disagree_delta(df_agree_disagree: pd.DataFrame, config: Config, output_path: Optional[str] = None):
    """Plot showing the performance delta between agree and disagree."""
    if df_agree_disagree.empty:
        return

    df_gold = df_agree_disagree[df_agree_disagree["base_target"] == "Gold"]
    if df_gold.empty:
        return

    methods_to_plot = ["Heuristic", "LLM_Inter4", "LLM_Inter3", "LLM_Inter2",
                       "LLM_Union", "CrossEncoder", "BM25", "TF-IDF"]
    methods_to_plot = [m for m in methods_to_plot if m in df_gold["method"].unique()]

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {
        'Heuristic': 'magenta', 'LLM_Inter4': 'brown', 'LLM_Inter3': 'purple',
        'LLM_Inter2': 'red', 'LLM_Union': 'orange', 'CrossEncoder': 'green',
        'BM25': 'cyan', 'TF-IDF': 'blue'
    }

    df_agree = df_gold[df_gold["agreement"] == "Agree"]
    df_disagree = df_gold[df_gold["agreement"] == "Disagree"]

    for method in methods_to_plot:
        df_m_agree = df_agree[df_agree["method"] == method].sort_values("k")
        df_m_disagree = df_disagree[df_disagree["method"] == method].sort_values("k")

        if not df_m_agree.empty and not df_m_disagree.empty:
            # Merge on k
            merged = df_m_agree[["k", "P@k"]].merge(
                df_m_disagree[["k", "P@k"]], on="k", suffixes=("_agree", "_disagree")
            )
            merged["delta"] = merged["P@k_agree"] - merged["P@k_disagree"]

            ax.plot(merged["k"], merged["delta"], color=colors.get(method, 'gray'),
                   marker='o', linewidth=2, label=method)

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel("k")
    ax.set_ylabel("Δ P@k (Agree - Disagree)")
    ax.set_title("Gold - P@k difference between AGREE and DISAGREE cases\n(positive = better on Agree, negative = better on Disagree)")
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# PLOTS
# ============================================================
def plot_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualisation."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in main_methods:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric}")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_inter3_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Visualisation of the 4 combinations of 3 LLMs."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(2, n_targets, figsize=(5 * n_targets, 8))
    if n_targets == 1:
        axes = axes.reshape(-1, 1)

    colors = {
        'Random': 'black',
        'Inter3_LQM': 'blue',
        'Inter3_LQQ32': 'green',
        'Inter3_LMQ32': 'red',
        'Inter3_QMQ32': 'purple',
        'LLM_Inter3': 'orange',
        'LLM_Inter4': 'brown'
    }

    methods_to_plot = ['Random', 'Inter3_LQM', 'Inter3_LQQ32', 'Inter3_LMQ32',
                       'Inter3_QMQ32', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        for row, metric in enumerate(["P@k", "NDCG@k"]):
            ax = axes[row, i]
            for method in methods_to_plot:
                if method not in results_dict:
                    continue
                df_res = results_dict[method]
                df_t = df_res[df_res["target"] == target].sort_values("k")
                if len(df_t) > 0:
                    ax.plot(df_t["k"], df_t[metric],
                           color=colors.get(method, 'gray'),
                           marker='o' if method != 'Random' else '.',
                           linewidth=2 if method != 'Random' else 1,
                           label=method, alpha=0.7 if method == 'Random' else 1.0)
            ax.set_xlabel("k")
            ax.set_ylabel(metric)
            ax.set_title(f"{target} - {metric} (Inter3 Combinations)")
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1.0)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_coverage_curves(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Coverage curves."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        df_first = next(iter(coverage_dict.values()))
        df_t_first = df_first[df_first["target"] == target].sort_values("k")

        if len(df_t_first) > 0:
            ax.plot(df_t_first["k"], df_t_first["random_expected"],
                   color='black', linestyle='--', linewidth=1.5,
                   label='Random (expected)', alpha=0.7)

            # Horizontal line for the total positives
            total_pos = df_t_first["total_positives"].iloc[0]
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["positives_retrieved"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k")
        ax.set_ylabel("YES retrieved")
        ax.set_title(f"{target} - # of YES retrieved")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_gain_vs_random(coverage_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of relative gain over chance."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'TF-IDF': 'blue', 'BM25': 'cyan', 'CrossEncoder': 'green',
        'LLM_Union': 'orange', 'LLM_Inter2': 'red',
        'LLM_Inter3': 'purple', 'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

        for method in main_methods:
            if method not in coverage_dict:
                continue
            df_cov = coverage_dict[method]
            df_t = df_cov[df_cov["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["delta_vs_random"],
                       color=colors.get(method, 'gray'),
                       marker='o', linewidth=2, label=method)

        ax.set_xlabel("k (number of documents)")
        ax.set_ylabel("Gain vs Random (number of YES)")
        ax.set_title(f"{target} - Gain over chance")
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


def plot_count_comparison(results_dict: Dict[str, pd.DataFrame], config: Config, output_path: Optional[str] = None):
    """Plot of the number of retrieved YES per method."""
    n_targets = len(config.targets)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    if n_targets == 1:
        axes = [axes]

    colors = {
        'Random': 'black', 'TF-IDF': 'blue', 'BM25': 'cyan',
        'CrossEncoder': 'green', 'LLM_Union': 'orange',
        'LLM_Inter2': 'red', 'LLM_Inter3': 'purple',
        'LLM_Inter4': 'brown', 'Heuristic': 'magenta'
    }

    main_methods = ['Random', 'TF-IDF', 'BM25', 'CrossEncoder', 'Heuristic',
                   'LLM_Union', 'LLM_Inter2', 'LLM_Inter3', 'LLM_Inter4']

    for i, target in enumerate(config.targets):
        ax = axes[i]
        total_pos = None

        for method in main_methods:
            if method not in results_dict:
                continue
            df_res = results_dict[method]
            df_t = df_res[df_res["target"] == target].sort_values("k")
            if len(df_t) > 0:
                ax.plot(df_t["k"], df_t["count"],
                       color=colors.get(method, 'gray'),
                       marker='o' if method != 'Random' else '.',
                       linewidth=2 if method != 'Random' else 1,
                       linestyle='--' if method == 'Random' else '-',
                       label=method, alpha=0.7 if method == 'Random' else 1.0)

                if method != 'Random' and total_pos is None:
                    total_pos = df_t["total_positives"].iloc[0]

        # Add the total line
        if total_pos is not None:
            ax.axhline(y=total_pos, color='gray', linestyle=':', linewidth=1,
                      label=f'Total YES ({total_pos})', alpha=0.5)

        ax.set_xlabel("k")
        ax.set_ylabel("Number of YES retrieved")
        ax.set_title(f"{target} - YES retrieved vs k")
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  Saved: {output_path}")
    plt.show()


# ============================================================
# MAIN
# ============================================================
def main(filter_union: bool = False, compute_crossencoder: bool = True):
    """Main pipeline."""
    print("="*70)
    print("FULLY UNSUPERVISED RANKING PIPELINE (4 LLMs)")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    print(f"  Mode: {'Union only' if filter_union else 'All data'}")
    print(f"  Cross-Encoder: {compute_crossencoder}")
    print("="*70)
    print("""
4 LLMs: Llama, Qwen, Mistral, Qwen32B

COMBINATIONS TESTED:
---------------------
- Union: at least 1 LLM says yes
- Inter2: at least 2 LLMs say yes
- Inter3: at least 3 of the 4 LLMs say yes
- Inter4: all 4 LLMs say yes

Specific combinations of 3:
- Inter3_LQM: Llama + Qwen + Mistral
- Inter3_LQQ32: Llama + Qwen + Qwen32B
- Inter3_LMQ32: Llama + Mistral + Qwen32B
- Inter3_QMQ32: Qwen + Mistral + Qwen32B

TIE HANDLING (TIE-BREAK):
------------------------------------
The binary methods (LLM_Union, LLM_Inter2, LLM_Inter3, LLM_Inter4)
produce 0/1 scores creating ties. To break them
deterministically:

  Score_final = Score_LLM + ε × Score_CrossEncoder  (ε = 1e-6)

Documents with the same LLM vote are ordered by their CrossEncoder score.
The continuous methods (TF-IDF, BM25, CrossEncoder, Heuristic) have no
significant ties and use a direct sort.
""")

    print("\nChecking dependencies...")
    install_dependencies()

    config = Config()
    suffix = "_union" if filter_union else "_all"
    config.output_dir = f"artifacts/outputs_unsupervised_4llm{suffix}"

    np.random.seed(config.random_seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # Load & prepare
    df_raw = load_data(config)
    df_full = prepare_features(df_raw, config, compute_crossencoder=compute_crossencoder)

    if filter_union:
        df = df_full[df_full["vote_union"] == 1].copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (union filter)")
    else:
        df = df_full.copy().reset_index(drop=True)
        print(f"\n  Dataset: {len(df)} rows (all data)")

    # LLM intersection analysis
    llm_stats = analyze_llm_intersection_coverage(df)
    print_llm_intersection_analysis(llm_stats)

    # Compute all scores
    print("\n" + "="*70)
    print("COMPUTING RANKINGS")
    print("="*70)

    all_scores = compute_all_ranking_scores(df)

    results_dict = {}
    coverage_dict = {}

    n_methods = len(all_scores) + 1  # +1 for Random
    print(f"\n  [1/{n_methods}] Random baseline...")
    results_dict["Random"] = compute_random_baseline(df, config)

    for i, (method_name, scores) in enumerate(all_scores.items(), 2):
        print(f"  [{i}/{n_methods}] {method_name}...")
        results_dict[method_name] = evaluate_method(df, scores, method_name, config)
        coverage_dict[method_name] = analyze_positive_coverage(df, scores, method_name, config)

    # Reporting
    print_comparison_table(results_dict, config)
    print_coverage_table(coverage_dict, config)
    print_detailed_count_table(results_dict, config)

    # Agree vs Disagree analysis
    print("\n" + "="*70)
    print("AGREE vs DISAGREE ANALYSIS")
    print("="*70)
    df_agree_disagree = analyze_agree_disagree_performance(df, all_scores, config)
    print_agree_disagree_comparison(df_agree_disagree, config)

    # Plots
    plot_comparison(results_dict, config, f"{config.output_dir}/comparison_main.png")
    plot_inter3_comparison(results_dict, config, f"{config.output_dir}/comparison_inter3.png")
    plot_coverage_curves(coverage_dict, config, f"{config.output_dir}/coverage.png")
    plot_gain_vs_random(coverage_dict, config, f"{config.output_dir}/gain_vs_random.png")
    plot_count_comparison(results_dict, config, f"{config.output_dir}/count_comparison.png")

    # Agree vs Disagree plots
    plot_agree_disagree_comparison(df_agree_disagree, config, f"{config.output_dir}/agree_vs_disagree.png")
    plot_agree_vs_disagree_delta(df_agree_disagree, config, f"{config.output_dir}/agree_disagree_delta.png")

    # Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    for name, scores in all_scores.items():
        df[f"score_{name.lower().replace('-', '_')}"] = scores

    df.to_csv(f"{config.output_dir}/df_with_scores_{timestamp}.csv", index=False)

    df_results = pd.concat(list(results_dict.values()), ignore_index=True)
    df_results.to_csv(f"{config.output_dir}/results_{timestamp}.csv", index=False)

    df_coverage = pd.concat(list(coverage_dict.values()), ignore_index=True)
    df_coverage.to_csv(f"{config.output_dir}/coverage_{timestamp}.csv", index=False)

    # Save agree/disagree results
    df_agree_disagree.to_csv(f"{config.output_dir}/agree_disagree_results_{timestamp}.csv", index=False)

    with open(f"{config.output_dir}/llm_stats_{timestamp}.json", "w") as f:
        json.dump(llm_stats, f, indent=2)

    print(f"\n  Results saved to: {config.output_dir}/")

    return df, results_dict, coverage_dict, llm_stats, df_agree_disagree


if __name__ == "__main__":
    df, results, coverage, llm_stats, agree_disagree = main(filter_union=False, compute_crossencoder=True)